# Gene-ius: mechanism-aware VUS interpretation (Omar's pipeline)
Nucleate Pittsburgh BioHack 2026 · BioTech Track · NVIDIA Problem Statement 2

**What this notebook does.** Builds the variant model, evidence, and outputs behind Gene-ius for 23 retina and heart genes: ClinVar data, AlphaMissense, AlphaFold, related-gene (paralog) evidence, gnomAD and 1000 Genomes healthy carriers, MGI mouse phenotypes, FoldX physics, a transparent logistic model, ACMG PP3/BP4 calibration, benchmarks, figures, demo cards, and evidence reports.

**Validation design (time machine).** Train only on ClinVar as of January 2023; test on variants that were uncertain then and have since been resolved by expert review. No variant ever counts as its own evidence, and every tool is compared on the same variants.

**Headline results**
- 38 vs. 18 later-proven disease variants in the top 500 of 10,173 uncertain 2023 variants (vs. AlphaMissense); gain +18, 95% CI +7 to +29
- vs. REVEL on the same variants: AUROC 0.953 vs. 0.893 (gain CI +0.026 to +0.102); top 5%: 27 vs. 17 (CI +2 to +19)
- Related genes: CACNA1C 0.76 → 0.92 AUROC (no AlphaMissense scores for this gene)
- Which disease: 81% vs. 65% for the gene's most common disease (p = 0.021)
- Calibrated PP3 evidence was right 91 of 91 times on variants resolved after 2023

**How to run.** Google Colab, with a Google Drive folder `MyDrive/biohack/` for inputs and outputs. Cells were run top to bottom in one session, and later cells reuse variables from earlier ones. Large downloads: ClinVar (~440 MB), the January 2023 ClinVar release (~140 MB), AlphaMissense. FoldX needs an academic license and is not included; its runs take hours.

**Outputs (saved to Drive, not in git):** `predictions.json/.csv`, `todays_uncertain_variants_ranked.csv`, `feature_table.csv`, `demo_cards.json`, `viewer_*.html`, `evidence_reports.pdf`, `figures/`, plus the evidence files (`evidence_calibration.csv`, `mouse_phenotypes_mgi.csv`, `kg1000_*.csv`, `foldx_*_results.csv`).

*For readability, print-only checks (GPU status, file listings, FoldX progress monitors) were removed. One superseded calibration cell is kept because later cells use its variables; it is marked below.*

## 1. Download ClinVar: current release and the January 2023 freeze

In [3]:
!wget -q --show-progress https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
!ls -lh variant_summary.txt.gz

variant_summary.txt 100%[===================>] 436.65M  21.3MB/s    in 14s     
-rw-r--r-- 1 root root 437M Nov  1 14:17 variant_summary.txt.gz


In [6]:
!curl -s https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/2023/ | grep -o 'href="variant_summary[^"]*"'

href="variant_summary_2023-01.txt.gz"
href="variant_summary_2023-02.txt.gz"
href="variant_summary_2023-03.txt.gz"
href="variant_summary_2023-04.txt.gz"
href="variant_summary_2023-05.txt.gz"
href="variant_summary_2023-06.txt.gz"
href="variant_summary_2023-07.txt.gz"
href="variant_summary_2023-08.txt.gz"
href="variant_summary_2023-09.txt.gz"
href="variant_summary_2023-10.txt.gz"
href="variant_summary_2023-11.txt.gz"
href="variant_summary_2023-12.txt.gz"


In [7]:
!wget -q --show-progress https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/2023/variant_summary_2023-01.txt.gz
!ls -lh variant_summary*.txt.gz

variant_summary_202 100%[===================>] 141.38M  54.7MB/s    in 2.6s    
-rw-r--r-- 1 root root 142M Jan  5  2023 variant_summary_2023-01.txt.gz
-rw-r--r-- 1 root root 437M Nov  1 14:17 variant_summary.txt.gz


## 2. Filter to the 23-gene panel and build the time-machine split

In [8]:
import pandas as pd

cur = pd.read_csv("variant_summary.txt.gz", sep="\t", nrows=5, low_memory=False)
old = pd.read_csv("variant_summary_2023-01.txt.gz", sep="\t", nrows=5, low_memory=False)

print("CURRENT:", list(cur.columns))
print()
print("2023:", list(old.columns))

CURRENT: ['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines', 'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated', 'ReviewStatusClinicalImpact', 'Oncogenicity', 'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity', 'SCVsForAggregateGermlineClassification', 'SCVsForAggregateSomaticClinicalImpact', 'SCVsForAggregateOncogenicityClassification']

2023: ['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVacc

In [9]:
import pandas as pd

GENES = ["RHO","ABCA4","GUCY2D","CRX","PRPH2","RPE65","CEP290","BEST1","NR2E3","RPGR","OPA1","PAX6","USH2A",
         "SCN5A","KCNQ1","KCNH2","KCNJ2","CACNA1C","MYH7","MYBPC3","LMNA","PKP2","GLA"]
COLS = ["#AlleleID","Type","Name","GeneID","GeneSymbol","ClinicalSignificance","PhenotypeIDS",
        "PhenotypeList","Assembly","ReviewStatus","NumberSubmitters","VariationID"]
pattern = r"(?:^|;)(?:" + "|".join(GENES) + r")(?:;|$)"

def load(path):
    parts = []
    for chunk in pd.read_csv(path, sep="\t", usecols=COLS, chunksize=500_000, dtype=str):
        chunk = chunk[chunk["Assembly"] == "GRCh38"]
        chunk = chunk[chunk["GeneSymbol"].fillna("").str.contains(pattern, regex=True)]
        parts.append(chunk)
    return pd.concat(parts, ignore_index=True)

old = load("variant_summary_2023-01.txt.gz")
cur = load("variant_summary.txt.gz")
print("2023 rows:", len(old), "| current rows:", len(cur))
print(cur["GeneSymbol"].value_counts().head(25))

2023 rows: 36911 | current rows: 67506
GeneSymbol
USH2A              10073
MYH7                5943
SCN5A               5179
ABCA4               4992
CEP290              4874
MYBPC3              4806
KCNH2               3931
CACNA1C             3841
KCNQ1               2672
LMNA                2522
PKP2                2403
RPGR                2287
GLA                 2149
OPA1                1955
GUCY2D              1819
RPE65               1244
BEST1               1066
PAX6                1030
NR2E3                939
PRPH2                906
RHO                  723
KCNJ2                695
CRX                  622
KCNQ1;KCNQ1OT1       532
USH2A;USH2A-AS1       41
Name: count, dtype: int64


In [10]:
import re

AA = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
      "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V"}
PROT = re.compile(r"\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)")
GENESET = set(GENES)

def missense(df):
    df = df[df["Type"] == "single nucleotide variant"].copy()
    m = df["Name"].str.extract(PROT)
    df["ref_aa"] = m[0].map(AA)
    df["position"] = pd.to_numeric(m[1], errors="coerce")
    df["alt_aa"] = m[2].map(AA)
    df = df.dropna(subset=["ref_aa", "alt_aa", "position"])
    df = df[df["ref_aa"] != df["alt_aa"]].copy()
    df["position"] = df["position"].astype(int)
    df["gene"] = df["GeneSymbol"].str.split(";").apply(lambda g: next((x for x in g if x in GENESET), None))
    return df

old_m = missense(old)
cur_m = missense(cur)
print("missense 2023:", len(old_m), "| missense now:", len(cur_m))
print(cur_m["gene"].value_counts())

missense 2023: 15960 | missense now: 28926
gene
USH2A      3918
MYH7       3211
SCN5A      2745
ABCA4      2072
MYBPC3     1939
KCNH2      1834
CEP290     1646
CACNA1C    1574
LMNA       1252
GLA        1149
KCNQ1      1097
PKP2       1070
GUCY2D      848
OPA1        775
RPGR        622
BEST1       612
RPE65       480
RHO         419
PRPH2       416
NR2E3       377
KCNJ2       369
PAX6        282
CRX         219
Name: count, dtype: int64


In [11]:
def simple_class(s):
    s = str(s).split(";")[0].strip().lower()
    if "conflicting" in s: return "conflicting"
    if s.startswith("uncertain significance"): return "VUS"
    if "pathogenic" in s and "benign" not in s: return "P"
    if "benign" in s and "pathogenic" not in s: return "B"
    return "other"

def stars(s):
    s = str(s).lower()
    if "practice guideline" in s: return 4
    if "expert panel" in s: return 3
    if "multiple submitters" in s and "no conflicts" in s: return 2
    if "criteria provided" in s: return 1
    return 0

for df in (old_m, cur_m):
    df["cls"] = df["ClinicalSignificance"].map(simple_class)
    df["stars"] = df["ReviewStatus"].map(stars)

merged = old_m[["VariationID","gene","ref_aa","position","alt_aa","cls","stars"]].merge(
    cur_m[["VariationID","cls","stars","PhenotypeIDS","PhenotypeList"]],
    on="VariationID", suffixes=("_2023","_now"))

vus = merged["cls_2023"] == "VUS"
merged["to_P_2star"] = vus & (merged["cls_now"] == "P") & (merged["stars_now"] >= 2)
merged["to_B_2star"] = vus & (merged["cls_now"] == "B") & (merged["stars_now"] >= 2)
merged["to_P_1star"] = vus & (merged["cls_now"] == "P") & (merged["stars_now"] >= 1)
merged["to_B_1star"] = vus & (merged["cls_now"] == "B") & (merged["stars_now"] >= 1)

print("How 2023 classes changed (rows = 2023, columns = now):")
print(pd.crosstab(merged["cls_2023"], merged["cls_now"]))
print()

summary = merged[vus].groupby("gene").agg(
    vus_2023=("VariationID", "size"),
    to_P_2star=("to_P_2star", "sum"),
    to_B_2star=("to_B_2star", "sum"),
    to_P_1star=("to_P_1star", "sum"),
    to_B_1star=("to_B_1star", "sum"),
)
summary["solved_2star"] = summary["to_P_2star"] + summary["to_B_2star"]
summary = summary.sort_values("solved_2star", ascending=False)
print(summary)
print()
print("TOTAL solved (2+ stars):", int(summary["solved_2star"].sum()),
      "| pathogenic:", int(summary["to_P_2star"].sum()),
      "| benign:", int(summary["to_B_2star"].sum()))
print("TOTAL solved (1+ star):", int(summary["to_P_1star"].sum() + summary["to_B_1star"].sum()))

merged.to_csv("clinvar_2023_vs_now.csv", index=False)

How 2023 classes changed (rows = 2023, columns = now):
cls_now        B     P   VUS  conflicting  other
cls_2023                                        
B            311     0    17           99      0
P              0  2072    56          235      1
VUS           54   272  9539         1146      0
conflicting   48   194    54         1154      0
other          1   103   103           20    428

         vus_2023  to_P_2star  to_B_2star  to_P_1star  to_B_1star  \
gene                                                                
USH2A        1493          29           0          53           2   
ABCA4         672          28           0          56           2   
GUCY2D        301           7           5           8           5   
RPE65         144          12           0          13           0   
MYH7         1478          11           0          39           0   
RPGR           81           4           5           6           7   
BEST1         170           8           0        

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/biohack', exist_ok=True)
merged.to_csv('/content/drive/MyDrive/biohack/clinvar_2023_vs_now.csv', index=False)
summary.to_csv('/content/drive/MyDrive/biohack/vus_reclassified_counts.csv')
print("Saved")

Mounted at /content/drive
Saved


In [13]:
old_ph = old_m[["VariationID", "PhenotypeIDS", "PhenotypeList"]].rename(
    columns={"PhenotypeIDS": "pheno_ids_2023", "PhenotypeList": "disease_2023"})
table = merged.merge(old_ph, on="VariationID", how="left")

def split(r):
    if r.cls_2023 in ("P", "B"): return "train"
    if r.cls_2023 == "VUS" and r.cls_now in ("P", "B") and r.stars_now >= 2: return "test"
    if r.cls_2023 == "VUS" and r.cls_now in ("P", "B"): return "test_1star"
    if r.cls_2023 == "VUS": return "unlabeled"
    return "exclude"

table["split"] = table.apply(split, axis=1)

# Leakage rule: hide the 2023 "condition tested for" on VUS
table.loc[table["cls_2023"] == "VUS", ["disease_2023", "pheno_ids_2023"]] = ""

out = table.rename(columns={
    "VariationID": "variant_id", "cls_2023": "class_2023", "cls_now": "class_now",
    "PhenotypeList": "disease_now", "PhenotypeIDS": "pheno_ids_now"})
out = out[["variant_id", "gene", "position", "ref_aa", "alt_aa",
           "class_2023", "stars_2023", "class_now", "stars_now", "split",
           "disease_2023", "pheno_ids_2023", "disease_now", "pheno_ids_now"]]
out.to_csv("/content/drive/MyDrive/biohack/variant_table.csv", index=False)

print(out["split"].value_counts())
print()
test = out[out["split"] == "test"]
generic = r"not provided|not specified"
specific = test[~test["disease_now"].fillna("").str.contains(generic, case=False)]
print("Test variants with a specific disease:", len(specific), "of", len(test))
print(specific["disease_now"].str.split("|").str[0].value_counts().head(15))

split
unlabeled     10685
train          2791
exclude        2105
test_1star      181
test            145
Name: count, dtype: int64

Test variants with a specific disease: 38 of 145
disease_now
Long QT syndrome                                        8
Congenital long QT syndrome                             4
Hypertrophic cardiomyopathy                             4
Fabry disease                                           3
Leber congenital amaurosis 2                            3
MYH7-related skeletal myopathy                          2
Leber congenital amaurosis 2;Retinitis pigmentosa 20    2
Retinitis pigmentosa                                    2
Dilated cardiomyopathy 1S                               1
Cone-rod dystrophy                                      1
Cardiovascular phenotype                                1
Retinal dystrophy                                       1
Charcot-Marie-Tooth disease type 2                      1
PRPH2-related disorder                              

## 3. Disease names → disease groups (MONDO)

In [14]:
GENERIC = {"not provided", "not specified", "see cases", "cardiovascular phenotype",
           "retinal dystrophy", "inborn genetic diseases"}

def specific_diseases(s):
    if not isinstance(s, str) or not s:
        return ""
    parts = [p.strip() for p in re.split(r"[|;]", s)]
    return "|".join(p for p in parts if p and p.lower() not in GENERIC)

out["diseases_now_specific"] = out["disease_now"].map(specific_diseases)
out["diseases_2023_specific"] = out["disease_2023"].map(specific_diseases)

def report(df, col, label):
    has = df[col] != ""
    print(label, has.sum(), "of", len(df))
    return has

test_p = out[(out["split"] == "test") & (out["class_now"] == "P")]
has = report(test_p, "diseases_now_specific", "Test (2+ star) pathogenic with a specific disease:")
print(test_p.loc[has, "diseases_now_specific"].str.split("|").explode().value_counts().head(20))
print()
report(out[out["split"].isin(["test", "test_1star"]) & (out["class_now"] == "P")],
       "diseases_now_specific", "Test incl. 1-star, pathogenic with a specific disease:")
report(out[(out["split"] == "train") & (out["class_2023"] == "P")],
       "diseases_2023_specific", "Train pathogenic with a specific disease:")

out.to_csv("/content/drive/MyDrive/biohack/variant_table.csv", index=False)
print("Saved")

Test (2+ star) pathogenic with a specific disease: 121 of 125
diseases_now_specific
Usher syndrome type 2A                  32
Retinitis pigmentosa 39                 28
USH2A-related disorder                  22
ABCA4-related disorder                  19
Leber congenital amaurosis 2            16
Usher syndrome                          15
Retinitis pigmentosa                    12
RPE65-related recessive retinopathy     12
Hypertrophic cardiomyopathy             11
Leber congenital amaurosis              11
Retinitis pigmentosa 20                 11
Leber congenital amaurosis 1            10
Long QT syndrome                         9
ABCA4-related retinopathy                7
Congenital long QT syndrome              7
Cone-rod dystrophy 6                     7
Fabry disease                            6
BEST1-related disorder                   6
GUCY2D-related recessive retinopathy     6
RPE65-related disorder                   4
Name: count, dtype: int64

Test incl. 1-star, pathogenic

In [15]:
RULES = [
    ("usher", "Usher syndrome"), ("joubert", "Joubert syndrome"), ("meckel", "Meckel syndrome"),
    ("senior-l", "Senior-Loken syndrome"), ("bardet", "Bardet-Biedl syndrome"), ("behr", "Behr syndrome"),
    ("jervell", "Jervell and Lange-Nielsen"), ("timothy", "Timothy syndrome"), ("andersen", "Andersen-Tawil syndrome"),
    ("leber congenital amaurosis", "Leber congenital amaurosis"), ("retinitis pigmentosa", "Retinitis pigmentosa"),
    ("cone-rod", "Cone-rod dystrophy"), ("stargardt", "Stargardt disease"), ("night blindness", "Night blindness"),
    ("vitelliform", "Best vitelliform macular dystrophy"), ("bestrophinopathy", "Bestrophinopathy"),
    ("enhanced s-cone", "Enhanced S-cone syndrome"), ("optic atrophy", "Optic atrophy"), ("aniridia", "Aniridia"),
    ("pattern dystrophy", "Pattern dystrophy"), ("macular", "Macular dystrophy"),
    ("romano-ward", "Long QT syndrome"), ("long qt", "Long QT syndrome"), ("short qt", "Short QT syndrome"),
    ("brugada", "Brugada syndrome"),
    ("hypertrophic cardiomyopathy", "Hypertrophic cardiomyopathy"), ("dilated cardiomyopathy", "Dilated cardiomyopathy"),
    ("noncompaction", "LV noncompaction"), ("arrhythmogenic", "Arrhythmogenic cardiomyopathy"),
    ("cardiomyopathy", None),  # unspecified cardiomyopathy is too vague
    ("muscular dystrophy", "Muscular dystrophy"), ("myopathy", "Skeletal myopathy"),
    ("charcot", "Charcot-Marie-Tooth"), ("progeria", "Progeria"), ("lipodystrophy", "Lipodystrophy"),
    ("fabry", "Fabry disease"),
    ("related", None),  # "GENE-related disorder" names only the gene
]
SYNDROMIC = {"Usher syndrome", "Joubert syndrome", "Meckel syndrome", "Senior-Loken syndrome",
             "Bardet-Biedl syndrome", "Behr syndrome", "Jervell and Lange-Nielsen",
             "Timothy syndrome", "Andersen-Tawil syndrome"}

def disease_groups(s):
    found = set()
    for name in (s or "").split("|"):
        low = name.lower()
        for kw, grp in RULES:
            if kw in low:
                if grp:
                    found.add(grp)
                break
    return "|".join(sorted(found))

out["group_now"] = out["diseases_now_specific"].fillna("").map(disease_groups)
out["group_2023"] = out["diseases_2023_specific"].fillna("").map(disease_groups)
out["syndromic_now"] = out["group_now"].map(lambda g: any(x in SYNDROMIC for x in g.split("|")) if g else None)

test_p = out[(out["split"] == "test") & (out["class_now"] == "P")]
print("Test pathogenic with a disease group:", (test_p["group_now"] != "").sum(), "of", len(test_p))
print()
print("Test: disease groups per gene")
t = test_p[test_p["group_now"] != ""].assign(grp=lambda d: d["group_now"].str.split("|")).explode("grp")
print(pd.crosstab(t["gene"], t["grp"]).to_string())
print()
train_p = out[(out["split"] == "train") & (out["class_2023"] == "P") & (out["group_2023"] != "")]
tr = train_p.assign(grp=lambda d: d["group_2023"].str.split("|")).explode("grp")
multi = tr.groupby("gene")["grp"].nunique()
print("Genes with 2+ disease groups in training (where the model must choose):")
print(tr[tr["gene"].isin(multi[multi >= 2].index)].groupby(["gene", "grp"]).size().to_string())

out.to_csv("/content/drive/MyDrive/biohack/variant_table.csv", index=False)
print("\nSaved")

Test pathogenic with a disease group: 90 of 125

Test: disease groups per gene
grp      Aniridia  Best vitelliform macular dystrophy  Bestrophinopathy  Brugada syndrome  Charcot-Marie-Tooth  Cone-rod dystrophy  Dilated cardiomyopathy  Fabry disease  Hypertrophic cardiomyopathy  Leber congenital amaurosis  Long QT syndrome  Macular dystrophy  Muscular dystrophy  Night blindness  Retinitis pigmentosa  Skeletal myopathy  Stargardt disease  Timothy syndrome  Usher syndrome
gene                                                                                                                                                                                                                                                                                                                                                                                                      
ABCA4           0                                   0                 0                 0                    0                   1 

## 4. AlphaMissense scores, residue checks, and the AlphaMissense baseline

In [16]:
!wget -q --show-progress https://storage.googleapis.com/dm_alphamissense/AlphaMissense_hg38.tsv.gz
!ls -lh AlphaMissense_hg38.tsv.gz

AlphaMissense_hg38. 100%[===================>] 613.18M  60.1MB/s    in 6.0s    
-rw-r--r-- 1 root root 614M Aug  3  2023 AlphaMissense_hg38.tsv.gz


In [17]:
import pandas as pd

if "out" not in globals():
    out = pd.read_csv("/content/drive/MyDrive/biohack/variant_table.csv", dtype={"variant_id": str})
out["variant_id"] = out["variant_id"].astype(str)
ids = set(out["variant_id"])

COLS = ["VariationID", "Assembly", "Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]
parts = []
for chunk in pd.read_csv("variant_summary.txt.gz", sep="\t", usecols=COLS, chunksize=500_000, dtype=str):
    parts.append(chunk[(chunk["Assembly"] == "GRCh38") & (chunk["VariationID"].isin(ids))])

coords = (pd.concat(parts, ignore_index=True)
          .drop_duplicates("VariationID")
          .rename(columns={"VariationID": "variant_id", "Chromosome": "chrom", "PositionVCF": "pos",
                           "ReferenceAlleleVCF": "ref", "AlternateAlleleVCF": "alt"}))

out = out.drop(columns=[c for c in ["chrom", "pos", "ref", "alt"] if c in out.columns])
out = out.merge(coords[["variant_id", "chrom", "pos", "ref", "alt"]], on="variant_id", how="left")

print("Variants with DNA positions:", out["pos"].notna().sum(), "of", len(out))
print(out[["variant_id", "gene", "ref_aa", "position", "alt_aa", "chrom", "pos", "ref", "alt"]].head())

Variants with DNA positions: 15907 of 15907
  variant_id    gene ref_aa  position alt_aa chrom        pos ref alt
0       1335  CEP290      W         7      C    12   88141287   C   A
1       2355   USH2A      C       319      Y     1  216325492   C   T
2       2356   USH2A      C       759      F     1  216247118   C   A
3       2359   USH2A      C       419      F     1  216324240   C   A
4       2362   USH2A      R      4674      G     1  215671085   T   C


In [18]:
import gzip

# find the header line (the file starts with a few comment lines)
with gzip.open("AlphaMissense_hg38.tsv.gz", "rt") as f:
    for i, line in enumerate(f):
        if line.startswith("#CHROM"):
            header_idx, header = i, line.lstrip("#").strip().split("\t")
            break

out["key"] = "chr" + out["chrom"].astype(str) + ":" + out["pos"].astype(str) + ":" + out["ref"] + ":" + out["alt"]
keys = set(out["key"])

parts = []
for chunk in pd.read_csv("AlphaMissense_hg38.tsv.gz", sep="\t", skiprows=header_idx + 1,
                         names=header, chunksize=2_000_000, dtype=str):
    k = chunk["CHROM"] + ":" + chunk["POS"] + ":" + chunk["REF"] + ":" + chunk["ALT"]
    hit = k.isin(keys)
    if hit.any():
        parts.append(chunk[hit].assign(key=k[hit]))
am = pd.concat(parts, ignore_index=True).drop_duplicates("key")

out = out.drop(columns=[c for c in ["uniprot_id", "protein_variant", "am_pathogenicity", "am_class"] if c in out.columns])
out = out.merge(am[["key", "uniprot_id", "protein_variant", "am_pathogenicity", "am_class"]], on="key", how="left")
out["am_pathogenicity"] = pd.to_numeric(out["am_pathogenicity"])

# residue check
pv = out["protein_variant"].fillna("")
out["uniprot_ref"] = pv.str[0]
out["uniprot_alt"] = pv.str[-1]
out["uniprot_pos"] = pd.to_numeric(pv.str[1:-1], errors="coerce")
has = out["protein_variant"].notna()
same_aa = (out["uniprot_ref"] == out["ref_aa"]) & (out["uniprot_alt"] == out["alt_aa"])
out["residue_check"] = "no_am_score"
out.loc[has & same_aa & (out["uniprot_pos"] == out["position"]), "residue_check"] = "exact"
out.loc[has & same_aa & (out["uniprot_pos"] != out["position"]), "residue_check"] = "numbering_offset"
out.loc[has & ~same_aa, "residue_check"] = "aa_mismatch"

print(out["residue_check"].value_counts())
print()
print("Test variants with an AlphaMissense score:",
      out.loc[out["split"] == "test", "am_pathogenicity"].notna().sum(), "of", (out["split"] == "test").sum())
print()
print("UniProt ID per gene (for the protein teammate):")
print(out.dropna(subset=["uniprot_id"]).groupby("gene")["uniprot_id"].agg(lambda s: s.value_counts().index[0]).to_string())

out.to_csv("/content/drive/MyDrive/biohack/variant_table.csv", index=False)
print("\nSaved")

residue_check
exact               13321
numbering_offset     1406
no_am_score          1178
aa_mismatch             2
Name: count, dtype: int64

Test variants with an AlphaMissense score: 135 of 145

UniProt ID per gene (for the protein teammate):
gene
ABCA4     P78363
BEST1     O76090
CEP290    O15078
CRX       O43186
GLA       P06280
GUCY2D    Q02846
KCNH2     Q12809
KCNJ2     P63252
KCNQ1     P51787
LMNA      P02545
MYBPC3    Q14896
MYH7      P12883
NR2E3     Q9Y5X4
OPA1      O60313
PAX6      P26367
PKP2      Q99959
RHO       P08100
RPE65     Q16518
RPGR      Q92834
SCN5A     Q14524
USH2A     O75445

Saved


In [19]:
from sklearn.metrics import roc_auc_score, average_precision_score
import math

def wilson(k, n, z=1.96):
    if n == 0: return ("n/a", "n/a")
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (round((c-h)*100, 1), round((c+h)*100, 1))

def evaluate(df, label):
    d = df.dropna(subset=["am_pathogenicity"])
    y = (d["class_now"] == "P").astype(int); s = d["am_pathogenicity"]
    npos, nneg = int(y.sum()), int((1 - y).sum())
    print(f"--- {label}: {len(d)} variants ({npos} pathogenic, {nneg} benign)")
    if y.nunique() == 2:
        print(f"AUROC {roc_auc_score(y, s):.3f} | AUPRC {average_precision_score(y, s):.3f}")
    tp = int(((s > 0.564) & (y == 1)).sum())
    tn = int(((s < 0.34) & (y == 0)).sum())
    print(f"Sensitivity: {tp}/{npos} = {tp/npos:.1%}, 95% CI {wilson(tp, npos)}")
    if nneg:
        print(f"Specificity: {tn}/{nneg} = {tn/nneg:.1%}, 95% CI {wilson(tn, nneg)}")
    print("Ambiguous scores:", int(((s >= 0.34) & (s <= 0.564)).sum()))
    print()

out = out[out["residue_check"] != "aa_mismatch"]
evaluate(out[out["split"] == "test"], "Test (2+ stars)")
evaluate(out[out["split"].isin(["test", "test_1star"])], "Test incl. 1-star")

vus = out[out["class_2023"] == "VUS"].copy()
vus["later_P"] = (vus["split"] == "test") & (vus["class_now"] == "P")
ranked = vus.dropna(subset=["am_pathogenicity"]).sort_values("am_pathogenicity", ascending=False)
n, hits = len(ranked), int(ranked["later_P"].sum()); base = hits / n
print(f"--- Ranking {n} of {len(vus)} 2023 VUS; {hits} later proven pathogenic (base rate {base:.2%})")
for k in [100, 250, 500, 1000]:
    h = int(ranked["later_P"].head(k).sum())
    print(f"Top {k}: {h} found | random expects {k*base:.1f} | enrichment {h/(k*base):.1f}x")

out.to_csv("/content/drive/MyDrive/biohack/variant_table.csv", index=False)

--- Test (2+ stars): 134 variants (122 pathogenic, 12 benign)
AUROC 0.941 | AUPRC 0.993
Sensitivity: 84/122 = 68.9%, 95% CI (60.2, 76.4)
Specificity: 11/12 = 91.7%, 95% CI (64.6, 98.5)
Ambiguous scores: 17

--- Test incl. 1-star: 294 variants (261 pathogenic, 33 benign)
AUROC 0.950 | AUPRC 0.992
Sensitivity: 200/261 = 76.6%, 95% CI (71.1, 81.4)
Specificity: 30/33 = 90.9%, 95% CI (76.4, 96.9)
Ambiguous scores: 33

--- Ranking 10173 of 11010 2023 VUS; 122 later proven pathogenic (base rate 1.20%)
Top 100: 3 found | random expects 1.2 | enrichment 2.5x
Top 250: 11 found | random expects 3.0 | enrichment 3.7x
Top 500: 18 found | random expects 6.0 | enrichment 3.0x
Top 1000: 34 found | random expects 12.0 | enrichment 2.8x


## 5. gnomAD benign negatives and the ancestry check

In [20]:
import requests, time

GNOMAD = "https://gnomad.broadinstitute.org/api"
Q = """
query GeneVariants($gene: String!) {
  gene(gene_symbol: $gene, reference_genome: GRCh38) {
    variants(dataset: gnomad_r4) {
      variant_id
      consequence
      exome { faf95 { popmax popmax_population } }
      genome { faf95 { popmax popmax_population } }
    }
  }
}
"""
RECESSIVE_OR_MIXED = {"ABCA4", "USH2A", "CEP290", "RPE65", "NR2E3", "GUCY2D", "BEST1", "KCNQ1"}

rows = []
for g in sorted(out["gene"].dropna().unique()):
    data = None
    for attempt in range(3):
        r = requests.post(GNOMAD, json={"query": Q, "variables": {"gene": g}}, timeout=180)
        if r.status_code == 200 and "errors" not in r.json():
            data = r.json()["data"]["gene"]["variants"]; break
        print(g, "retrying:", r.status_code, r.text[:150]); time.sleep(20)
    if data is None:
        print(g, "FAILED, skipping"); continue
    n_mis = 0
    for v in data:
        if v["consequence"] != "missense_variant": continue
        n_mis += 1
        fafs = [x["faf95"]["popmax"] for x in (v["exome"], v["genome"])
                if x and x.get("faf95") and x["faf95"]["popmax"] is not None]
        rows.append({"gene": g, "gnomad_id": v["variant_id"], "faf95_popmax": max(fafs) if fafs else 0.0})
    print(f"{g}: {n_mis} missense variants")
    time.sleep(6)

gn = pd.DataFrame(rows)
gn["threshold"] = gn["gene"].map(lambda g: 0.05 if g in RECESSIVE_OR_MIXED else 0.001)
gn["key"] = "chr" + gn["gnomad_id"].str.replace("-", ":", regex=False)
benign_freq = gn[(gn["faf95_popmax"] >= gn["threshold"]) &
                 (~gn["key"].isin(set(out.loc[out["class_now"] == "P", "key"])))].copy()

print("\nFrequency-based benign variants:", len(benign_freq))
print(benign_freq.groupby("gene").size().to_string())
benign_freq.to_csv("/content/drive/MyDrive/biohack/gnomad_benign_negatives.csv", index=False)
print("Saved")

ABCA4: 3262 missense variants
BEST1: 1123 missense variants
CACNA1C: 3303 missense variants
CEP290: 5292 missense variants
CRX: 392 missense variants
GLA: 407 missense variants
GUCY2D: 2237 missense variants
KCNH2: 2388 missense variants
KCNJ2: 361 missense variants
KCNQ1: 1714 missense variants
LMNA: 1285 missense variants
MYBPC3: 2666 missense variants
MYH7: 1987 missense variants
NR2E3: 941 missense variants
OPA1: 1551 missense variants
PAX6: 776 missense variants
PKP2: 1448 missense variants
PRPH2: 493 missense variants
RHO: 451 missense variants
RPE65: 720 missense variants
RPGR: 2074 missense variants
SCN5A: 2631 missense variants
USH2A: 7444 missense variants

Frequency-based benign variants: 0
Series([], )
Saved


In [21]:
print("Diagnosis: nonzero frequencies in first run:", int((gn["faf95_popmax"] > 0).sum()), "of", len(gn))

Q2 = """
query GeneVariants($gene: String!) {
  gene(gene_symbol: $gene, reference_genome: GRCh38) {
    variants(dataset: gnomad_r4) {
      variant_id
      consequence
      exome { ac an populations { id ac an } }
      genome { ac an populations { id ac an } }
    }
  }
}
"""

def grpmax(v, min_an=2000, min_ac=3):
    tot = {}
    for part in (v.get("exome"), v.get("genome")):
        if not part: continue
        for p in part.get("populations") or []:
            pid = p["id"]
            if "_" in pid or pid in ("XX", "XY"):  # skip sex-specific subgroups
                continue
            a = tot.setdefault(pid, [0, 0])
            a[0] += p["ac"] or 0; a[1] += p["an"] or 0
    best, best_pop = 0.0, None
    for pid, (ac, an) in tot.items():
        if an >= min_an and ac >= min_ac and ac / an > best:
            best, best_pop = ac / an, pid
    return best, best_pop

rows = []
for g in sorted(out["gene"].dropna().unique()):
    data = None
    for attempt in range(3):
        r = requests.post(GNOMAD, json={"query": Q2, "variables": {"gene": g}}, timeout=180)
        if r.status_code == 200 and "errors" not in r.json():
            data = r.json()["data"]["gene"]["variants"]; break
        print(g, "retrying:", r.status_code, r.text[:150]); time.sleep(20)
    if data is None:
        print(g, "FAILED, skipping"); continue
    for v in data:
        if v["consequence"] != "missense_variant": continue
        af, pop = grpmax(v)
        rows.append({"gene": g, "gnomad_id": v["variant_id"], "grpmax_af": af, "grpmax_pop": pop})
    print(g, "done")
    time.sleep(6)

gn = pd.DataFrame(rows)
print("\nVariants with frequency > 0:", int((gn["grpmax_af"] > 0).sum()), "of", len(gn))
gn["threshold"] = gn["gene"].map(lambda g: 0.05 if g in RECESSIVE_OR_MIXED else 0.001)
gn["key"] = "chr" + gn["gnomad_id"].str.replace("-", ":", regex=False)
pathogenic_keys = set(out.loc[out["class_now"] == "P", "key"])
benign_freq = gn[(gn["grpmax_af"] >= gn["threshold"]) & (~gn["key"].isin(pathogenic_keys))].copy()

Diagnosis: nonzero frequencies in first run: 0 of 44946
ABCA4 done
BEST1 done
CACNA1C done
CEP290 done
CRX done
GLA done
GUCY2D done
KCNH2 done
KCNJ2 done
KCNQ1 done
LMNA done
MYBPC3 done
MYH7 done
NR2E3 done
OPA1 done
PAX6 done
PKP2 done
PRPH2 done
RHO done
RPE65 done
RPGR done
SCN5A done
USH2A done

Variants with frequency > 0: 11822 of 44946


In [22]:
gn["threshold"] = gn["gene"].map(lambda g: 0.05 if g in RECESSIVE_OR_MIXED else 0.001)
gn["key"] = "chr" + gn["gnomad_id"].str.replace("-", ":", regex=False)
pathogenic_keys = set(out.loc[out["class_now"] == "P", "key"])
gn["clinvar_pathogenic"] = gn["key"].isin(pathogenic_keys)

print("How many variants pass each frequency (excluding ClinVar pathogenic):")
ok = gn[~gn["clinvar_pathogenic"]]
summary_af = pd.DataFrame({
    ">=0.1%": ok[ok["grpmax_af"] >= 0.001].groupby("gene").size(),
    ">=1%": ok[ok["grpmax_af"] >= 0.01].groupby("gene").size(),
    ">=5%": ok[ok["grpmax_af"] >= 0.05].groupby("gene").size(),
}).fillna(0).astype(int)
summary_af["recessive_or_mixed"] = summary_af.index.isin(RECESSIVE_OR_MIXED)
print(summary_af.to_string())

benign_freq = ok[ok["grpmax_af"] >= ok["threshold"]].copy()
print("\nFrequency-based benign variants (using our thresholds):", len(benign_freq))
print("Common variants that ClinVar calls pathogenic (excluded):", int((gn["clinvar_pathogenic"] & (gn["grpmax_af"] >= gn["threshold"])).sum()))
benign_freq.to_csv("/content/drive/MyDrive/biohack/gnomad_benign_negatives.csv", index=False)
print("Saved")

How many variants pass each frequency (excluding ClinVar pathogenic):
         >=0.1%  >=1%  >=5%  recessive_or_mixed
gene                                           
ABCA4        78    22     8                True
BEST1        21     8     4                True
CACNA1C      36    11     4               False
CEP290       58    12     4                True
CRX           9     4     2               False
GLA           6     1     1               False
GUCY2D       28     6     4                True
KCNH2        21     5     2               False
KCNQ1        19     7     1                True
LMNA         15     3     2               False
MYBPC3       43    12     2               False
MYH7         12     1     0               False
NR2E3        16     5     2                True
OPA1         21     6     2               False
PAX6         10     3     1               False
PKP2         43     9     1               False
PRPH2        19     4     3               False
RHO          10   

In [23]:
cv = out[["key", "class_now", "stars_now"]]
bf = benign_freq.merge(cv, on="key", how="left")
bf["clinvar_now"] = bf["class_now"].fillna("not in table")
print("ClinVar labels of the frequency-based benign variants:")
print(pd.crosstab(bf["gene"], bf["clinvar_now"]).to_string())

drop = bf["clinvar_now"].isin(["conflicting", "other"])
print("\nRemoving (conflicting or other in ClinVar):", int(drop.sum()))
print(bf.loc[drop, ["gene", "gnomad_id", "grpmax_af", "grpmax_pop", "clinvar_now"]].to_string(index=False))
bf = bf[~drop].copy()

keys_b = set(bf["key"])
parts = []
for chunk in pd.read_csv("AlphaMissense_hg38.tsv.gz", sep="\t", skiprows=header_idx + 1,
                         names=header, chunksize=2_000_000, dtype=str):
    k = chunk["CHROM"] + ":" + chunk["POS"] + ":" + chunk["REF"] + ":" + chunk["ALT"]
    hit = k.isin(keys_b)
    if hit.any():
        parts.append(chunk[hit].assign(key=k[hit]))
amb = pd.concat(parts).drop_duplicates("key")[["key", "uniprot_id", "protein_variant", "am_pathogenicity"]]
bf = bf.merge(amb, on="key", how="left")
bf["am_pathogenicity"] = pd.to_numeric(bf["am_pathogenicity"])

s = bf["am_pathogenicity"].dropna()
n, tn = len(s), int((s < 0.34).sum())
print(f"\nAlphaMissense on {n} frequency-based benign variants:")
print(f"Called likely benign: {tn}/{n} = {tn/n:.1%}, 95% CI {wilson(tn, n)}")
print(f"Ambiguous: {int(((s >= 0.34) & (s <= 0.564)).sum())} | Called likely pathogenic: {int((s > 0.564).sum())}")

tp_scores = out[(out["split"] == "test") & (out["class_now"] == "P")]["am_pathogenicity"].dropna()
y = [1] * len(tp_scores) + [0] * n
print("AUROC, test pathogenic vs frequency-based benign:",
      round(roc_auc_score(y, list(tp_scores) + list(s)), 3))

bf.to_csv("/content/drive/MyDrive/biohack/gnomad_benign_negatives.csv", index=False)
print("Saved")

ClinVar labels of the frequency-based benign variants:
clinvar_now   B  VUS  conflicting  not in table
gene                                           
ABCA4         5    0            2             1
BEST1         1    0            0             3
CACNA1C      10    0           14            12
CEP290        4    0            0             0
CRX           6    0            3             0
GLA           3    0            1             2
GUCY2D        4    0            0             0
KCNH2         7    0           13             1
KCNQ1         1    0            0             0
LMNA          1    0            5             9
MYBPC3       21    0           21             1
MYH7          8    1            3             0
NR2E3         2    0            0             0
OPA1          6    0            6             9
PAX6          0    1            1             8
PKP2         13    0           23             7
PRPH2         5    0           14             0
RHO           3    2            5

In [24]:
conf = benign_freq.merge(cv, on="key", how="left")
conf = conf[conf["class_now"] == "conflicting"].merge(
    out[["key", "protein_variant", "ref_aa", "position", "alt_aa"]], on="key", how="left")

print("Common variants still 'conflicting' in ClinVar:", len(conf))
print("\nAncestry group where each is most common:")
print(conf["grpmax_pop"].value_counts().to_string())
non_eur = conf["grpmax_pop"].isin(["afr", "amr", "eas", "sas", "mid"])
print(f"\nMost common in African, Latino, East Asian, South Asian, or Middle Eastern groups: "
      f"{int(non_eur.sum())}/{len(conf)} = {non_eur.mean():.0%}")
print("\nMost common ones:")
print(conf.sort_values("grpmax_af", ascending=False)[
    ["gene", "protein_variant", "grpmax_af", "grpmax_pop"]].head(10).to_string(index=False))

conf.to_csv("/content/drive/MyDrive/biohack/common_but_conflicting.csv", index=False)
print("\nSaved")

Common variants still 'conflicting' in ClinVar: 149

Ancestry group where each is most common:
grpmax_pop
asj          30
sas          28
afr          26
eas          19
fin          13
mid          12
nfe          11
amr           9
remaining     1

Most common in African, Latino, East Asian, South Asian, or Middle Eastern groups: 94/149 = 63%

Most common ones:
   gene protein_variant  grpmax_af grpmax_pop
   RPGR             NaN   0.154766        fin
  SCN5A          S1103Y   0.079994        afr
  KCNH2          R1047L   0.070249        fin
  ABCA4           R212H   0.069520        afr
  ABCA4          N1868I   0.067328        nfe
  SCN5A          V1951L   0.037156        amr
  SCN5A          S1787N   0.019927        asj
CACNA1C             NaN   0.015723        sas
  KCNH2           R148W   0.014795        asj
   RPGR             NaN   0.013766        eas

Saved


In [25]:
from scipy.stats import fisher_exact

allc = benign_freq.merge(cv, on="key", how="left")
allc["clinvar_now"] = allc["class_now"].fillna("not in table")
allc["non_eur"] = allc["grpmax_pop"].isin(["afr", "amr", "eas", "sas", "mid"])

print(allc.groupby("clinvar_now")["non_eur"].agg(share_non_european="mean", n="size").round(2).to_string())

c = allc[allc["clinvar_now"] == "conflicting"]["non_eur"]
b = allc[allc["clinvar_now"] == "B"]["non_eur"]
table = [[int(c.sum()), int((~c).sum())], [int(b.sum()), int((~b).sum())]]
odds, p = fisher_exact(table)
print(f"\nConflicting: {c.mean():.0%} non-European | Benign: {b.mean():.0%} non-European")
print(f"Odds ratio {odds:.2f}, Fisher exact p = {p:.2g}")

              share_non_european    n
clinvar_now                          
B                           0.74  173
VUS                         1.00    5
conflicting                 0.63  149
not in table                0.55   71

Conflicting: 63% non-European | Benign: 74% non-European
Odds ratio 0.60, Fisher exact p = 0.04


## 6. Transparent model: features, layers, and the time-machine test

In [26]:
!pip -q install biopython
from Bio.Align import substitution_matrices
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

B62 = substitution_matrices.load("BLOSUM62")
KD = dict(A=1.8,R=-4.5,N=-3.5,D=-3.5,C=2.5,Q=-3.5,E=-3.5,G=-0.4,H=-3.2,I=4.5,L=3.8,K=-3.9,M=1.9,F=2.8,P=-1.6,S=-0.8,T=-0.7,W=-0.9,Y=-1.3,V=4.2)
VOL = dict(A=88.6,R=173.4,N=114.1,D=111.1,C=108.5,Q=143.8,E=138.4,G=60.1,H=153.2,I=166.7,L=166.7,K=168.6,M=162.9,F=189.9,P=112.7,S=89.0,T=116.1,W=227.8,Y=193.6,V=140.0)
CHG = dict(D=-1, E=-1, K=1, R=1, H=0.1)
GENE_LIST = sorted(out["gene"].dropna().unique())

def featurize(df, ref, alt, gene, use_am):
    r, a = df[ref], df[alt]
    X = pd.DataFrame(index=df.index)
    X["d_hydrophobicity"] = a.map(KD) - r.map(KD)
    X["abs_d_hydrophobicity"] = X["d_hydrophobicity"].abs()
    X["d_volume"] = a.map(VOL) - r.map(VOL)
    X["abs_d_volume"] = X["d_volume"].abs()
    X["d_charge"] = a.map(lambda x: CHG.get(x, 0)) - r.map(lambda x: CHG.get(x, 0))
    X["abs_d_charge"] = X["d_charge"].abs()
    X["blosum62"] = [B62[x][y] for x, y in zip(r, a)]
    X["to_proline"] = (a == "P").astype(int); X["from_proline"] = (r == "P").astype(int)
    X["from_glycine"] = (r == "G").astype(int); X["to_glycine"] = (a == "G").astype(int)
    X["cysteine_change"] = ((r == "C") | (a == "C")).astype(int)
    for g in GENE_LIST:
        X["gene_" + g] = (df[gene] == g).astype(int)
    if use_am:
        X["am_missing"] = df["am_pathogenicity"].isna().astype(int)
        X["alphamissense"] = df["am_pathogenicity"].fillna(0.5)
    return X

tr = out[out["split"] == "train"].dropna(subset=["ref_aa", "alt_aa"])
ytr = (tr["class_2023"] == "P").astype(int)
models = {}
for name, use_am in [("physchem (no AlphaMissense)", False), ("physchem + AlphaMissense", True)]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
    m.fit(featurize(tr, "ref_aa", "alt_aa", "gene", use_am), ytr)
    models[name] = (m, use_am)

# test set: same 134 variants AlphaMissense can score
te = out[(out["split"] == "test") & out["am_pathogenicity"].notna()]
yte = (te["class_now"] == "P").astype(int)
# frequency-based benign variants: use AlphaMissense's protein change
fb = bf.dropna(subset=["protein_variant", "am_pathogenicity"]).copy()
fb["ref_aa"], fb["alt_aa"] = fb["protein_variant"].str[0], fb["protein_variant"].str[-1]
vus = out[(out["class_2023"] == "VUS") & out["am_pathogenicity"].notna()].copy()
vus["later_P"] = (vus["split"] == "test") & (vus["class_now"] == "P")
base = vus["later_P"].mean()

def score(name, df):
    if name == "AlphaMissense alone":
        return df["am_pathogenicity"].values
    m, use_am = models[name]
    return m.predict_proba(featurize(df, "ref_aa", "alt_aa", "gene", use_am))[:, 1]

print(f"{'model':32s} {'AUROC test':>10s} {'AUROC vs freq-benign':>21s} {'top-500 hits':>13s}")
for name in ["AlphaMissense alone", "physchem (no AlphaMissense)", "physchem + AlphaMissense"]:
    a1 = roc_auc_score(yte, score(name, te))
    tp_s = score(name, te[yte == 1]); fb_s = score(name, fb)
    a2 = roc_auc_score([1]*len(tp_s) + [0]*len(fb_s), list(tp_s) + list(fb_s))
    vus["s"] = score(name, vus)
    hits = int(vus.sort_values("s", ascending=False)["later_P"].head(500).sum())
    print(f"{name:32s} {a1:10.3f} {a2:21.3f} {hits:7d} ({hits/(500*base):.1f}x)")

m = models["physchem (no AlphaMissense)"][0]
coef = pd.Series(m[-1].coef_[0], index=featurize(tr.head(1), "ref_aa", "alt_aa", "gene", False).columns)
print("\nWhat the no-AlphaMissense model learned (standardized weights; + means more likely pathogenic):")
print(coef[[c for c in coef.index if not c.startswith("gene_")]].sort_values().round(2).to_string())

model                            AUROC test  AUROC vs freq-benign  top-500 hits
AlphaMissense alone                   0.941                 0.931      18 (3.0x)
physchem (no AlphaMissense)           0.835                 0.803      12 (2.0x)
physchem + AlphaMissense              0.969                 0.945      29 (4.8x)

What the no-AlphaMissense model learned (standardized weights; + means more likely pathogenic):
blosum62               -0.49
d_hydrophobicity       -0.12
to_glycine             -0.11
d_volume               -0.07
abs_d_hydrophobicity    0.01
from_proline            0.07
d_charge                0.11
abs_d_volume            0.17
from_glycine            0.17
abs_d_charge            0.26
cysteine_change         0.34
to_proline              0.51


In [27]:
def featurize2(df, use_am, use_gene=True, use_chem=True):
    X = featurize(df, "ref_aa", "alt_aa", "gene", use_am)
    if not use_gene: X = X[[c for c in X.columns if not c.startswith("gene_")]]
    if not use_chem: X = X[[c for c in X.columns if c.startswith("gene_") or c in ("alphamissense", "am_missing")]]
    return X

extra = {"physchem + AM, no gene": (True, False, True), "gene only": (False, True, False)}
for name, (use_am, use_gene, use_chem) in extra.items():
    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
    m.fit(featurize2(tr, use_am, use_gene, use_chem), ytr)
    models[name] = (m, (use_am, use_gene, use_chem))

def score2(name, df):
    if name == "AlphaMissense alone": return df["am_pathogenicity"].values
    m, flags = models[name]
    if isinstance(flags, bool): flags = (flags, True, True)
    return m.predict_proba(featurize2(df, *flags))[:, 1]

names = ["AlphaMissense alone", "physchem + AlphaMissense", "physchem + AM, no gene", "gene only"]
for name in names:
    vus["s"] = score2(name, vus)
    hits = int(vus.sort_values("s", ascending=False)["later_P"].head(500).sum())
    print(f"{name:28s} AUROC test {roc_auc_score(yte, score2(name, te)):.3f} | top-500 hits {hits} ({hits/(500*base):.1f}x)")

rng = np.random.default_rng(0)
s_am, s_new = score2("AlphaMissense alone", te), score2("physchem + AlphaMissense", te)
diffs = []
for _ in range(2000):
    i = rng.integers(0, len(te), len(te)); y = yte.values[i]
    if y.min() != y.max():
        diffs.append(roc_auc_score(y, s_new[i]) - roc_auc_score(y, s_am[i]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"\nAUROC gain over AlphaMissense: {np.mean(diffs):+.3f}, 95% CI {lo:+.3f} to {hi:+.3f}")

v_am, v_new, lp = score2("AlphaMissense alone", vus), score2("physchem + AlphaMissense", vus), vus["later_P"].values
hd = []
for _ in range(500):
    i = rng.integers(0, len(vus), len(vus))
    top = lambda s: lp[i][np.argsort(-s[i])[:500]].sum()
    hd.append(top(v_new) - top(v_am))
lo, hi = np.percentile(hd, [2.5, 97.5])
print(f"Top-500 hit gain over AlphaMissense: {np.mean(hd):+.1f}, 95% CI {lo:+.1f} to {hi:+.1f}")

AlphaMissense alone          AUROC test 0.941 | top-500 hits 18 (3.0x)
physchem + AlphaMissense     AUROC test 0.969 | top-500 hits 29 (4.8x)
physchem + AM, no gene       AUROC test 0.945 | top-500 hits 14 (2.3x)
gene only                    AUROC test 0.758 | top-500 hits 3 (0.5x)

AUROC gain over AlphaMissense: +0.028, 95% CI -0.014 to +0.092
Top-500 hit gain over AlphaMissense: +10.3, 95% CI +0.0 to +21.0


## 7. Which disease? Prediction from the nearest known variants

In [28]:
from collections import Counter

def groups(s):
    return [g for g in str(s).split("|") if g and g != "nan"]

trP = out[(out["split"] == "train") & (out["class_2023"] == "P")].copy()
trP["groups"] = trP["group_2023"].map(groups)
trP = trP[trP["groups"].str.len() > 0]
teP = out[(out["split"] == "test") & (out["class_now"] == "P")].copy()
teP["groups"] = teP["group_now"].map(groups)
teP = teP[teP["groups"].str.len() > 0]
for d in (trP, teP):
    d["pos"] = d["uniprot_pos"].fillna(d["position"])

rows = []
for _, v in teP.iterrows():
    t = trP[trP["gene"] == v["gene"]]
    cnt = Counter(g for gs in t["groups"] for g in gs)
    if len(cnt) < 2:
        continue
    majority = cnt.most_common(1)[0][0]
    near = t.assign(d=(t["pos"] - v["pos"]).abs()).sort_values("d").head(5)
    nearest = Counter(g for gs in near["groups"] for g in gs).most_common(1)[0][0]
    rows.append(dict(gene=v["gene"], truth="|".join(v["groups"]), choices=len(cnt),
                     majority_ok=majority in v["groups"], nearest5_ok=nearest in v["groups"],
                     chance=1 / len(cnt)))
res = pd.DataFrame(rows)

n = len(res)
k_maj, k_near = int(res["majority_ok"].sum()), int(res["nearest5_ok"].sum())
print("Test variants where the model must choose among 2+ diseases:", n)
print(f"Random chance:        {res['chance'].mean():.0%}")
print(f"Most common disease:  {k_maj}/{n} = {k_maj/n:.0%}, 95% CI {wilson(k_maj, n)}")
print(f"Nearest 5 in sequence: {k_near}/{n} = {k_near/n:.0%}, 95% CI {wilson(k_near, n)}")
print()
print(res.groupby("gene").agg(n=("truth", "size"), choices=("choices", "first"),
                              majority=("majority_ok", "mean"), nearest5=("nearest5_ok", "mean")).round(2).to_string())
res.to_csv("/content/drive/MyDrive/biohack/disease_prediction_baselines.csv", index=False)

Test variants where the model must choose among 2+ diseases: 89
Random chance:        31%
Most common disease:  71/89 = 80%, 95% CI (70.3, 86.8)
Nearest 5 in sequence: 79/89 = 89%, 95% CI (80.5, 93.8)

          n  choices  majority  nearest5
gene                                    
ABCA4     7        6      0.14      0.86
BEST1     4        5      0.50      0.75
CACNA1C   1        4      1.00      1.00
CRX       1        3      1.00      1.00
GLA       6        2      1.00      1.00
GUCY2D    7        5      1.00      1.00
KCNH2     2        2      1.00      1.00
KCNQ1     6        3      1.00      1.00
LMNA      3        7      1.00      1.00
MYH7     11        6      1.00      1.00
RHO       1        3      1.00      1.00
RPE65    11        2      1.00      1.00
RPGR      2        3      1.00      1.00
SCN5A     1        3      0.00      1.00
USH2A    26        3      0.65      0.69


In [29]:
from scipy.stats import mannwhitneyu

single = res[~res["truth"].str.contains(r"\|")]
n1 = len(single)
k1, k2 = int(single["majority_ok"].sum()), int(single["nearest5_ok"].sum())
print(f"Single-disease test variants: {n1}")
print(f"Random chance:         {single['chance'].mean():.0%}")
print(f"Most common disease:   {k1}/{n1} = {k1/n1:.0%}, 95% CI {wilson(k1, n1)}")
print(f"Nearest 5 in sequence: {k2}/{n1} = {k2/n1:.0%}, 95% CI {wilson(k2, n1)}")
print(single.groupby("gene").agg(n=("truth", "size"), majority=("majority_ok", "mean"),
                                 nearest5=("nearest5_ok", "mean")).round(2).to_string())

u = trP[trP["gene"] == "USH2A"].copy()
u["usher"] = u["groups"].map(lambda g: "Usher syndrome" in g)
u["rp_only"] = u["groups"].map(lambda g: "Retinitis pigmentosa" in g and "Usher syndrome" not in g)
a = u.loc[u["usher"], "am_pathogenicity"].dropna()
b = u.loc[u["rp_only"], "am_pathogenicity"].dropna()
stat, p = mannwhitneyu(a, b, alternative="two-sided")
print(f"\nUSH2A (2023 training data), AlphaMissense damage score:")
print(f"Usher syndrome: median {a.median():.2f} (n={len(a)}) | RP only: median {b.median():.2f} (n={len(b)}) | Mann-Whitney p = {p:.2g}")

Single-disease test variants: 48
Random chance:         30%
Most common disease:   30/48 = 62%, 95% CI (48.4, 74.8)
Nearest 5 in sequence: 39/48 = 81%, 95% CI (68.1, 89.8)
         n  majority  nearest5
gene                          
ABCA4    6       0.0      1.00
BEST1    4       0.5      0.75
GLA      6       1.0      1.00
GUCY2D   1       1.0      1.00
KCNH2    2       1.0      1.00
KCNQ1    6       1.0      1.00
LMNA     2       1.0      1.00
MYH7     6       1.0      1.00
RHO      1       1.0      1.00
RPE65    1       1.0      1.00
RPGR     2       1.0      1.00
SCN5A    1       0.0      1.00
USH2A   10       0.1      0.20

USH2A (2023 training data), AlphaMissense damage score:
Usher syndrome: median 0.67 (n=59) | RP only: median 0.57 (n=52) | Mann-Whitney p = 0.21


In [30]:
from scipy.stats import binomtest

VAGUE = {"Macular dystrophy"}
clean = lambda gs: [g for g in gs if g not in VAGUE]
trP2 = trP.assign(groups=trP["groups"].map(clean)); trP2 = trP2[trP2["groups"].str.len() > 0]
teP2 = teP.assign(groups=teP["groups"].map(clean)); teP2 = teP2[teP2["groups"].str.len() > 0]

def run(trd, ted):
    rows = []
    for _, v in ted.iterrows():
        t = trd[trd["gene"] == v["gene"]]
        cnt = Counter(g for gs in t["groups"] for g in gs)
        if len(cnt) < 2: continue
        maj = cnt.most_common(1)[0][0]
        nb = t.assign(d=(t["pos"] - v["pos"]).abs()).sort_values("d").head(5)
        near = Counter(g for gs in nb["groups"] for g in gs).most_common(1)[0][0]
        rows.append(dict(gene=v["gene"], truth="|".join(v["groups"]), single=len(v["groups"]) == 1,
                         majority=maj, nearest=near, majority_ok=maj in v["groups"],
                         nearest5_ok=near in v["groups"], chance=1 / len(cnt)))
    return pd.DataFrame(rows)

r2 = run(trP2, teP2)
s2 = r2[r2["single"]]
n = len(s2); k1, k2 = int(s2["majority_ok"].sum()), int(s2["nearest5_ok"].sum())
print(f"Single-disease test variants: {n} | chance {s2['chance'].mean():.0%}")
print(f"Most common disease:   {k1}/{n} = {k1/n:.0%}, 95% CI {wilson(k1, n)}")
print(f"Nearest 5 in sequence: {k2}/{n} = {k2/n:.0%}, 95% CI {wilson(k2, n)}")

b = int((s2["nearest5_ok"] & ~s2["majority_ok"]).sum())
c = int((~s2["nearest5_ok"] & s2["majority_ok"]).sum())
p = binomtest(b, b + c, 0.5).pvalue if b + c else float("nan")
print(f"When they disagree: nearest right {b} times, most-common right {c} times, paired p = {p:.2g}")

print("\nPer gene:")
print(s2.groupby("gene").agg(n=("truth", "size"), majority=("majority_ok", "mean"),
                             nearest5=("nearest5_ok", "mean")).round(2).to_string())
print("\nABCA4 detail:")
print(s2[s2["gene"] == "ABCA4"][["truth", "majority", "nearest", "nearest5_ok"]].to_string(index=False))

Single-disease test variants: 48 | chance 32%
Most common disease:   31/48 = 65%, 95% CI (50.4, 76.6)
Nearest 5 in sequence: 39/48 = 81%, 95% CI (68.1, 89.8)
When they disagree: nearest right 9 times, most-common right 1 times, paired p = 0.021

Per gene:
         n  majority  nearest5
gene                          
ABCA4    6      0.17      1.00
BEST1    4      0.50      0.75
GLA      6      1.00      1.00
GUCY2D   1      1.00      1.00
KCNH2    2      1.00      1.00
KCNQ1    6      1.00      1.00
LMNA     2      1.00      1.00
MYH7     6      1.00      1.00
RHO      1      1.00      1.00
RPE65    1      1.00      1.00
RPGR     2      1.00      1.00
SCN5A    1      0.00      1.00
USH2A   10      0.10      0.20

ABCA4 detail:
               truth             majority              nearest  nearest5_ok
   Stargardt disease Retinitis pigmentosa    Stargardt disease         True
   Stargardt disease Retinitis pigmentosa    Stargardt disease         True
   Stargardt disease Retinitis pigme

## 8. Today's ranked list of uncertain variants

In [31]:
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)

# 1. Retrain on everything known today
cur_lab = out[out["class_now"].isin(["P", "B"]) & (out["stars_now"] >= 1)].dropna(subset=["ref_aa", "alt_aa"])
y_now = (cur_lab["class_now"] == "P").astype(int)
final = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
final.fit(featurize(cur_lab, "ref_aa", "alt_aa", "gene", True), y_now)
print(f"Trained on {len(cur_lab)} variants classified today ({int(y_now.sum())} pathogenic, {int((1 - y_now).sum())} benign)")

# 2. Score today's uncertain variants
unc = out[out["class_now"].isin(["VUS", "conflicting"])].dropna(subset=["ref_aa", "alt_aa"]).copy()
unc["our_score"] = final.predict_proba(featurize(unc, "ref_aa", "alt_aa", "gene", True))[:, 1]
unc["percentile"] = (unc["our_score"].rank(pct=True) * 100).round(1)

# 3. Likely disease from the 5 nearest known pathogenic variants
known = out[(out["class_now"] == "P") & (out["group_now"].fillna("") != "")].copy()
known["groups"] = known["group_now"].map(groups).map(clean)
known = known[known["groups"].str.len() > 0]
for d in (known, unc):
    d["pos"] = d["uniprot_pos"].fillna(d["position"])
by_gene = {g: k for g, k in known.groupby("gene")}

def predict_disease(v):
    k = by_gene.get(v["gene"])
    if k is None or k.empty:
        return pd.Series({"predicted_disease": "", "support": "", "nearest_known": ""})
    nb = k.assign(d=(k["pos"] - v["pos"]).abs()).sort_values("d").head(5)
    top, n = Counter(g for gs in nb["groups"] for g in gs).most_common(1)[0]
    near = "; ".join(f"{r.ref_aa}{int(r.position)}{r.alt_aa} ({'/'.join(r.groups)}, {int(r.d)} aa away)"
                     for r in nb.head(3).itertuples())
    return pd.Series({"predicted_disease": top, "support": f"{n}/5 nearest", "nearest_known": near})

unc = pd.concat([unc, unc.apply(predict_disease, axis=1)], axis=1)

# 4. Suggested confirmation test by protein type
ASSAY = {
    "SCN5A": "patch clamp (sodium current)", "KCNQ1": "patch clamp (potassium current)",
    "KCNH2": "patch clamp (potassium current)", "KCNJ2": "patch clamp (potassium current)",
    "CACNA1C": "patch clamp (calcium current)", "BEST1": "patch clamp (chloride current)",
    "RHO": "trafficking + light-activation assay", "ABCA4": "trafficking + ATPase activity",
    "PRPH2": "expression + localization", "USH2A": "expression + localization",
    "GLA": "enzyme activity (+ migalastat response)", "RPE65": "isomerase activity",
    "GUCY2D": "guanylate cyclase activity", "CRX": "transcription reporter assay",
    "NR2E3": "transcription reporter assay", "PAX6": "transcription reporter assay",
    "MYH7": "myosin ATPase / motility assay", "MYBPC3": "expression + stability",
    "LMNA": "nuclear localization", "PKP2": "desmosome localization",
    "OPA1": "GTPase + mitochondrial fusion", "CEP290": "cilia localization", "RPGR": "cilia localization",
}

# 5. Save and show
unc["protein_change"] = "p." + unc["ref_aa"] + unc["position"].astype(int).astype(str) + unc["alt_aa"]
unc["suggested_assay"] = unc["gene"].map(ASSAY)
unc["clinvar_link"] = "https://www.ncbi.nlm.nih.gov/clinvar/variation/" + unc["variant_id"].astype(str) + "/"
ranked = unc.sort_values("our_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)
cols = ["rank", "gene", "protein_change", "class_now", "our_score", "percentile", "am_pathogenicity",
        "predicted_disease", "support", "nearest_known", "suggested_assay", "clinvar_link"]
ranked[cols].to_csv("/content/drive/MyDrive/biohack/todays_uncertain_variants_ranked.csv", index=False)

print(f"\nRanked {len(ranked)} uncertain variants. Saved todays_uncertain_variants_ranked.csv\n")
print("Top 15:")
print(ranked[["rank", "gene", "protein_change", "class_now", "our_score", "am_pathogenicity",
              "predicted_disease", "support"]].head(15).round(3).to_string(index=False))
print("\nGenes in the top 100:")
print(ranked.head(100)["gene"].value_counts().to_string())
print("\nTop 3 in the demo genes:")
for g in ["ABCA4", "RPE65"]:
    print(f"\n{g}:")
    print(ranked[ranked["gene"] == g][["rank", "protein_change", "our_score", "predicted_disease",
                                        "support", "nearest_known"]].head(3).round(3).to_string(index=False))

Trained on 3053 variants classified today (2639 pathogenic, 414 benign)

Ranked 12423 uncertain variants. Saved todays_uncertain_variants_ranked.csv

Top 15:
 rank  gene protein_change   class_now  our_score  am_pathogenicity                  predicted_disease     support
    1 BEST1        p.R150P conflicting      1.000             0.995                   Bestrophinopathy 4/5 nearest
    2 BEST1        p.R150C conflicting      1.000             0.883                   Bestrophinopathy 4/5 nearest
    3 BEST1        p.L319P conflicting      1.000             0.991 Best vitelliform macular dystrophy 4/5 nearest
    4   GLA        p.S201P         VUS      1.000             0.927                      Fabry disease 5/5 nearest
    5   GLA         p.C90R conflicting      1.000             0.942                      Fabry disease 5/5 nearest
    6   GLA         p.W44C conflicting      1.000             0.986                      Fabry disease 5/5 nearest
    7   GLA         p.W44C conflictin

In [32]:
kp = out[out["class_now"] == "P"].copy()
kp["pos"] = kp["uniprot_pos"].fillna(kp["position"])
kp_change = set(zip(kp["gene"], kp["ref_aa"], kp["pos"], kp["alt_aa"]))
kp_pos = {g: s.values for g, s in kp.groupby("gene")["pos"]}

def evidence(v):
    tags = []
    arr = kp_pos.get(v["gene"], np.array([]))
    if (v["gene"], v["ref_aa"], v["pos"], v["alt_aa"]) in kp_change:
        tags.append("PS1-like: same protein change already pathogenic")
    elif np.any(arr == v["pos"]):
        tags.append("PM5-like: different change at a known pathogenic residue")
    n5 = int(((arr >= v["pos"] - 5) & (arr <= v["pos"] + 5)).sum())
    if n5 >= 3:
        tags.append(f"hotspot: {n5} known pathogenic within 5 residues")
    return "; ".join(tags)

dedup = (ranked.groupby(["gene", "protein_change"], as_index=False)
         .agg(dna_variants=("variant_id", "size"), our_score=("our_score", "max"),
              **{c: (c, "first") for c in ["ref_aa", "alt_aa", "pos", "class_now", "am_pathogenicity",
                                            "predicted_disease", "support", "nearest_known",
                                            "suggested_assay", "clinvar_link"]}))
dedup = dedup.sort_values("our_score", ascending=False).reset_index(drop=True)
dedup.insert(0, "rank", dedup.index + 1)
dedup["rank_in_gene"] = dedup.groupby("gene").cumcount() + 1
dedup["evidence"] = dedup.apply(evidence, axis=1)

cols = ["rank", "rank_in_gene", "gene", "protein_change", "dna_variants", "class_now", "am_pathogenicity",
        "predicted_disease", "support", "evidence", "nearest_known", "suggested_assay", "clinvar_link"]
dedup[cols].to_csv("/content/drive/MyDrive/biohack/todays_uncertain_variants_ranked.csv", index=False)

print(f"{len(dedup)} unique protein changes (from {len(ranked)} DNA variants). Saved.\n")
print("How many carry each kind of evidence:")
for t in ["PS1-like", "PM5-like", "hotspot"]:
    print(f"  {t}: {int(dedup['evidence'].str.contains(t).sum())} overall, "
          f"{int(dedup.head(100)['evidence'].str.contains(t).sum())} in the top 100")
print("\n#1 in each gene:")
print(dedup[dedup["rank_in_gene"] == 1][["gene", "protein_change", "rank", "class_now",
                                          "predicted_disease", "support", "evidence"]].to_string(index=False))

12345 unique protein changes (from 12423 DNA variants). Saved.

How many carry each kind of evidence:
  PS1-like: 26 overall, 0 in the top 100
  PM5-like: 796 overall, 27 in the top 100
  hotspot: 2015 overall, 51 in the top 100

#1 in each gene:
   gene protein_change  rank   class_now             predicted_disease     support                                                                                                evidence
  BEST1        p.R150P     1 conflicting              Bestrophinopathy 4/5 nearest                                                                                                        
    GLA        p.S201P     4         VUS                 Fabry disease 5/5 nearest                                                           hotspot: 7 known pathogenic within 5 residues
  ABCA4       p.R2040P     9         VUS             Stargardt disease 4/5 nearest PM5-like: different change at a known pathogenic residue; hotspot: 7 known pathogenic within 5 residues
   LM

In [33]:
top = dedup.head(100)
for name, d in [("Top 100", top), ("All uncertain", dedup)]:
    print(f"{name:14s} | from arginine: {(d['ref_aa'] == 'R').mean():.0%} | to proline: {(d['alt_aa'] == 'P').mean():.0%} "
          f"| R->P or R->C: {((d['ref_aa'] == 'R') & d['alt_aa'].isin(['P', 'C'])).mean():.0%}")
trp = out[(out["split"] == "train") & (out["class_2023"] == "P")]
trb = out[(out["split"] == "train") & (out["class_2023"] == "B")]
for name, d in [("Known pathogenic", trp), ("Known benign", trb)]:
    print(f"{name:14s} | from arginine: {(d['ref_aa'] == 'R').mean():.0%} | to proline: {(d['alt_aa'] == 'P').mean():.0%} "
          f"| R->P or R->C: {((d['ref_aa'] == 'R') & d['alt_aa'].isin(['P', 'C'])).mean():.0%}")

Top 100        | from arginine: 51% | to proline: 59% | R->P or R->C: 47%
All uncertain  | from arginine: 12% | to proline: 5% | R->P or R->C: 3%
Known pathogenic | from arginine: 15% | to proline: 9% | R->P or R->C: 4%
Known benign   | from arginine: 11% | to proline: 1% | R->P or R->C: 2%


## 9. Nearby known variants (PS1/PM5-like evidence) and ablations

In [35]:
out["pos"] = out["uniprot_pos"].fillna(out["position"])
ref = out[out["split"] == "train"]
def index(cls):
    d = ref[ref["class_2023"] == cls]
    return {g: (s["pos"].values, s["variant_id"].astype(str).values) for g, s in d.groupby("gene")}
REF_P, REF_B = index("P"), index("B")
EMPTY = (np.array([]), np.array([]))

def add_neigh(df, w=5):
    X = featurize(df, "ref_aa", "alt_aa", "gene", True)
    same, nP, nB = [], [], []
    for g, p, vid in zip(df["gene"], df["pos"], df["variant_id"].astype(str)):
        pp, pid = REF_P.get(g, EMPTY); bp, bid = REF_B.get(g, EMPTY)
        mp, mb = pid != vid, bid != vid          # never count the variant itself
        same.append(int(((pp == p) & mp).sum()))
        nP.append(int(((np.abs(pp - p) <= w) & mp).sum()))
        nB.append(int(((np.abs(bp - p) <= w) & mb).sum()))
    X["same_residue_pathogenic"] = np.minimum(same, 3)
    X["log_nearby_pathogenic"] = np.log1p(nP)
    X["log_nearby_benign"] = np.log1p(nB)
    return X

tr = out[out["split"] == "train"].dropna(subset=["ref_aa", "alt_aa"])
ytr = (tr["class_2023"] == "P").astype(int)
m_nb = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
m_nb.fit(add_neigh(tr), ytr)

te = out[(out["split"] == "test") & out["am_pathogenicity"].notna()]
yte = (te["class_now"] == "P").astype(int)
fb["pos"] = pd.to_numeric(fb["protein_variant"].str[1:-1]); fb["variant_id"] = "gnomad"
vus = out[(out["class_2023"] == "VUS") & out["am_pathogenicity"].notna()].copy()
vus["later_P"] = (vus["split"] == "test") & (vus["class_now"] == "P")
base = vus["later_P"].mean()

def sc(name, df):
    if name == "AlphaMissense alone": return df["am_pathogenicity"].values
    if name == "+ nearby known variants": return m_nb.predict_proba(add_neigh(df))[:, 1]
    m, use_am = models["physchem + AlphaMissense"]
    return m.predict_proba(featurize(df, "ref_aa", "alt_aa", "gene", True))[:, 1]

print(f"{'model':28s} {'AUROC test':>10s} {'vs gnomAD benign':>17s} {'top-500':>9s} {'top-100 R->P/C':>15s}")
for name in ["AlphaMissense alone", "physchem + AlphaMissense", "+ nearby known variants"]:
    a1 = roc_auc_score(yte, sc(name, te))
    tps, fbs = sc(name, te[yte == 1]), sc(name, fb)
    a2 = roc_auc_score([1] * len(tps) + [0] * len(fbs), list(tps) + list(fbs))
    vus["s"] = sc(name, vus); r = vus.sort_values("s", ascending=False)
    hits = int(r["later_P"].head(500).sum())
    rpc = ((r.head(100)["ref_aa"] == "R") & r.head(100)["alt_aa"].isin(["P", "C"])).mean()
    print(f"{name:28s} {a1:10.3f} {a2:17.3f} {hits:4d} ({hits/(500*base):.1f}x) {rpc:14.0%}")

rng = np.random.default_rng(1)
s_am, s_nb = sc("AlphaMissense alone", te), sc("+ nearby known variants", te)
d = []
for _ in range(2000):
    i = rng.integers(0, len(te), len(te)); y = yte.values[i]
    if y.min() != y.max(): d.append(roc_auc_score(y, s_nb[i]) - roc_auc_score(y, s_am[i]))
print(f"\nAUROC gain over AlphaMissense: {np.mean(d):+.3f}, 95% CI {np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}")
v_am, v_nb, lp = sc("AlphaMissense alone", vus), sc("+ nearby known variants", vus), vus["later_P"].values
hd = []
for _ in range(500):
    i = rng.integers(0, len(vus), len(vus))
    hd.append(lp[i][np.argsort(-v_nb[i])[:500]].sum() - lp[i][np.argsort(-v_am[i])[:500]].sum())
print(f"Top-500 hit gain over AlphaMissense: {np.mean(hd):+.1f}, 95% CI {np.percentile(hd, 2.5):+.1f} to {np.percentile(hd, 97.5):+.1f}")

model                        AUROC test  vs gnomAD benign   top-500  top-100 R->P/C
AlphaMissense alone               0.941             0.931   18 (3.0x)             4%
physchem + AlphaMissense          0.969             0.945   29 (4.8x)            19%
+ nearby known variants           0.974             0.970   36 (6.0x)            13%

AUROC gain over AlphaMissense: +0.034, 95% CI -0.017 to +0.115
Top-500 hit gain over AlphaMissense: +18.0, 95% CI +7.0 to +29.0


In [36]:
def make_index(df):
    return {g: (s["pos"].values, s["variant_id"].astype(str).values) for g, s in df.groupby("gene")}

known_now = out[out["class_now"].isin(["P", "B"]) & (out["stars_now"] >= 1)]
DEP_P = make_index(known_now[known_now["class_now"] == "P"])
DEP_B = make_index(known_now[known_now["class_now"] == "B"])

def add_neigh_dep(df, w=5):
    X = featurize(df, "ref_aa", "alt_aa", "gene", True)
    same, nP, nB = [], [], []
    for g, p, vid in zip(df["gene"], df["pos"], df["variant_id"].astype(str)):
        pp, pid = DEP_P.get(g, EMPTY); bp, bid = DEP_B.get(g, EMPTY)
        mp, mb = pid != vid, bid != vid
        same.append(int(((pp == p) & mp).sum()))
        nP.append(int(((np.abs(pp - p) <= w) & mp).sum()))
        nB.append(int(((np.abs(bp - p) <= w) & mb).sum()))
    X["same_residue_pathogenic"] = np.minimum(same, 3)
    X["log_nearby_pathogenic"] = np.log1p(nP)
    X["log_nearby_benign"] = np.log1p(nB)
    return X

cur_lab = known_now.dropna(subset=["ref_aa", "alt_aa"])
final_nb = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
final_nb.fit(add_neigh_dep(cur_lab), (cur_lab["class_now"] == "P").astype(int))

unc2 = out[out["class_now"].isin(["VUS", "conflicting"])].dropna(subset=["ref_aa", "alt_aa"]).copy()
unc2["new_score"] = final_nb.predict_proba(add_neigh_dep(unc2))[:, 1]
unc2["protein_change"] = "p." + unc2["ref_aa"] + unc2["position"].astype(int).astype(str) + unc2["alt_aa"]
best = unc2.groupby(["gene", "protein_change"], as_index=False)["new_score"].max()

dedup2 = dedup.drop(columns=["rank", "rank_in_gene"]).merge(best, on=["gene", "protein_change"], how="left")
dedup2 = dedup2.sort_values("new_score", ascending=False).reset_index(drop=True)
dedup2.insert(0, "rank", dedup2.index + 1)
dedup2["rank_in_gene"] = dedup2.groupby("gene").cumcount() + 1

cols = ["rank", "rank_in_gene", "gene", "protein_change", "dna_variants", "class_now", "am_pathogenicity",
        "predicted_disease", "support", "evidence", "nearest_known", "suggested_assay", "clinvar_link"]
dedup2[cols].to_csv("/content/drive/MyDrive/biohack/todays_uncertain_variants_ranked.csv", index=False)

t = dedup2.head(100)
print(f"Top 100 now: R->P or R->C {((t['ref_aa'] == 'R') & t['alt_aa'].isin(['P', 'C'])).mean():.0%} (was 47%) | "
      f"PM5-like {int(t['evidence'].str.contains('PM5').sum())} | hotspot {int(t['evidence'].str.contains('hotspot').sum())}")
print("\nTop 10:")
print(dedup2[["rank", "gene", "protein_change", "class_now", "am_pathogenicity", "predicted_disease",
              "support", "evidence"]].head(10).round(3).to_string(index=False))
print("\n#1 in each gene:")
print(dedup2[dedup2["rank_in_gene"] == 1][["gene", "protein_change", "rank", "predicted_disease",
                                            "support", "evidence"]].to_string(index=False))
print("\nSaved")

Top 100 now: R->P or R->C 13% (was 47%) | PM5-like 92 | hotspot 98

Top 10:
 rank  gene protein_change   class_now  am_pathogenicity           predicted_disease     support                                                                                                 evidence
    1   GLA        p.S297C conflicting             0.787               Fabry disease 5/5 nearest PM5-like: different change at a known pathogenic residue; hotspot: 17 known pathogenic within 5 residues
    2 KCNQ1        p.R591P         VUS             0.997            Long QT syndrome 5/5 nearest PM5-like: different change at a known pathogenic residue; hotspot: 10 known pathogenic within 5 residues
    3 KCNH2        p.N629I         VUS             0.997            Long QT syndrome 5/5 nearest PM5-like: different change at a known pathogenic residue; hotspot: 17 known pathogenic within 5 residues
    4  LMNA         p.R89P conflicting             1.000         Charcot-Marie-Tooth 5/5 nearest  PM5-like: differen

In [37]:
vus["nb_score"] = sc("+ nearby known variants", vus)
vus["pct_ours"] = vus["nb_score"].rank(pct=True)
vus["pct_am"] = vus["am_pathogenicity"].rank(pct=True)
Xv = add_neigh(vus)
vus["n_near_P"] = np.expm1(Xv["log_nearby_pathogenic"].values).round().astype(int)

lp = vus[vus["later_P"]].copy()
lp["known_neighbors"] = np.where(lp["n_near_P"] > 0, "yes (1+ within 5)", "none within 5")
print("Where later-proven pathogenic variants ranked (median percentile; 1.0 = top):")
print(lp.groupby("known_neighbors").agg(n=("gene", "size"),
                                         ours=("pct_ours", "median"),
                                         alphamissense=("pct_am", "median")).round(3).to_string())
print(f"\nShare of all 2023 VUS with no known pathogenic variant within 5 residues: {(vus['n_near_P'] == 0).mean():.0%}")

Where later-proven pathogenic variants ranked (median percentile; 1.0 = top):
                    n   ours  alphamissense
known_neighbors                            
none within 5      24  0.660          0.672
yes (1+ within 5)  98  0.922          0.796

Share of all 2023 VUS with no known pathogenic variant within 5 residues: 62%


## 10. AlphaFold structures and 3D neighbors (tested as an ablation; used for explanations)

In [38]:
import requests

UNIPROT = out.dropna(subset=["uniprot_id"]).groupby("gene")["uniprot_id"].agg(lambda s: s.value_counts().index[0]).to_dict()
UNIPROT.update({"CACNA1C": "Q13936", "PRPH2": "P23942"})

coords = {}
for gene, acc in sorted(UNIPROT.items()):
    try:
        meta = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{acc}", timeout=60).json()
        pdb = requests.get(meta[0]["pdbUrl"], timeout=180).text
    except Exception as e:
        print(f"{gene} ({acc}): failed ({e})"); continue
    ca = {}
    for line in pdb.splitlines():
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            ca[int(line[22:26])] = (float(line[30:38]), float(line[38:46]), float(line[46:54]), float(line[60:66]))
    coords[gene] = ca
    plddt = np.array([v[3] for v in ca.values()])
    print(f"{gene} ({acc}): {len(ca)} residues, {np.mean(plddt >= 70):.0%} with pLDDT >= 70")

import pickle
pickle.dump(coords, open("/content/drive/MyDrive/biohack/alphafold_ca_coords.pkl", "wb"))
print("\nSaved alphafold_ca_coords.pkl")

ABCA4 (P78363): 2273 residues, 77% with pLDDT >= 70
BEST1 (O76090): 585 residues, 64% with pLDDT >= 70
CACNA1C (Q13936): 2221 residues, 49% with pLDDT >= 70
CEP290 (O15078): 2479 residues, 34% with pLDDT >= 70
CRX (O43186): 299 residues, 22% with pLDDT >= 70
GLA (P06280): 429 residues, 92% with pLDDT >= 70
GUCY2D (Q02846): 1103 residues, 83% with pLDDT >= 70
KCNH2 (Q12809): 1159 residues, 50% with pLDDT >= 70
KCNJ2 (P63252): 427 residues, 78% with pLDDT >= 70
KCNQ1 (P51787): 676 residues, 55% with pLDDT >= 70
LMNA (P02545): 664 residues, 69% with pLDDT >= 70
MYBPC3 (Q14896): 1274 residues, 79% with pLDDT >= 70
MYH7 (P12883): 1935 residues, 65% with pLDDT >= 70
NR2E3 (Q9Y5X4): 410 residues, 63% with pLDDT >= 70
OPA1 (O60313): 960 residues, 72% with pLDDT >= 70
PAX6 (P26367): 422 residues, 46% with pLDDT >= 70
PKP2 (Q99959): 881 residues, 51% with pLDDT >= 70
PRPH2 (P23942): 346 residues, 86% with pLDDT >= 70
RHO (P08100): 348 residues, 90% with pLDDT >= 70
RPE65 (Q16518): 533 residues, 

In [39]:
GENE_ARR = {}
for g, ca in coords.items():
    res = np.array(sorted(ca))
    GENE_ARR[g] = (res, np.array([ca[r][:3] for r in res]), np.array([ca[r][3] for r in res]))

def xyz_of(g, p):
    a = GENE_ARR.get(g)
    if a is None or pd.isna(p): return None, np.nan
    res, xyz, pl = a
    i = np.searchsorted(res, int(p))
    return (xyz[i], pl[i]) if i < len(res) and res[i] == int(p) else (None, np.nan)

def ref_xyz(REF):
    r3 = {}
    for g, (pp, pid) in REF.items():
        pts, ids, seq = [], [], []
        for p, vid in zip(pp, pid):
            c, pl = xyz_of(g, p)
            if c is not None and pl >= 70:
                pts.append(c); ids.append(vid); seq.append(p)
        r3[g] = (np.array(pts).reshape(-1, 3), np.array(ids), np.array(seq))
    return r3

R3_P, R3_B = ref_xyz(REF_P), ref_xyz(REF_B)   # 2023 knowledge only

def add_3d(df, radius=10.0):
    X = add_neigh(df)
    n3P, n3onlyP, n3B, contact, plddt = [], [], [], [], []
    for g, p, vid in zip(df["gene"], df["pos"], df["variant_id"].astype(str)):
        c, pl = xyz_of(g, p)
        if c is None or pl < 70:
            n3P.append(0); n3onlyP.append(0); n3B.append(0); contact.append(0)
            plddt.append(0 if c is None else pl); continue
        res, xyz, _ = GENE_ARR[g]
        contact.append(int((np.linalg.norm(xyz - c, axis=1) <= radius).sum()) - 1)
        plddt.append(pl)
        P, B = R3_P.get(g), R3_B.get(g)
        if P is not None and len(P[0]):
            m = (np.linalg.norm(P[0] - c, axis=1) <= radius) & (P[1] != vid)
            n3P.append(int(m.sum())); n3onlyP.append(int((m & (np.abs(P[2] - p) > 5)).sum()))
        else:
            n3P.append(0); n3onlyP.append(0)
        if B is not None and len(B[0]):
            n3B.append(int(((np.linalg.norm(B[0] - c, axis=1) <= radius) & (B[1] != vid)).sum()))
        else:
            n3B.append(0)
    X["log_3d_pathogenic"] = np.log1p(n3P)
    X["log_3d_only_pathogenic"] = np.log1p(n3onlyP)
    X["log_3d_benign"] = np.log1p(n3B)
    X["contact_number"] = contact
    X["plddt"] = plddt
    X["low_confidence"] = (np.array(plddt) < 70).astype(int)
    return X

m_3d = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
m_3d.fit(add_3d(tr), ytr)

def sc2(name, df):
    return m_3d.predict_proba(add_3d(df))[:, 1] if name == "+ 3D neighbors + burial" else sc(name, df)

names = ["AlphaMissense alone", "+ nearby known variants", "+ 3D neighbors + burial"]
print(f"{'model':26s} {'AUROC test':>10s} {'vs gnomAD benign':>17s} {'top-500':>10s} {'no-neighbor rank':>17s}")
no_nb = vus["n_near_P"] == 0
for name in names:
    a1 = roc_auc_score(yte, sc2(name, te))
    tps, fbs = sc2(name, te[yte == 1]), sc2(name, fb)
    a2 = roc_auc_score([1] * len(tps) + [0] * len(fbs), list(tps) + list(fbs))
    s = sc2(name, vus); vus["s"] = s; vus["pct"] = vus["s"].rank(pct=True)
    hits = int(vus.sort_values("s", ascending=False)["later_P"].head(500).sum())
    nn = vus.loc[vus["later_P"] & no_nb, "pct"].median()
    print(f"{name:26s} {a1:10.3f} {a2:17.3f} {hits:4d} ({hits/(500*base):.1f}x) {nn:17.3f}")

rng = np.random.default_rng(2)
v_am, v_3d, v_nb, lpv = (sc2("AlphaMissense alone", vus), sc2("+ 3D neighbors + burial", vus),
                         sc2("+ nearby known variants", vus), vus["later_P"].values)
g_am, g_nb = [], []
for _ in range(500):
    i = rng.integers(0, len(vus), len(vus)); top = lambda s: lpv[i][np.argsort(-s[i])[:500]].sum()
    g_am.append(top(v_3d) - top(v_am)); g_nb.append(top(v_3d) - top(v_nb))
print(f"\nTop-500 gain vs AlphaMissense: {np.mean(g_am):+.1f} (95% CI {np.percentile(g_am, 2.5):+.1f} to {np.percentile(g_am, 97.5):+.1f})")
print(f"Top-500 gain vs previous model: {np.mean(g_nb):+.1f} (95% CI {np.percentile(g_nb, 2.5):+.1f} to {np.percentile(g_nb, 97.5):+.1f})")

coef = pd.Series(m_3d[-1].coef_[0], index=add_3d(tr.head(3)).columns)
print("\nStructure weights (+ means more likely pathogenic):")
print(coef[["log_3d_pathogenic", "log_3d_only_pathogenic", "log_3d_benign", "contact_number", "plddt", "low_confidence"]].round(2).to_string())

model                      AUROC test  vs gnomAD benign    top-500  no-neighbor rank
AlphaMissense alone             0.941             0.931   18 (3.0x)             0.672
+ nearby known variants         0.974             0.970   36 (6.0x)             0.660
+ 3D neighbors + burial         0.980             0.970   36 (6.0x)             0.664

Top-500 gain vs AlphaMissense: +18.4 (95% CI +8.5 to +28.5)
Top-500 gain vs previous model: -0.2 (95% CI -3.0 to +2.0)

Structure weights (+ means more likely pathogenic):
log_3d_pathogenic         0.07
log_3d_only_pathogenic    0.19
log_3d_benign             0.00
contact_number           -0.05
plddt                     0.17
low_confidence           -0.05


In [40]:
import json

X3 = add_3d(vus)
has3d = np.expm1(X3["log_3d_only_pathogenic"].values).round() > 0
grp = (vus["later_P"] & no_nb).values
print(f"Later-proven pathogenic with no sequence neighbors: {int(grp.sum())}; "
      f"of these, with any 3D-only known neighbor: {int((grp & has3d).sum())}\n")

kpn = known_now[known_now["class_now"] == "P"].copy()
kpn["groups_txt"] = kpn["group_now"].fillna("").str.replace("|", "/", regex=False)

def neighbors_3d(g, p, radius=10.0):
    c, pl = xyz_of(g, p)
    if c is None: return [], np.nan, None
    res, xyz, _ = GENE_ARR[g]
    contact = int((np.linalg.norm(xyz - c, axis=1) <= radius).sum()) - 1
    found = []
    for r in kpn[kpn["gene"] == g].itertuples():
        c2, _ = xyz_of(g, r.pos)
        if c2 is None or r.pos == p: continue
        d = float(np.linalg.norm(c2 - c))
        if d <= radius:
            found.append((d, f"{r.ref_aa}{int(r.pos)}{r.alt_aa} ({r.groups_txt or 'pathogenic'}, {d:.1f} Å, {int(abs(r.pos - p))} aa apart)"))
    return [t for _, t in sorted(found)[:5]], pl, contact

TREATMENT = {
    "RPE65": "Luxturna requires confirmed disease-causing variants in both RPE65 copies, so resolving this VUS could affect gene therapy eligibility.",
    "RHO": "Dominant RHO disease: adding a working copy may not be enough; knockdown-and-replace approaches are in development.",
    "MYH7": "If confirmed: family screening; a myosin inhibitor (mavacamten) is approved for obstructive HCM.",
    "KCNQ1": "If confirmed: beta-blockers, avoiding QT-prolonging drugs, and family testing are standard for long QT 1.",
    "ABCA4": "No approved therapy for Stargardt; trials are ongoing, and patients are usually advised to avoid vitamin A supplements.",
}
DEMO = [("MYH7", "p.R403P"), ("RHO", "p.C187S"), ("RPE65", "p.H68P"), ("ABCA4", "p.R2040P"), ("KCNQ1", "p.R591P")]

cards = []
for g, pc in DEMO:
    row = dedup2[(dedup2["gene"] == g) & (dedup2["protein_change"] == pc)]
    if row.empty:
        print("Not found:", g, pc); continue
    r = row.iloc[0]
    nb3, pl, contact = neighbors_3d(g, r["pos"])
    cards.append({
        "gene": g, "variant": pc, "clinvar": r["clinvar_link"], "current_status": r["class_now"],
        "rank_overall": int(r["rank"]), "rank_in_gene": int(r["rank_in_gene"]),
        "alphamissense": None if pd.isna(r["am_pathogenicity"]) else round(float(r["am_pathogenicity"]), 3),
        "predicted_disease": r["predicted_disease"], "disease_support": r["support"],
        "evidence": r["evidence"], "nearest_in_sequence": r["nearest_known"], "nearest_in_3d": nb3,
        "plddt": None if pd.isna(pl) else round(float(pl), 1), "contact_number": contact,
        "suggested_assay": r["suggested_assay"], "treatment_context": TREATMENT.get(g, ""),
        "disclaimer": "Research use only. Prioritization, not a clinical classification.",
    })

for c in cards:
    print(f"=== {c['gene']} {c['variant']} ({c['current_status']}) | rank {c['rank_overall']} overall, #{c['rank_in_gene']} in gene")
    print(f"Predicted: {c['predicted_disease']} ({c['disease_support']}) | AlphaMissense {c['alphamissense']} | pLDDT {c['plddt']} | contacts {c['contact_number']}")
    print(f"Evidence: {c['evidence']}")
    print(f"Nearest in sequence: {c['nearest_in_sequence']}")
    print(f"Nearest in 3D: {'; '.join(c['nearest_in_3d']) or 'none within 10 Å'}")
    print(f"Assay: {c['suggested_assay']} | Treatment: {c['treatment_context']}\n")

json.dump(cards, open("/content/drive/MyDrive/biohack/demo_cards.json", "w"), indent=2)
print("Saved demo_cards.json")

Later-proven pathogenic with no sequence neighbors: 24; of these, with any 3D-only known neighbor: 5

=== MYH7 p.R403P (VUS) | rank 5 overall, #1 in gene
Predicted: Hypertrophic cardiomyopathy (5/5 nearest) | AlphaMissense 0.997 | pLDDT 85.5 | contacts 19
Evidence: PM5-like: different change at a known pathogenic residue; hotspot: 7 known pathogenic within 5 residues
Nearest in sequence: R403Q (Hypertrophic cardiomyopathy, 0 aa away); R403G (Hypertrophic cardiomyopathy, 0 aa away); R403L (Hypertrophic cardiomyopathy, 0 aa away)
Nearest in 3D: T412S (pathogenic, 4.4 Å, 9 aa apart); V411I (Dilated cardiomyopathy/Hypertrophic cardiomyopathy, 5.5 Å, 8 aa apart); T604A (pathogenic, 6.0 Å, 201 aa apart); N602K (Hypertrophic cardiomyopathy, 9.1 Å, 199 aa apart); N602S (Hypertrophic cardiomyopathy, 9.1 Å, 199 aa apart)
Assay: myosin ATPase / motility assay | Treatment: If confirmed: family screening; a myosin inhibitor (mavacamten) is approved for obstructive HCM.

=== RHO p.C187S (conflicting

## 11. FoldX physics on rhodopsin

In [41]:
!mkdir -p /content/foldx
!unzip -o -q /content/drive/MyDrive/biohack/foldx_Linux.zip -d /content/foldx
!ls -laR /content/foldx | head -40

/content/foldx:
total 113048
drwxr-xr-x 2 root root      4096 Nov  7 19:02 .
drwxr-xr-x 1 root root      4096 Nov  7 19:02 ..
-rwxrwxr-x 1 root root 113132176 Jan  6  2026 foldx_20261231
-rw-r--r-- 1 root root   2573870 Nov 15  2021 rotabase.txt
-rw-r--r-- 1 root root     36791 Jan  6  2026 yasaraPlugin.zip


In [42]:
%cd /content/foldx
!chmod +x foldx_20261231
!./foldx_20261231 2>&1 | head -15

import requests
meta = requests.get("https://alphafold.ebi.ac.uk/api/prediction/P08100", timeout=60).json()
open("RHO.pdb", "w").write(requests.get(meta[0]["pdbUrl"], timeout=120).text)
!ls -la RHO.pdb

!time ./foldx_20261231 --command=RepairPDB --pdb=RHO.pdb 2>&1 | tail -8
!ls -la RHO_Repair*

/content/foldx
   ********************************************
   ***                                      ***
   ***             FoldX 4 (c)              ***
   ***                                      ***
   ***     code by the FoldX Consortium     ***
   ***                                      ***
   ***     Jesper Borg, Frederic Rousseau   ***
   ***    Joost Schymkowitz, Luis Serrano   ***
   ***    Peter Vanhee, Erik Verschueren    ***
   ***     Lies Baeten, Javier Delgado      ***
   ***       and Francois Stricher          ***
   *** and any other of the 9! permutations ***
   ***   based on an original concept by    ***
   ***   Raphael Guerois and Luis Serrano   ***
   ********************************************
-rw-r--r-- 1 root root 227933 Nov  7 19:03 RHO.pdb
Total          = 				  -99.23

End of Repair
Output PDB ./RHO_Repair.pdb
Your file run OK
End time of FoldX: Sat Nov  7 19:06:34 2026
Total time spend: 173.52 seconds.
Cleaning RepairPDB...DONE

real	3m0.355s
user	

In [43]:
import io, math, subprocess

# 1. Pick 2 benign controls in confidently predicted regions
rho_b = out[(out["gene"] == "RHO") & (out["class_now"] == "B")].copy()
rho_b["plddt"] = [xyz_of("RHO", p)[1] for p in rho_b["pos"]]
rho_b = rho_b[rho_b["plddt"] >= 70].sort_values("plddt", ascending=False).head(2)
controls = [(r.ref_aa, int(r.pos), r.alt_aa) for r in rho_b.itertuples()]
muts = [("C", 187, "S"), ("P", 23, "H"), ("G", 90, "D")] + controls
labels = {("C", 187, "S"): "C187S (disulfide, RP)", ("P", 23, "H"): "P23H (misfolding, RP)",
          ("G", 90, "D"): "G90D (constitutively active, night blindness)"}
for c in controls: labels[c] = f"{c[0]}{c[1]}{c[2]} (benign control)"

# 2. Check the residues in the structure
AA3 = dict(ALA="A", ARG="R", ASN="N", ASP="D", CYS="C", GLN="Q", GLU="E", GLY="G", HIS="H", ILE="I",
           LEU="L", LYS="K", MET="M", PHE="F", PRO="P", SER="S", THR="T", TRP="W", TYR="Y", VAL="V")
seq = {int(l[22:26]): AA3.get(l[17:20], "?") for l in open("RHO_Repair.pdb") if l.startswith("ATOM") and l[12:16].strip() == "CA"}
for wt, p, mt in muts:
    print(f"{wt}{p}{mt}: structure has {seq.get(p)} at {p} -> {'OK' if seq.get(p) == wt else 'MISMATCH'}")
muts = [m for m in muts if seq.get(m[1]) == m[0]]

# 3. Run BuildModel (3 repeats each)
open("individual_list.txt", "w").write("".join(f"{wt}A{p}{mt};\n" for wt, p, mt in muts))
print("\nRunning FoldX BuildModel... (several minutes)")
subprocess.run("./foldx_20261231 --command=BuildModel --pdb=RHO_Repair.pdb "
               "--mutant-file=individual_list.txt --numberOfRuns=3 > buildmodel.log 2>&1", shell=True)

# 4. Read results: ddG and energy terms, averaged over repeats
lines = open("Dif_RHO_Repair.fxout").read().splitlines()
start = next(i for i, l in enumerate(lines) if l.startswith("Pdb"))
dif = pd.read_csv(io.StringIO("\n".join(lines[start:])), sep="\t")
dif["m"] = dif["Pdb"].str.extract(r"_(\d+)_\d+\.pdb$")[0].astype(int) - 1
dif["variant"] = dif["m"].map(lambda i: labels[muts[i]])
terms = [c for c in ["total energy", "Van der Waals", "Solvation Hydrophobic", "Solvation Polar",
                     "Backbone Hbond", "Sidechain Hbond", "Electrostatics", "Van der Waals clashes",
                     "disulfide"] if c in dif.columns]
summary_fx = dif.groupby("variant")[terms].mean().round(2)
summary_fx["ddG_sd"] = dif.groupby("variant")["total energy"].std().round(2)
print("\nΔΔG (kcal/mol; positive = destabilizing) and energy terms:")
print(summary_fx.sort_values("total energy", ascending=False).to_string())

# 5. Fraction folded, using f = 1 / (1 + exp(-ΔG/RT))
RT = 0.616
print("\nFraction folded at 37 °C for different baseline stabilities (ΔG_wt):")
for v, ddg in summary_fx["total energy"].sort_values(ascending=False).items():
    fr = [1 / (1 + math.exp(-(gwt - ddg) / RT)) for gwt in (3, 5, 8)]
    print(f"{v:48s} ΔΔG {ddg:+.2f} | ΔG_wt=3: {fr[0]:.0%}  ΔG_wt=5: {fr[1]:.0%}  ΔG_wt=8: {fr[2]:.0%}")
dif.to_csv("/content/drive/MyDrive/biohack/foldx_RHO_buildmodel.csv", index=False)

C187S: structure has C at 187 -> OK
P23H: structure has P at 23 -> OK
G90D: structure has G at 90 -> OK
V157I: structure has V at 157 -> OK
G51A: structure has G at 51 -> OK

Running FoldX BuildModel... (several minutes)

ΔΔG (kcal/mol; positive = destabilizing) and energy terms:
                                               total energy  Van der Waals  Solvation Hydrophobic  Solvation Polar  Backbone Hbond  Sidechain Hbond  Electrostatics  Van der Waals clashes  disulfide  ddG_sd
variant                                                                                                                                                                                                      
P23H (misfolding, RP)                                 16.63          -1.14                  -0.52             3.52           -0.01            -1.36           -0.00                  14.09       0.00    0.38
G90D (constitutively active, night blindness)          6.95          -1.17                  -1.13    

## 12. Feature table, UniProt regions, and the predictions export

In [44]:
import re

def mondo_ids(s):
    return "|".join(sorted(set(re.findall(r"MONDO:\d+", s)))) if isinstance(s, str) else ""

ft = out.copy()
use_up = ft["residue_check"].isin(["exact", "numbering_offset"])
ft["ref"] = np.where(use_up, ft["uniprot_ref"], ft["ref_aa"])
ft["alt"] = np.where(use_up, ft["uniprot_alt"], ft["alt_aa"])
ft["pos2"] = np.where(use_up, ft["uniprot_pos"], ft["position"]).astype(int)
ft["uniprot_id"] = ft["uniprot_id"].fillna(ft["gene"].map(UNIPROT))

feat = pd.DataFrame({
    "variant_id": ft["gene"] + ":" + ft["ref"] + ft["pos2"].astype(str) + ft["alt"],
    "variation_id": ft["variant_id"].astype(str),
    "gene": ft["gene"], "uniprot_id": ft["uniprot_id"],
    "pos": ft["pos2"], "ref": ft["ref"], "alt": ft["alt"],
    "numbering": np.where(use_up, "uniprot_canonical", "clinvar_unverified"),
    "split": ft["split"],
    "cls_2023": ft["class_2023"], "stars_2023": ft["stars_2023"],
    "diseases_2023": ft["pheno_ids_2023"].map(mondo_ids),
    "disease_groups_2023": ft["group_2023"].fillna(""),
    "cls_now": ft["class_now"], "stars_now": ft["stars_now"],          # TEST ANSWERS: never features/edges
    "diseases_now": ft["pheno_ids_now"].map(mondo_ids),                # TEST ANSWERS
    "disease_groups_now": ft["group_now"].fillna(""),                  # TEST ANSWERS
    "am_score": ft["am_pathogenicity"], "am_class": ft["am_class"],
    "plddt": [xyz_of(g, p)[1] for g, p in zip(ft["gene"], ft["pos2"])],
    "genomic_key": ft["key"],
})

feat.to_csv("/content/drive/MyDrive/biohack/feature_table.csv", index=False)
print("Rows:", len(feat), "| unique variation_id:", feat["variation_id"].nunique(),
      "| unique variant_id:", feat["variant_id"].nunique())
print(feat["numbering"].value_counts().to_string())
print(feat["split"].value_counts().to_string())
print("\nTrain rows with a 2023 MONDO disease:", int(((feat["split"] == "train") & (feat["diseases_2023"] != "")).sum()))
print(feat[feat["gene"] == "RHO"].head(3).to_string(index=False))
print("\nSaved feature_table.csv")

Rows: 15905 | unique variation_id: 15905 | unique variant_id: 15702
numbering
uniprot_canonical     14727
clinvar_unverified     1178
split
unlabeled     10685
train          2790
exclude        2105
test_1star      181
test            144

Train rows with a 2023 MONDO disease: 2004
variant_id variation_id gene uniprot_id  pos ref alt         numbering split cls_2023  stars_2023                                           diseases_2023                  disease_groups_2023 cls_now  stars_now                                                                        diseases_now                   disease_groups_now  am_score          am_class  plddt        genomic_key
  RHO:P23H        13013  RHO     P08100   23   P   H uniprot_canonical train        P           2 MONDO:0007639|MONDO:0012498|MONDO:0013395|MONDO:0019118 Night blindness|Retinitis pigmentosa       P          2                                           MONDO:0007639|MONDO:0012498|MONDO:0013395 Night blindness|Retinitis pigmentosa 

In [45]:
import os, glob, subprocess

path = next((p for p in ["/content/drive/MyDrive/biohack/Uniprot/uniprot_features.csv",
                         "/content/drive/MyDrive/biohack/uniprot_features.csv"] if os.path.exists(p)), None)
if path is None:
    from google.colab import files
    up = files.upload()                                   # choose Ellie's zip
    subprocess.run("unzip -o -q /content/*.zip -d /content/uni", shell=True)
    path = glob.glob("/content/uni/**/uniprot_features.csv", recursive=True)[0]
uf = pd.read_csv(path)
print("Loaded", len(uf), "UniProt features from", path)

FLAGS, TOPO, SITES = {}, {}, {}
for r in uf.itertuples():
    g, s, e = r.gene, int(r.start), int(r.end)
    if r.feature_type in ("Domain", "Transmembrane"):
        col = "in_domain" if r.feature_type == "Domain" else "transmembrane"
        for p in range(s, e + 1): FLAGS.setdefault((g, p), {})[col] = 1
    elif r.feature_type == "Topological domain":
        for p in range(s, e + 1): TOPO[(g, p)] = str(r.description).split(";")[0].strip().lower()
    elif r.feature_type == "Disulfide bond":                 # start/end are the two cysteines
        for p in (s, e):
            FLAGS.setdefault((g, p), {})["disulfide_cys"] = 1; SITES.setdefault(g, set()).add(p)
    elif r.feature_type in ("Binding site", "Active site"):
        col = "binding_site" if r.feature_type == "Binding site" else "active_site"
        for p in range(s, e + 1):
            FLAGS.setdefault((g, p), {})[col] = 1; SITES.setdefault(g, set()).add(p)
SITE_XYZ = {g: np.array([c for c in (xyz_of(g, p)[0] for p in ps) if c is not None]) for g, ps in SITES.items()}

UCOLS = ["in_domain", "transmembrane", "disulfide_cys", "binding_site", "active_site",
         "topo_extracellular", "topo_cytoplasmic", "topo_other", "dist_to_site"]
def uniprot_feats(df):
    rows = []
    for g, p in zip(df["gene"], df["pos"]):
        d = dict.fromkeys(UCOLS, 0); d["dist_to_site"] = 30.0
        if not pd.isna(p):
            p = int(p); d.update(FLAGS.get((g, p), {}))
            t = TOPO.get((g, p))
            if t:
                d["topo_extracellular" if "extracellular" in t else "topo_cytoplasmic" if "cytoplasmic" in t else "topo_other"] = 1
            c0, _ = xyz_of(g, p); S = SITE_XYZ.get(g)
            if c0 is not None and S is not None and len(S):
                d["dist_to_site"] = min(30.0, float(np.min(np.linalg.norm(S - c0, axis=1))))
        rows.append(d)
    return pd.DataFrame(rows, index=df.index)[UCOLS]

def add_up(df):
    return pd.concat([add_neigh(df), uniprot_feats(df)], axis=1)

m_up = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
m_up.fit(add_up(tr), ytr)

def sc3(name, df):
    return m_up.predict_proba(add_up(df))[:, 1] if name == "+ UniProt regions" else sc(name, df)

print(f"\n{'model':26s} {'AUROC test':>10s} {'vs gnomAD benign':>17s} {'top-500':>10s} {'no-neighbor rank':>17s}")
for name in ["AlphaMissense alone", "+ nearby known variants", "+ UniProt regions"]:
    a1 = roc_auc_score(yte, sc3(name, te))
    tps, fbs = sc3(name, te[yte == 1]), sc3(name, fb)
    a2 = roc_auc_score([1] * len(tps) + [0] * len(fbs), list(tps) + list(fbs))
    vus["s"] = sc3(name, vus); vus["pct"] = vus["s"].rank(pct=True)
    hits = int(vus.sort_values("s", ascending=False)["later_P"].head(500).sum())
    nn = vus.loc[vus["later_P"] & no_nb, "pct"].median()
    print(f"{name:26s} {a1:10.3f} {a2:17.3f} {hits:4d} ({hits/(500*base):.1f}x) {nn:17.3f}")

rng = np.random.default_rng(3)
v_up, v_nb, lpv = sc3("+ UniProt regions", vus), sc3("+ nearby known variants", vus), vus["later_P"].values
g = []
for _ in range(500):
    i = rng.integers(0, len(vus), len(vus)); top = lambda s: lpv[i][np.argsort(-s[i])[:500]].sum()
    g.append(top(v_up) - top(v_nb))
print(f"\nTop-500 gain vs previous model: {np.mean(g):+.1f} (95% CI {np.percentile(g, 2.5):+.1f} to {np.percentile(g, 97.5):+.1f})")

coef = pd.Series(m_up[-1].coef_[0], index=add_up(tr.head(3)).columns)
print("\nUniProt weights (+ means more likely pathogenic):")
print(coef[UCOLS].round(2).to_string())

uf_out = uniprot_feats(out)
for c in UCOLS: feat[c] = uf_out[c].values
feat.to_csv("/content/drive/MyDrive/biohack/feature_table.csv", index=False)
print("\nAdded region flags to feature_table.csv")
print("Variants flagged:", {c: int(feat[c].sum()) for c in UCOLS[:5]})

Loaded 406 UniProt features from /content/drive/MyDrive/biohack/Uniprot/uniprot_features.csv

model                      AUROC test  vs gnomAD benign    top-500  no-neighbor rank
AlphaMissense alone             0.941             0.931   18 (3.0x)             0.672
+ nearby known variants         0.974             0.970   36 (6.0x)             0.660
+ UniProt regions               0.973             0.970   39 (6.5x)             0.667

Top-500 gain vs previous model: +2.0 (95% CI -2.5 to +7.0)

UniProt weights (+ means more likely pathogenic):
in_domain             0.21
transmembrane        -0.13
disulfide_cys         0.24
binding_site          0.22
active_site           0.02
topo_extracellular    0.05
topo_cytoplasmic     -0.23
topo_other            0.01
dist_to_site          0.05

Added region flags to feature_table.csv
Variants flagged: {'in_domain': 4895, 'transmembrane': 1238, 'disulfide_cys': 104, 'binding_site': 70, 'active_site': 2}


In [46]:
import json

mondo_map = dict(zip(feat["variation_id"], feat["diseases_now"].fillna("")))
kn = out[(out["class_now"] == "P") & (out["stars_now"] >= 1)].copy()
kn["groups"] = kn["group_now"].fillna("").map(groups).map(clean)
kn["mondo"] = kn["variant_id"].astype(str).map(mondo_map).fillna("").str.split("|")
kn = kn[kn["groups"].str.len() > 0]
KN = {g: d for g, d in kn.groupby("gene")}

def disease_from_neighbors(gene, pos, vid, k=5):
    d = KN.get(gene)
    if d is None or pd.isna(pos): return {}, {}, ""
    d = d[d["variant_id"].astype(str) != str(vid)]          # never use the variant itself
    nb = d.assign(dist=(d["pos"] - pos).abs()).sort_values("dist").head(k)
    if nb.empty: return {}, {}, ""
    gc = Counter(x for gs in nb["groups"] for x in gs)
    mc = Counter(x for ms in nb["mondo"] for x in ms if x)
    near = "; ".join(f"{r.ref_aa}{int(r.position)}{r.alt_aa} ({'/'.join(r.groups)}, {int(r.dist)} aa)"
                     for r in nb.head(3).itertuples())
    return ({x: round(c / len(nb), 2) for x, c in gc.most_common()},
            {x: round(c / len(nb), 2) for x, c in mc.most_common(5)}, near)

def confidence(s, am):
    if 0.35 <= s <= 0.65: return "unsure"
    if pd.isna(am): return "medium"
    agree = (s > 0.65 and am > 0.564) or (s < 0.35 and am < 0.34)
    if not agree: return "unsure"
    return "high" if (s >= 0.9 or s <= 0.1) else "medium"

s_today = final_nb.predict_proba(add_neigh_dep(out))[:, 1]
s_2023 = m_nb.predict_proba(add_neigh(out))[:, 1]
vkey = dict(zip(feat["variation_id"], feat["variant_id"]))

recs = []
for i, (_, r) in enumerate(out.iterrows()):
    dg, dm, near = disease_from_neighbors(r["gene"], r["pos"], r["variant_id"])
    am = r["am_pathogenicity"]
    recs.append({
        "variant": vkey.get(str(r["variant_id"]), ""), "variation_id": str(r["variant_id"]), "gene": r["gene"],
        "harmful": round(float(s_today[i]), 4), "harmful_2023_model": round(float(s_2023[i]), 4),
        "alphamissense": None if pd.isna(am) else round(float(am), 4),
        "confidence": confidence(s_today[i], am),
        "mechanism": {}, "mechanism_status": "pending FoldX",
        "disease": dm, "disease_groups": dg,
        "treatment": TREATMENT.get(r["gene"], ""), "suggested_assay": ASSAY.get(r["gene"], ""),
        "evidence": evidence(r), "nearest_known": near,
        "clinvar_today": r["class_now"], "role_2023": r["split"],
        "in_training_today": bool(r["class_now"] in ("P", "B") and r["stars_now"] >= 1),
        "disclaimer": "Research use only. Prioritization, not a clinical classification.",
    })

json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)

p = pd.DataFrame(recs)
print("Exported", len(p), "predictions")
print("\nConfidence, uncertain variants only (VUS + conflicting today):")
print(p[p["clinvar_today"].isin(["VUS", "conflicting"])]["confidence"].value_counts().to_string())
print("\nTime-machine check (2023 test variants): 2023-model score by what experts later decided")
t = p[p["role_2023"] == "test"]
print(t.groupby("clinvar_today")["harmful_2023_model"].describe()[["count", "mean", "50%"]].round(3).to_string())
print("\nExample record:")
print(json.dumps(next(x for x in recs if x["variant"] == "RHO:C187S"), indent=2))

Exported 15905 predictions

Confidence, uncertain variants only (VUS + conflicting today):
confidence
high      7218
medium    2751
unsure    2454

Time-machine check (2023 test variants): 2023-model score by what experts later decided
               count   mean    50%
clinvar_today                     
B               20.0  0.092  0.043
P              124.0  0.830  0.983

Example record:
{
  "variant": "RHO:C187S",
  "variation_id": "1502009",
  "gene": "RHO",
  "harmful": 1.0,
  "harmful_2023_model": 0.9999,
  "alphamissense": 0.997,
  "confidence": "high",
  "mechanism": {},
  "mechanism_status": "pending FoldX",
  "disease": {
    "MONDO:0013395": 1.0,
    "MONDO:0019200": 0.2
  },
  "disease_groups": {
    "Retinitis pigmentosa": 1.0
  },
  "treatment": "Dominant RHO disease: adding a working copy may not be enough; knockdown-and-replace approaches are in development.",
  "suggested_assay": "trafficking + light-activation assay",
  "evidence": "PM5-like: different change at a kno

In [47]:
def confidence2(s, am, ev):
    if 0.35 <= s <= 0.65: return "unsure"
    if pd.isna(am): return "medium"
    agree = (s > 0.65 and am > 0.564) or (s < 0.35 and am < 0.34)
    if not agree: return "unsure"
    strong = (s >= 0.97 and any(t in ev for t in ("PS1", "PM5", "hotspot"))) or (s <= 0.03 and am < 0.2)
    return "high" if strong else "medium"

for x in recs:
    x["confidence"] = confidence2(x["harmful"], np.nan if x["alphamissense"] is None else x["alphamissense"], x["evidence"])
json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)
p = pd.DataFrame(recs)
print("Stricter confidence, uncertain variants only:")
print(p[p["clinvar_today"].isin(["VUS", "conflicting"])]["confidence"].value_counts().to_string())

# Time-machine candidates: test variants only, disease predicted from 2023 data only
o = out.set_index(out["variant_id"].astype(str))
tm = p[(p["role_2023"] == "test") & (p["clinvar_today"] == "P")].copy()
tm["pos"] = tm["variation_id"].map(o["pos"])
tm["truth"] = tm["variation_id"].map(o["group_now"]).fillna("").map(groups).map(clean)
pct = dict(zip(vus["variant_id"].astype(str), vus["pct_ours"]))
tm["rank_pct_2023"] = tm["variation_id"].map(pct)

def pred_2023(gene, pos):
    t = trP2[trP2["gene"] == gene]
    if t.empty or pd.isna(pos): return ""
    nb = t.assign(d=(t["pos"] - pos).abs()).sort_values("d").head(5)
    return Counter(g for gs in nb["groups"] for g in gs).most_common(1)[0][0]

tm["disease_2023_pred"] = [pred_2023(g, ps) for g, ps in zip(tm["gene"], tm["pos"])]
tm["disease_ok"] = [d in t for d, t in zip(tm["disease_2023_pred"], tm["truth"])]
tm = tm.sort_values(["disease_ok", "harmful_2023_model"], ascending=False)
cols = ["variant", "gene", "harmful_2023_model", "rank_pct_2023", "alphamissense",
        "disease_2023_pred", "truth", "disease_ok", "evidence"]
print("\nBest time-machine demo variants (uncertain in 2023, pathogenic now):")
print(tm[tm["truth"].str.len() > 0][cols].head(12).round(3).to_string(index=False))
tm[cols + ["variation_id"]].to_csv("/content/drive/MyDrive/biohack/time_machine_candidates.csv", index=False)
print("\nSaved predictions (updated) and time_machine_candidates.csv")


Stricter confidence, uncertain variants only:
confidence
medium    6060
high      3909
unsure    2454

Best time-machine demo variants (uncertain in 2023, pathogenic now):
    variant  gene  harmful_2023_model  rank_pct_2023  alphamissense           disease_2023_pred                                                                    truth  disease_ok                                                                                         evidence
KCNQ1:R591C KCNQ1               1.000          0.999          0.927            Long QT syndrome                                                       [Long QT syndrome]        True PS1-like: same protein change already pathogenic; hotspot: 10 known pathogenic within 5 residues
KCNQ1:E261K KCNQ1               1.000          0.997          0.999            Long QT syndrome                                                       [Long QT syndrome]        True PS1-like: same protein change already pathogenic; hotspot: 12 known pathogenic within 5 res

In [48]:
def make_ev_index(kpdf):
    change = {}
    for g, r, p, a, vid in zip(kpdf["gene"], kpdf["ref_aa"], kpdf["pos"], kpdf["alt_aa"], kpdf["variant_id"].astype(str)):
        change.setdefault((g, r, p, a), set()).add(vid)
    pos_ids = {g: (s["pos"].values, s["variant_id"].astype(str).values) for g, s in kpdf.groupby("gene")}
    return change, pos_ids

EV_TODAY = make_ev_index(out[out["class_now"] == "P"])
EV_2023 = make_ev_index(out[(out["split"] == "train") & (out["class_2023"] == "P")])

def evidence_x(gene, ref, pos, alt, vid, EV):
    change, pos_ids = EV
    vid = str(vid); tags = []
    arr, ids = pos_ids.get(gene, (np.array([]), np.array([])))
    m = ids != vid                                            # never count the variant itself
    if change.get((gene, ref, pos, alt), set()) - {vid}:
        tags.append("PS1-like: same protein change already pathogenic (different DNA change)")
    elif np.any((arr == pos) & m):
        tags.append("PM5-like: different change at a known pathogenic residue")
    n5 = int(((np.abs(arr - pos) <= 5) & m).sum())
    if n5 >= 3:
        tags.append(f"hotspot: {n5} known pathogenic within 5 residues")
    return "; ".join(tags)

for x, (_, r) in zip(recs, out.iterrows()):
    args = (r["gene"], r["ref_aa"], r["pos"], r["alt_aa"], r["variant_id"])
    x["evidence"] = evidence_x(*args, EV_TODAY)
    x["evidence_2023"] = evidence_x(*args, EV_2023)
    x["confidence"] = confidence2(x["harmful"], np.nan if x["alphamissense"] is None else x["alphamissense"], x["evidence"])

json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)

ev23 = {x["variation_id"]: x["evidence_2023"] for x in recs}
tm["evidence_2023"] = tm["variation_id"].map(ev23)
top = tm[(tm["truth"].str.len() > 0) & tm["disease_ok"]].head(12)
print("Time-machine candidates with 2023-only evidence:")
print(top[["variant", "harmful_2023_model", "rank_pct_2023", "disease_2023_pred", "evidence_2023"]]
      .round(3).to_string(index=False))

print("\nStory pairs: solved since 2023 -> uncertain today at the same residue")
for _, r in top.iterrows():
    same = dedup2[(dedup2["gene"] == r["gene"]) & (dedup2["pos"] == r["pos"])]
    for _, s in same.iterrows():
        print(f"  {r['variant']:13s} (2023 rank top {100 * (1 - r['rank_pct_2023']):.1f}%, now pathogenic)  ->  "
              f"{s['gene']} {s['protein_change']} today: rank #{int(s['rank'])} overall, #{int(s['rank_in_gene'])} in gene, {s['class_now']}")

tm.to_csv("/content/drive/MyDrive/biohack/time_machine_candidates.csv", index=False)
print("\nSaved")

Time-machine candidates with 2023-only evidence:
    variant  harmful_2023_model  rank_pct_2023           disease_2023_pred                                                                                            evidence_2023
KCNQ1:R591C               1.000          0.999            Long QT syndrome  PM5-like: different change at a known pathogenic residue; hotspot: 8 known pathogenic within 5 residues
KCNQ1:E261K               1.000          0.997            Long QT syndrome PM5-like: different change at a known pathogenic residue; hotspot: 10 known pathogenic within 5 residues
  GLA:D266A               1.000          0.998               Fabry disease PM5-like: different change at a known pathogenic residue; hotspot: 14 known pathogenic within 5 residues
  GLA:S297Y               1.000          0.999               Fabry disease PM5-like: different change at a known pathogenic residue; hotspot: 16 known pathogenic within 5 residues
KCNQ1:R243S               1.000          0.993     

## 13. Related genes (paralogs): matching residues across gene families

*The first version of this cell also re-loaded the January 2023 ClinVar file from the wrong folder and failed; that duplicate block was removed, and the next cell loads it correctly.*

In [49]:
FAMILIES = {
    "SCN5A": ["SCN1A", "SCN2A", "SCN3A", "SCN4A", "SCN8A", "SCN9A", "SCN10A", "SCN11A"],
    "CACNA1C": ["CACNA1A", "CACNA1B", "CACNA1D", "CACNA1E", "CACNA1F", "CACNA1S"],
    "KCNQ1": ["KCNQ2", "KCNQ3", "KCNQ4", "KCNQ5"],
    "KCNH2": ["KCNH1", "KCNH5", "KCNH6", "KCNH7"],
    "KCNJ2": ["KCNJ1", "KCNJ5", "KCNJ6", "KCNJ10", "KCNJ11", "KCNJ13"],
    "MYH7": ["MYH6", "MYH2", "MYH3", "MYH7B", "MYH8"],
    "MYBPC3": ["MYBPC1", "MYBPC2"],
    "PAX6": ["PAX2", "PAX3", "PAX7"],
    "CRX": ["OTX2"],
    "ABCA4": ["ABCA1", "ABCA3", "ABCA12"],
    "GLA": ["NAGA"],
    "PRPH2": ["ROM1"],
    "LMNA": ["LMNB1", "LMNB2"],
    "BEST1": ["BEST2", "BEST3"],
    "RHO": ["OPN1LW", "OPN1MW", "OPN1SW"],
    "GUCY2D": ["GUCY2F", "NPR2"],
}
PARALOGS = sorted({p for ps in FAMILIES.values() for p in ps})

def uniprot_seq(gene):
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": f"gene_exact:{gene} AND organism_id:9606 AND reviewed:true",
                             "fields": "accession,sequence", "format": "tsv"}, timeout=60)
    rows = r.text.strip().split("\n")[1:]
    return tuple(rows[0].split("\t")) if rows else (None, None)

SEQ = {}
um = pd.read_csv("/content/drive/MyDrive/biohack/Uniprot/uniprot_mapping.csv")
for g, s in zip(um["gene"], um["sequence"]): SEQ[g] = s
for g in PARALOGS:
    acc, s = uniprot_seq(g)
    if s: SEQ[g] = s
    else: print("No sequence for", g)
print("Sequences:", len(SEQ), "| paralogs found:", sum(g in SEQ for g in PARALOGS), "of", len(PARALOGS))

Sequences: 76 | paralogs found: 53 of 53


In [50]:
import os
os.chdir("/content")
CV23 = "/content/variant_summary_2023-01.txt.gz"
if not os.path.exists(CV23):
    subprocess.run("wget -q https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/2023/variant_summary_2023-01.txt.gz -O " + CV23, shell=True)

LOAD = ["Type", "Name", "GeneSymbol", "ClinicalSignificance", "ReviewStatus", "Assembly", "VariationID"]
pat2 = r"(?:^|;)(?:" + "|".join(PARALOGS) + r")(?:;|$)"
parts = []
for chunk in pd.read_csv(CV23, sep="\t", usecols=LOAD, chunksize=500_000, dtype=str):
    parts.append(chunk[(chunk["Assembly"] == "GRCh38") & chunk["GeneSymbol"].fillna("").str.contains(pat2, regex=True)])
pv = pd.concat(parts, ignore_index=True)
pv = pv[pv["Type"] == "single nucleotide variant"].copy()
m = pv["Name"].str.extract(PROT)
pv["ref"], pv["pos"], pv["alt"] = m[0].map(AA), pd.to_numeric(m[1], errors="coerce"), m[2].map(AA)
pv = pv.dropna(subset=["ref", "pos", "alt"])
pv = pv[pv["ref"] != pv["alt"]].copy()
pv["pos"] = pv["pos"].astype(int)
pv["gene"] = pv["GeneSymbol"].str.split(";").apply(lambda g: next((x for x in g if x in PARALOGS), None))
pv["cls"] = pv["ClinicalSignificance"].map(simple_class)
pv["stars"] = pv["ReviewStatus"].map(stars)
pv = pv[(pv["cls"] == "P") & (pv["stars"] >= 1)]
pv["seq_ok"] = [SEQ.get(g, "")[p - 1:p] == r for g, p, r in zip(pv["gene"], pv["pos"], pv["ref"])]
print(f"2023 pathogenic missense in paralogs: {len(pv)} | position matches UniProt sequence: {pv['seq_ok'].mean():.0%}")
pv = pv[pv["seq_ok"]]
print(pv.groupby("gene").size().sort_values(ascending=False).to_string())

json.dump(SEQ, open("/content/drive/MyDrive/biohack/paralog_sequences.json", "w"))
pv.to_csv("/content/drive/MyDrive/biohack/paralog_pathogenic_2023.csv", index=False)
print("\nSaved")

2023 pathogenic missense in paralogs: 1975 | position matches UniProt sequence: 98%
gene
SCN1A      622
SCN2A      276
KCNQ2      256
SCN8A      157
CACNA1A     97
SCN4A       75
PAX3        36
MYH3        31
NPR2        31
KCNQ3       28
KCNJ11      24
KCNQ4       24
SCN9A       23
KCNH1       22
ABCA3       21
CACNA1E     20
ABCA12      19
SCN3A       19
PAX2        18
ABCA1       18
CACNA1S     17
KCNJ10      14
KCNQ5       13
KCNJ5        8
CACNA1D      7
SCN11A       7
MYH6         7
KCNJ13       5
MYH2         5
CACNA1F      4
LMNB1        4
OPN1MW       4
KCNH5        3
LMNB2        3
OPN1LW       2
KCNJ1        2
KCNJ6        2
MYBPC1       1
NAGA         1
MYH8         1
PAX7         1

Saved


In [52]:
from Bio import Align

aligner = Align.PairwiseAligner()
aligner.substitution_matrix = B62
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5
aligner.end_open_gap_score = 0
aligner.end_extend_gap_score = 0
aligner.mode = "global"

pv_counts = {g: Counter(d["pos"]) for g, d in pv.groupby("gene")}
W, PARA, pair_rows = 7, {}, []
for G, fam in FAMILIES.items():
    sG = SEQ.get(G)
    if not sG:
        continue
    L = len(sG)
    same, near = np.zeros(L + 1), np.zeros(L + 1)
    cons, nal = np.zeros(L + 1), np.zeros(L + 1)
    for Q in fam:
        sQ = SEQ.get(Q)
        if not sQ:
            continue
        aln = aligner.align(sG, sQ)[0]
        mapped = np.zeros(L + 1, bool)
        ident = np.zeros(L + 1)
        qpos = np.zeros(L + 1, int)
        for (s1, e1), (s2, e2) in zip(*aln.aligned):
            for k in range(e1 - s1):
                t, q = s1 + k + 1, s2 + k + 1
                mapped[t], qpos[t] = True, q
                ident[t] = float(sG[t - 1] == sQ[q - 1])
        kern = np.ones(2 * W + 1)
        num = np.convolve(ident, kern, "same")
        den = np.maximum(np.convolve(mapped.astype(float), kern, "same"), 1)
        ok = mapped & (num / den >= 0.3)
        cnt = pv_counts.get(Q, Counter())
        for t in np.where(ok)[0]:
            q = qpos[t]
            same[t] += cnt.get(q, 0)
            near[t] += sum(cnt.get(q + d, 0) for d in range(-3, 4))
            cons[t] += ident[t]
            nal[t] += 1
        idn = round(float(ident[mapped].mean()), 2) if mapped.any() else 0
        pair_rows.append((G, Q, idn, int(ok.sum()), sum(cnt.values())))
    PARA[G] = (same, near, cons, nal)

pairs = pd.DataFrame(pair_rows, columns=["gene", "paralog", "identity", "usable", "paralog_P"])
print("Alignments:")
print(pairs.sort_values(["gene", "paralog_P"], ascending=[True, False]).to_string(index=False))

PCOLS = ["log_para_same", "log_para_near", "para_conserved_frac", "para_aligned"]

def para_feats(df):
    rows = []
    for g, p, r in zip(df["gene"], df["pos"], df["ref_aa"]):
        d = dict.fromkeys(PCOLS, 0.0)
        A = PARA.get(g)
        if A is not None and not pd.isna(p):
            p = int(p)
            if 0 < p < len(A[0]) and SEQ[g][p - 1:p] == r:
                same, near, cons, nal = A
                d["log_para_same"] = np.log1p(same[p])
                d["log_para_near"] = np.log1p(near[p])
                d["para_aligned"] = nal[p]
                d["para_conserved_frac"] = cons[p] / nal[p] if nal[p] else 0.0
        rows.append(d)
    return pd.DataFrame(rows, index=df.index)[PCOLS]

def add_para(df):
    return pd.concat([add_neigh(df), para_feats(df)], axis=1)

m_para = make_pipeline(StandardScaler(),
                       LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
m_para.fit(add_para(tr), ytr)

def sc4(name, df):
    if name == "+ paralogs":
        return m_para.predict_proba(add_para(df))[:, 1]
    return sc(name, df)

print()
print("model | AUROC test | vs gnomAD benign | top-500 | no-neighbor rank")
for name in ["AlphaMissense alone", "+ nearby known variants", "+ paralogs"]:
    a1 = roc_auc_score(yte, sc4(name, te))
    tps, fbs = sc4(name, te[yte == 1]), sc4(name, fb)
    a2 = roc_auc_score([1] * len(tps) + [0] * len(fbs), list(tps) + list(fbs))
    vus["s"] = sc4(name, vus)
    vus["pct"] = vus["s"].rank(pct=True)
    hits = int(vus.sort_values("s", ascending=False)["later_P"].head(500).sum())
    nn = vus.loc[vus["later_P"] & no_nb, "pct"].median()
    print(name, "|", round(a1, 3), "|", round(a2, 3), "|", hits,
          f"({hits / (500 * base):.1f}x)", "|", round(nn, 3))

pf = para_feats(vus)
has_para = ((pf["log_para_same"] > 0) | (pf["log_para_near"] > 0)).values
grp = (vus["later_P"] & no_nb).values
print()
print("No-neighbor later-pathogenic with paralog evidence:",
      int((grp & has_para).sum()), "of", int(grp.sum()))

rng = np.random.default_rng(4)
v_pa = sc4("+ paralogs", vus)
v_nb = sc4("+ nearby known variants", vus)
lpv = vus["later_P"].values
g = []
for _ in range(500):
    i = rng.integers(0, len(vus), len(vus))
    g.append(lpv[i][np.argsort(-v_pa[i])[:500]].sum() - lpv[i][np.argsort(-v_nb[i])[:500]].sum())
lo, hi = np.percentile(g, 2.5), np.percentile(g, 97.5)
print(f"Top-500 gain vs previous model: {np.mean(g):+.1f} (95% CI {lo:+.1f} to {hi:+.1f})")

coef = pd.Series(m_para[-1].coef_[0], index=add_para(tr.head(3)).columns)
print()
print("Paralog weights (+ means more likely pathogenic):")
print(coef[PCOLS].round(2).to_string())

/usr/local/lib/python3.13/dist-packages/Bio/Align/__init__.py:4412: BiopythonDeprecationWarning: The attribute 'end_open_gap_score' was renamed to 'open_end_gap_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/Bio/Align/__init__.py:4412: BiopythonDeprecationWarning: The attribute 'end_extend_gap_score' was renamed to 'extend_end_gap_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(


Alignments:
   gene paralog  identity  usable  paralog_P
  ABCA4   ABCA3      0.41    1081         21
  ABCA4  ABCA12      0.37    1304         19
  ABCA4   ABCA1      0.54    1874         18
  BEST1   BEST2      0.58     450          0
  BEST1   BEST3      0.51     435          0
CACNA1C CACNA1A      0.47    1573         97
CACNA1C CACNA1E      0.47    1515         20
CACNA1C CACNA1S      0.69    1743         17
CACNA1C CACNA1D      0.71    1929          7
CACNA1C CACNA1F      0.67    1727          4
CACNA1C CACNA1B      0.47    1499          0
    CRX    OTX2      0.58     258          0
    GLA    NAGA      0.51     335          1
 GUCY2D    NPR2      0.43     683         31
 GUCY2D  GUCY2F      0.54     835          0
  KCNH2   KCNH1      0.42     596         22
  KCNH2   KCNH5      0.41     600          3
  KCNH2   KCNH6      0.75     853          0
  KCNH2   KCNH7      0.66     941          0
  KCNJ2  KCNJ11      0.48     317         24
  KCNJ2  KCNJ10      0.42     262         1

In [53]:
from sklearn.model_selection import StratifiedKFold

CH = ["SCN5A", "KCNQ1", "KCNH2", "KCNJ2", "CACNA1C"]
trc = tr[tr["gene"].isin(CH)].copy()
yc = (trc["class_2023"] == "P").astype(int).values
print("Channel variants (2023):", len(trc), "| pathogenic:", int(yc.sum()),
      "| benign:", int((1 - yc).sum()))

Xa, Xb = add_neigh(trc), add_para(trc)
pa, pb = np.zeros(len(trc)), np.zeros(len(trc))
skf = StratifiedKFold(5, shuffle=True, random_state=0)
for trn, val in skf.split(Xa, yc):
    for X, store in ((Xa, pa), (Xb, pb)):
        m = make_pipeline(StandardScaler(),
                          LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
        m.fit(X.iloc[trn], yc[trn])
        store[val] = m.predict_proba(X.iloc[val])[:, 1]

def boot_diff(y, s1, s2, n=1000, seed=5):
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            d.append(roc_auc_score(y[i], s2[i]) - roc_auc_score(y[i], s1[i]))
    return np.mean(d), np.percentile(d, 2.5), np.percentile(d, 97.5)

print()
print("All channel variants:")
print("  AUROC without paralogs:", round(roc_auc_score(yc, pa), 3))
print("  AUROC with paralogs:   ", round(roc_auc_score(yc, pb), 3))
mu, lo, hi = boot_diff(yc, pa, pb)
print(f"  Gain: {mu:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

no_own = np.expm1(Xa["log_nearby_pathogenic"].values).round() == 0
ys = yc[no_own]
print()
print("Channel variants with NO known pathogenic neighbor in their own gene:",
      int(no_own.sum()), "| pathogenic:", int(ys.sum()))
if len(set(ys)) == 2:
    print("  AUROC without paralogs:", round(roc_auc_score(ys, pa[no_own]), 3))
    print("  AUROC with paralogs:   ", round(roc_auc_score(ys, pb[no_own]), 3))
    mu, lo, hi = boot_diff(ys, pa[no_own], pb[no_own])
    print(f"  Gain: {mu:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

print()
print("Per gene (AUROC without -> with):")
for gname in CH:
    k = (trc["gene"] == gname).values
    if len(set(yc[k])) == 2:
        print(f"  {gname}: {roc_auc_score(yc[k], pa[k]):.3f} -> {roc_auc_score(yc[k], pb[k]):.3f}"
              f"  (n={int(k.sum())})")

Channel variants (2023): 572 | pathogenic: 494 | benign: 78

All channel variants:
  AUROC without paralogs: 0.962
  AUROC with paralogs:    0.969
  Gain: +0.007 (95% CI +0.002 to +0.013)

Channel variants with NO known pathogenic neighbor in their own gene: 122 | pathogenic: 57
  AUROC without paralogs: 0.831
  AUROC with paralogs:    0.896
  Gain: +0.066 (95% CI +0.026 to +0.111)

Per gene (AUROC without -> with):
  SCN5A: 0.954 -> 0.960  (n=124)
  KCNQ1: 0.995 -> 0.994  (n=167)
  KCNH2: 0.919 -> 0.923  (n=173)
  KCNJ2: 1.000 -> 1.000  (n=46)
  CACNA1C: 0.763 -> 0.882  (n=62)


In [54]:
QMAP = {}
for G in CH:
    sG = SEQ[G]
    for Q in FAMILIES[G]:
        sQ = SEQ.get(Q)
        if not sQ:
            continue
        aln = aligner.align(sG, sQ)[0]
        qpos = np.zeros(len(sG) + 1, int)
        for (s1, e1), (s2, e2) in zip(*aln.aligned):
            for k in range(e1 - s1):
                qpos[s1 + k + 1] = s2 + k + 1
        QMAP[(G, Q)] = qpos

pv_pos = {g: d.groupby("pos").apply(lambda x: [f"p.{a}{p}{b}" for a, p, b in zip(x["ref"], x["pos"], x["alt"])]).to_dict()
          for g, d in pv.groupby("gene")}

cand = dedup2[dedup2["gene"].isin(CH)].copy()
cand["own_near"] = [int((np.abs(DEP_P.get(g, EMPTY)[0] - p) <= 5).sum()) for g, p in zip(cand["gene"], cand["pos"])]
cand["para_same"] = np.expm1(para_feats(cand)["log_para_same"].values).round().astype(int)
pick = cand[(cand["own_near"] == 0) & (cand["para_same"] > 0)].sort_values("rank")
print("Uncertain channel variants today with NO own-gene neighbors but a paralog disease variant at the same residue:",
      len(pick))
print()
for _, r in pick.head(10).iterrows():
    borrowed = []
    for Q in FAMILIES[r["gene"]]:
        qp = QMAP.get((r["gene"], Q))
        if qp is not None and int(r["pos"]) < len(qp) and qp[int(r["pos"])]:
            borrowed += [f"{Q} {c}" for c in pv_pos.get(Q, {}).get(int(qp[int(r["pos"])]), [])]
    am = "none" if pd.isna(r["am_pathogenicity"]) else round(r["am_pathogenicity"], 3)
    print(f"{r['gene']} {r['protein_change']} ({r['class_now']}) | rank #{int(r['rank'])} | AlphaMissense {am}")
    print(f"    borrowed: {'; '.join(borrowed[:4])}")

/tmp/ipykernel_421/365514216.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pv_pos = {g: d.groupby("pos").apply(lambda x: [f"p.{a}{p}{b}" for a, p, b in zip(x["ref"], x["pos"], x["alt"])]).to_dict()
/tmp/ipykernel_421/365514216.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pv_pos = {g: d.groupby("pos").apply(lambda x: [f"p.{a}{p}{b}" for a, p, b in zip(x["ref"], x["pos"], x["alt"])]).to_dict()


Uncertain channel variants today with NO own-gene neighbors but a paralog disease variant at the same residue: 201

KCNJ2 p.R46C (VUS) | rank #1065 | AlphaMissense 0.987
    borrowed: KCNJ11 p.R34C; KCNJ11 p.R34H
KCNQ1 p.L134P (conflicting) | rank #1380 | AlphaMissense 0.996
    borrowed: KCNQ2 p.F104S
SCN5A p.R808P (VUS) | rank #1384 | AlphaMissense 0.987
    borrowed: SCN1A p.R859C; SCN1A p.R859H; SCN4A p.R669H; SCN8A p.R844Q
SCN5A p.A1496P (VUS) | rank #1461 | AlphaMissense 0.999
    borrowed: SCN2A p.A1500T; SCN2A p.A1500V; SCN8A p.A1491V
SCN5A p.A332P (VUS) | rank #1521 | AlphaMissense 0.987
    borrowed: SCN1A p.A342V; SCN1A p.A342S; SCN1A p.A342E; SCN2A p.A344V
KCNJ2 p.C154Y (conflicting) | rank #1561 | AlphaMissense 0.998
    borrowed: KCNJ10 p.C140R
KCNQ1 p.L131P (VUS) | rank #1564 | AlphaMissense 0.958
    borrowed: KCNQ2 p.L101H
SCN5A p.R808C (conflicting) | rank #1589 | AlphaMissense 0.943
    borrowed: SCN1A p.R859C; SCN1A p.R859H; SCN4A p.R669H; SCN8A p.R844Q
SCN5A p.R130

In [56]:
RANK_COLS = ["rank", "rank_in_gene", "gene", "protein_change", "dna_variants", "class_now",
             "am_pathogenicity", "predicted_disease", "support", "evidence", "paralog_evidence",
             "nearest_known", "suggested_assay", "clinvar_link"]
dedup3[RANK_COLS].to_csv("/content/drive/MyDrive/biohack/todays_uncertain_variants_ranked.csv", index=False)

s_today = final_para.predict_proba(add_para_dep(out))[:, 1]
s_2023 = m_para.predict_proba(add_para(out))[:, 1]
for i, (x, (_, r)) in enumerate(zip(recs, out.iterrows())):
    x["harmful"] = round(float(s_today[i]), 4)
    x["harmful_2023_model"] = round(float(s_2023[i]), 4)
    x["paralog_evidence"] = borrowed_text(r["gene"], r["pos"])
    ev = x["evidence"] + ("; hotspot: paralog" if x["paralog_evidence"] else "")
    am = np.nan if x["alphamissense"] is None else x["alphamissense"]
    x["confidence"] = confidence2(x["harmful"], am, ev)
json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)
print("Saved ranked list and predictions with paralog evidence")

old = dict(zip(zip(dedup2["gene"], dedup2["protein_change"]), dedup2["rank"]))
print()
print("Demo examples: old rank -> new rank")
for g, pc in [("SCN5A", "p.R808P"), ("SCN5A", "p.R1308C"), ("SCN5A", "p.R1305C"),
              ("KCNJ2", "p.R46C"), ("KCNQ1", "p.L134P"), ("KCNQ1", "p.R591P")]:
    r = dedup3[(dedup3["gene"] == g) & (dedup3["protein_change"] == pc)]
    if len(r):
        r = r.iloc[0]
        print(f"  {g} {pc}: #{old.get((g, pc))} -> #{int(r['rank'])} | {r['paralog_evidence']}")

print()
print("Top CACNA1C uncertain variants (AlphaMissense has no scores here):")
cc = dedup3[(dedup3["gene"] == "CACNA1C") & (dedup3["paralog_evidence"] != "")]
for _, r in cc.head(5).iterrows():
    print(f"  {r['protein_change']} ({r['class_now']}) | rank #{int(r['rank'])}, "
          f"#{int(r['rank_in_gene'])} in gene | borrowed: {r['paralog_evidence']}")


Saved ranked list and predictions with paralog evidence

Demo examples: old rank -> new rank
  SCN5A p.R808P: #1384 -> #724 | SCN1A p.R859C; SCN1A p.R859H; SCN4A p.R669H; SCN8A p.R844Q
  SCN5A p.R1308C: #1590 -> #641 | SCN1A p.R1322T; SCN2A p.R1312S; SCN4A p.R1135H; SCN4A p.R1135S
  SCN5A p.R1305C: #1601 -> #1334 | SCN4A p.R1132Q
  KCNJ2 p.R46C: #1065 -> #336 | KCNJ11 p.R34C; KCNJ11 p.R34H
  KCNQ1 p.L134P: #1380 -> #967 | KCNQ2 p.F104S
  KCNQ1 p.R591P: #2 -> #9 | 

Top CACNA1C uncertain variants (AlphaMissense has no scores here):
  p.G402R (VUS) | rank #318, #1 in gene | borrowed: CACNA1D p.G403D; CACNA1D p.G403R; CACNA1E p.G348R; CACNA1F p.G369D
  p.G392V (VUS) | rank #4020, #22 in gene | borrowed: CACNA1F p.G359R
  p.I651N (VUS) | rank #4167, #28 in gene | borrowed: CACNA1A p.I613M; CACNA1E p.I603L
  p.I651V (VUS) | rank #4748, #50 in gene | borrowed: CACNA1A p.I613M; CACNA1E p.I603L
  p.I751V (VUS) | rank #4772, #52 in gene | borrowed: CACNA1D p.I750M; CACNA1D p.I750F; CACNA1E p.I7

## 14. FoldX on RHO, RPE65 and GLA (background runs)

In [60]:
import shutil

In [61]:
FX = "/content/foldx/foldx_20261231"
GENES_FX = ["RHO", "RPE65", "GLA"]

def collect(gene):
    v = out[(out["gene"] == gene) & out["split"].isin(["train", "test", "test_1star"])
            & out["residue_check"].isin(["exact", "numbering_offset"])]
    muts = {(r, int(p), a) for r, p, a in zip(v["uniprot_ref"], v["uniprot_pos"], v["uniprot_alt"])}
    b = fb[fb["gene"] == gene]
    muts |= {(pv_[0], int(pv_[1:-1]), pv_[-1]) for pv_ in b["protein_variant"].dropna()}
    top = dedup3[dedup3["gene"] == gene].head(30)
    muts |= {(r, int(p), a) for r, p, a in zip(top["ref_aa"], top["pos"], top["alt_aa"])}
    return {m for m in muts if (xyz_of(gene, m[1])[1] or 0) >= 70}

for gene in GENES_FX:
    folder = f"/content/foldx_runs/{gene}"
    os.makedirs(folder, exist_ok=True)
    shutil.copy("/content/foldx/rotabase.txt", folder)
    if gene == "RHO" and os.path.exists("/content/foldx/RHO_Repair.pdb"):
        shutil.copy("/content/foldx/RHO_Repair.pdb", f"{folder}/RHO_Repair.pdb")
    pdb = f"{folder}/{gene}.pdb"
    if not os.path.exists(pdb):
        meta = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT[gene]}", timeout=60).json()
        open(pdb, "w").write(requests.get(meta[0]["pdbUrl"], timeout=180).text)
    seq_ = {int(l[22:26]): AA3.get(l[17:20], "?") for l in open(pdb)
            if l.startswith("ATOM") and l[12:16].strip() == "CA"}
    muts = sorted(m for m in collect(gene) if seq_.get(m[1]) == m[0] and m[0] != m[2])
    open(f"{folder}/individual_list.txt", "w").write("".join(f"{r}A{p}{a};\n" for r, p, a in muts))
    pd.DataFrame(muts, columns=["ref", "pos", "alt"]).assign(idx=range(1, len(muts) + 1)) \
      .to_csv(f"{folder}/mutations.csv", index=False)
    script = "\n".join([
        f"cd {folder}",
        f"[ -f {gene}_Repair.pdb ] || {FX} --command=RepairPDB --pdb={gene}.pdb > repair.log 2>&1",
        f"{FX} --command=BuildModel --pdb={gene}_Repair.pdb --mutant-file=individual_list.txt "
        f"--numberOfRuns=1 > build.log 2>&1",
        f"cp Dif_{gene}_Repair.fxout /content/drive/MyDrive/biohack/foldx_{gene}_dif.fxout",
        f"cp mutations.csv /content/drive/MyDrive/biohack/foldx_{gene}_mutations.csv",
        "echo DONE > done.txt",
    ])
    open(f"{folder}/run.sh", "w").write(script + "\n")
    subprocess.Popen(f"nohup bash {folder}/run.sh > {folder}/nohup.log 2>&1 &", shell=True)
    print(f"{gene}: started FoldX on {len(muts)} variants")

RHO: started FoldX on 135 variants
RPE65: started FoldX on 106 variants
GLA: started FoldX on 247 variants


## 15. Figures for the pitch

In [62]:
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
FIG = "/content/drive/MyDrive/biohack/figures"
os.makedirs(FIG, exist_ok=True)
DARK, GREY, LIGHT = "#1f3b57", "#9aa5b1", "#d5dbe1"

# 1. Time-saved curve
kmax = 2000
ks = np.arange(1, kmax + 1)
lp_vals = vus["later_P"].values
def cum_hits(scores):
    return lp_vals[np.argsort(-scores)][:kmax].cumsum()
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(ks, cum_hits(sc4("+ paralogs", vus)), color=DARK, lw=2.6, label="Our model (2023 data only)")
ax.plot(ks, cum_hits(vus["am_pathogenicity"].values), color=GREY, lw=2, label="AlphaMissense")
ax.plot(ks, ks * base, color=GREY, lw=1.5, ls="--", label="Random order")
ax.axvline(500, color=LIGHT, lw=1)
ax.set_xlabel("Uncertain 2023 variants tested, in ranked order")
ax.set_ylabel("Later proven pathogenic")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout(); fig.savefig(f"{FIG}/1_time_saved_curve.png", dpi=200); plt.close(fig)

# 2. What each layer adds (top-500 hits)
labels = ["AlphaMissense\nalone", "+ chemistry\nand gene", "+ nearby known\nvariants", "+ related genes\n(final)"]
vals = [18, 29, 36, 38]
fig, ax = plt.subplots(figsize=(7.5, 4.3))
bars = ax.bar(labels, vals, color=[GREY, GREY, GREY, DARK], width=0.6)
for b_, v in zip(bars, vals):
    ax.text(b_.get_x() + b_.get_width() / 2, v + 0.8, str(v), ha="center", fontsize=12)
ax.set_ylabel("Later proven pathogenic in top 500")
ax.set_ylim(0, 45)
fig.tight_layout(); fig.savefig(f"{FIG}/2_layers_top500.png", dpi=200); plt.close(fig)

# 3. Paralog result
groups_ = ["Channel variants with no known\nneighbors in own gene (n=122)", "CACNA1C\n(no AlphaMissense score)"]
without, with_ = [0.831, 0.763], [0.896, 0.882]
x = np.arange(len(groups_))
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.bar(x - 0.18, without, 0.36, color=GREY, label="Without related genes")
ax.bar(x + 0.18, with_, 0.36, color=DARK, label="With related genes")
for xi, a, b_ in zip(x, without, with_):
    ax.text(xi - 0.18, a + 0.01, f"{a:.2f}", ha="center"); ax.text(xi + 0.18, b_ + 0.01, f"{b_:.2f}", ha="center")
ax.set_xticks(x); ax.set_xticklabels(groups_)
ax.set_ylabel("AUROC (5-fold cross-validation)"); ax.set_ylim(0.5, 1.0)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(f"{FIG}/3_paralogs.png", dpi=200); plt.close(fig)

# 4. Disease prediction
names_ = ["Random\nchoice", "Gene's most\ncommon disease", "Nearest known\nvariants"]
acc = [32, 65, 81]
lo_ = [0, 65 - 50.4, 81 - 68.1]; hi_ = [0, 76.6 - 65, 89.8 - 81]
fig, ax = plt.subplots(figsize=(7, 4.3))
ax.bar(names_, acc, color=[LIGHT, GREY, DARK], width=0.6, yerr=[lo_, hi_], capsize=6, ecolor="#555")
for i, v in enumerate(acc):
    ax.text(i + 0.18, v + 1.5, f"{v}%", fontsize=12)
ax.set_ylabel("Correct disease (%)"); ax.set_ylim(0, 100)
fig.tight_layout(); fig.savefig(f"{FIG}/4_disease_prediction.png", dpi=200); plt.close(fig)

# 5. FoldX on rhodopsin
fx = summary_fx.sort_values("total energy")
drivers = {"P23H": "steric clashes", "C187S": "broken disulfide", "G90D": "buried charge"}
cols_fx = [DARK if any(k in v for k in drivers) else LIGHT for v in fx.index]
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.barh([v.split(" (")[0] for v in fx.index], fx["total energy"], color=cols_fx)
for i, (v, val) in enumerate(fx["total energy"].items()):
    tag = next((d for k, d in drivers.items() if k in v), "benign control")
    ax.text(max(val, 0) + 0.3, i, f"{val:+.1f}  ({tag})", va="center", fontsize=11)
ax.axvline(0, color="#555", lw=1)
ax.set_xlabel("FoldX ΔΔG (kcal/mol, positive = destabilizing)")
ax.set_xlim(-2, 22)
fig.tight_layout(); fig.savefig(f"{FIG}/5_foldx_rho.png", dpi=200); plt.close(fig)

print("Saved 5 figures to", FIG)
for f in sorted(os.listdir(FIG)):
    print("  ", f)

Saved 5 figures to /content/drive/MyDrive/biohack/figures
   1_time_saved_curve.png
   2_layers_top500.png
   3_paralogs.png
   4_disease_prediction.png
   5_foldx_rho.png


## 16. Demo cards and 3D viewers

In [68]:
def view_3d(gene, p, extra=None, radius=10.0):
    meta = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT[gene]}", timeout=60).json()
    pdb = requests.get(meta[0]["pdbUrl"], timeout=180).text
    c, _ = xyz_of(gene, p)
    nb = sorted({int(k.pos) for k in kpn[kpn["gene"] == gene].itertuples()
                 if c is not None and k.pos != p and xyz_of(gene, k.pos)[0] is not None
                 and np.linalg.norm(xyz_of(gene, k.pos)[0] - c) <= radius})
    v = py3Dmol.view(width=800, height=550)
    v.addModel(pdb, "pdb")
    v.setStyle({"cartoon": {"color": "lightgrey"}})
    if nb:
        v.addStyle({"resi": nb}, {"stick": {"color": "orange"}, "cartoon": {"color": "orange"}})
    if extra:
        v.addStyle({"resi": extra}, {"stick": {"color": "green"}, "cartoon": {"color": "green"}})
    v.addStyle({"resi": [p]}, {"stick": {"color": "magenta"}, "sphere": {"color": "magenta", "radius": 1.3}})
    v.addResLabels({"resi": [p]}, {"fontSize": 14, "fontColor": "white", "backgroundColor": "black"})
    v.zoomTo({"within": {"distance": 15, "sel": {"resi": [p]}}})
    return v, nb

for card in cards:
    gene, pc = card["gene"], card["variant"]
    p = int(dedup3[(dedup3["gene"] == gene) & (dedup3["protein_change"] == pc)].iloc[0]["pos"])
    extra = [110] if gene == "RHO" else None
    v, nb = view_3d(gene, p, extra)
    open(f"/content/drive/MyDrive/biohack/{card['viewer_file']}", "w").write(v._make_html())
    print(f"{gene} {pc}: {len(nb)} known neighbors in 3D | saved {card['viewer_file']}")

MYH7 p.R403P: 5 known neighbors in 3D | saved viewer_MYH7_pR403P.html
RHO p.C187S: 16 known neighbors in 3D | saved viewer_RHO_pC187S.html
RPE65 p.H68P: 9 known neighbors in 3D | saved viewer_RPE65_pH68P.html
ABCA4 p.R2040P: 6 known neighbors in 3D | saved viewer_ABCA4_pR2040P.html
KCNQ1 p.R591P: 6 known neighbors in 3D | saved viewer_KCNQ1_pR591P.html
SCN5A p.R808P: 0 known neighbors in 3D | saved viewer_SCN5A_pR808P.html
CACNA1C p.G402R: 3 known neighbors in 3D | saved viewer_CACNA1C_pG402R.html


## 17. Isoform numbering fix and regenerated outputs

In [69]:
unv = feat["numbering"] == "clinvar_unverified"
match = [SEQ.get(g, "")[p - 1:p] == r for g, p, r in zip(feat["gene"], feat["pos"], feat["ref"])]
feat["numbering"] = np.where(unv & np.array(match), "uniprot_sequence_match",
                    np.where(unv, "mismatch_check_needed", feat["numbering"]))
print(feat["numbering"].value_counts().to_string())
print(feat[feat["numbering"] == "mismatch_check_needed"].groupby("gene").size().to_string())
feat.to_csv("/content/drive/MyDrive/biohack/feature_table.csv", index=False)

numbering
uniprot_canonical         14727
uniprot_sequence_match      676
mismatch_check_needed       502
gene
CACNA1C    446
LMNA         5
OPA1        20
PAX6         4
RPGR        26
SCN5A        1


In [70]:
import re, time

GENES_FIX = ["CACNA1C", "RPGR", "OPA1", "LMNA", "PAX6", "SCN5A"]
name_by_id = dict(zip(cur["VariationID"].astype(str), cur["Name"]))
num_by_id = dict(zip(feat["variation_id"], feat["numbering"]))

tx_of, gene_of_tx = {}, {}
for vid, g in zip(out["variant_id"].astype(str), out["gene"]):
    if g not in GENES_FIX:
        continue
    if g != "CACNA1C" and num_by_id.get(vid) == "uniprot_canonical":
        continue
    m = re.match(r"(NM_\d+\.\d+)", str(name_by_id.get(vid, "")))
    if m:
        tx_of[vid] = m.group(1)
        gene_of_tx[m.group(1)] = g
print("Variants to map:", len(tx_of), "| transcripts:", sorted(gene_of_tx))

def fetch_translation(acc):
    t = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
                     params={"db": "nuccore", "id": acc, "rettype": "gb", "retmode": "text"},
                     timeout=60).text
    m = re.search(r'/translation="([A-Z\s]+)"', t)
    return re.sub(r"\s", "", m.group(1)) if m else None

TXMAP = {}
for acc, g in gene_of_tx.items():
    s = fetch_translation(acc)
    time.sleep(0.4)
    if not s:
        print(acc, "FAILED"); continue
    aln = aligner.align(s, SEQ[g])[0]
    mp = {}
    for (s1, e1), (s2, e2) in zip(*aln.aligned):
        for k in range(e1 - s1):
            mp[s1 + k + 1] = s2 + k + 1
    TXMAP[acc] = mp
    print(f"{acc} ({g}): transcript {len(s)} aa vs canonical {len(SEQ[g])} aa")

out_vid = out["variant_id"].astype(str)
new_pos, status = {}, {}
for vid, g, p, r in zip(out_vid, out["gene"], out["position"], out["ref_aa"]):
    acc = tx_of.get(vid)
    if acc is None or acc not in TXMAP:
        continue
    q = TXMAP[acc].get(int(p))
    if q and SEQ[g][q - 1:q] == r:
        new_pos[vid], status[vid] = q, "mapped_via_transcript"
    else:
        status[vid] = "unmappable"

print()
print(pd.Series(status).value_counts().to_string())
st = pd.Series(status)
print(pd.crosstab(out.set_index(out_vid).loc[st.index, "gene"], st).to_string())

# apply to the working table and the feature table
mask = out_vid.isin(new_pos)
out.loc[mask, "pos"] = out_vid[mask].map(new_pos).astype(float)
fmask = feat["variation_id"].isin(new_pos)
feat.loc[fmask, "pos"] = feat.loc[fmask, "variation_id"].map(new_pos).astype(int)
feat.loc[fmask, "numbering"] = "mapped_via_transcript"
unm = feat["variation_id"].isin([k for k, v in status.items() if v == "unmappable"])
feat.loc[unm, "numbering"] = "unmappable"
feat.loc[fmask, "variant_id"] = (feat.loc[fmask, "gene"] + ":" + feat.loc[fmask, "ref"]
                                 + feat.loc[fmask, "pos"].astype(str) + feat.loc[fmask, "alt"])
feat.to_csv("/content/drive/MyDrive/biohack/feature_table.csv", index=False)
print()
print(feat["numbering"].value_counts().to_string())

# re-run the CACNA1C paralog test with corrected positions
ref = out[out["split"] == "train"]
REF_P, REF_B = index("P"), index("B")
tr = out[out["split"] == "train"].dropna(subset=["ref_aa", "alt_aa"])
trc = tr[tr["gene"].isin(CH)].copy()
yc = (trc["class_2023"] == "P").astype(int).values
Xa, Xb = add_neigh(trc), add_para(trc)
pa, pb = np.zeros(len(trc)), np.zeros(len(trc))
for trn, val in StratifiedKFold(5, shuffle=True, random_state=0).split(Xa, yc):
    for X, store in ((Xa, pa), (Xb, pb)):
        mdl = make_pipeline(StandardScaler(),
                            LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
        mdl.fit(X.iloc[trn], yc[trn])
        store[val] = mdl.predict_proba(X.iloc[val])[:, 1]
k = (trc["gene"] == "CACNA1C").values
print()
print("CACNA1C AUROC without -> with related genes:",
      round(roc_auc_score(yc[k], pa[k]), 3), "->", round(roc_auc_score(yc[k], pb[k]), 3), f"(n={int(k.sum())})")
no_own = np.expm1(Xa["log_nearby_pathogenic"].values).round() == 0
mu, lo, hi = boot_diff(yc[no_own], pa[no_own], pb[no_own])
print(f"All channels, no own-gene neighbors: gain {mu:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

Variants to map: 878 | transcripts: ['NM_000335.5', 'NM_000719.7', 'NM_001034853.2', 'NM_001099404.2', 'NM_001167623.2', 'NM_001368894.2', 'NM_005572.4', 'NM_130837.3', 'NM_170707.4']
NM_001368894.2 (PAX6): transcript 436 aa vs canonical 422 aa
NM_005572.4 (LMNA): transcript 572 aa vs canonical 664 aa
NM_000719.7 (CACNA1C): transcript 2138 aa vs canonical 2221 aa
NM_130837.3 (OPA1): transcript 1015 aa vs canonical 960 aa
NM_000335.5 (SCN5A): transcript 2015 aa vs canonical 2016 aa
NM_001167623.2 (CACNA1C): transcript 2138 aa vs canonical 2221 aa
NM_170707.4 (LMNA): transcript 664 aa vs canonical 664 aa
NM_001099404.2 (SCN5A): transcript 2016 aa vs canonical 2016 aa
NM_001034853.2 (RPGR): transcript 1152 aa vs canonical 1020 aa

mapped_via_transcript    822
unmappable                56
col_0    mapped_via_transcript  unmappable
gene                                      
CACNA1C                    798           2
LMNA                         4           5
OPA1                         4  

In [71]:
known_now = out[out["class_now"].isin(["P", "B"]) & (out["stars_now"] >= 1)]
DEP_P = make_index(known_now[known_now["class_now"] == "P"])
DEP_B = make_index(known_now[known_now["class_now"] == "B"])
cur_lab = known_now.dropna(subset=["ref_aa", "alt_aa"])

def fit_lr(X, y):
    mdl = make_pipeline(StandardScaler(),
                        LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
    return mdl.fit(X, y)

final_para = fit_lr(add_para_dep(cur_lab), (cur_lab["class_now"] == "P").astype(int))
ytr = (tr["class_2023"] == "P").astype(int)
m_para = fit_lr(add_para(tr), ytr)

kn = out[(out["class_now"] == "P") & (out["stars_now"] >= 1)].copy()
kn["groups"] = kn["group_now"].fillna("").map(groups).map(clean)
kn["mondo"] = kn["variant_id"].astype(str).map(mondo_map).fillna("").str.split("|")
kn = kn[kn["groups"].str.len() > 0]
KN = {g: d for g, d in kn.groupby("gene")}
EV_TODAY = make_ev_index(out[out["class_now"] == "P"])
EV_2023 = make_ev_index(out[(out["split"] == "train") & (out["class_2023"] == "P")])
kpn = out[out["class_now"] == "P"].copy()
kpn["groups_txt"] = kpn["group_now"].fillna("").str.replace("|", "/", regex=False)
vkey = dict(zip(feat["variation_id"], feat["variant_id"]))

s_today = final_para.predict_proba(add_para_dep(out))[:, 1]
s_2023 = m_para.predict_proba(add_para(out))[:, 1]
recs = []
for i, (_, r) in enumerate(out.iterrows()):
    vid = str(r["variant_id"])
    dg, dm, near = disease_from_neighbors(r["gene"], r["pos"], vid)
    args = (r["gene"], r["ref_aa"], r["pos"], r["alt_aa"], vid)
    ev, ev23 = evidence_x(*args, EV_TODAY), evidence_x(*args, EV_2023)
    para_txt = borrowed_text(r["gene"], r["pos"])
    am = r["am_pathogenicity"]
    recs.append({
        "variant": vkey.get(vid, ""), "variation_id": vid, "gene": r["gene"],
        "harmful": round(float(s_today[i]), 4), "harmful_2023_model": round(float(s_2023[i]), 4),
        "alphamissense": None if pd.isna(am) else round(float(am), 4),
        "confidence": confidence2(s_today[i], am, ev + ("; hotspot" if para_txt else "")),
        "mechanism": {}, "mechanism_status": "pending FoldX",
        "disease": dm, "disease_groups": dg,
        "treatment": TREATMENT.get(r["gene"], ""), "suggested_assay": ASSAY.get(r["gene"], ""),
        "evidence": ev, "evidence_2023": ev23, "paralog_evidence": para_txt, "nearest_known": near,
        "clinvar_today": r["class_now"], "role_2023": r["split"],
        "in_training_today": bool(r["class_now"] in ("P", "B") and r["stars_now"] >= 1),
        "disclaimer": "Research use only. Prioritization, not a clinical classification.",
    })
json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)
print("Predictions regenerated:", len(recs))

P = pd.DataFrame(recs)
P["pos"] = out["pos"].values
P["ref_aa"], P["alt_aa"] = out["ref_aa"].values, out["alt_aa"].values
P["protein_change"] = "p." + out["ref_aa"].values + out["position"].astype(int).astype(str).values + out["alt_aa"].values
P["clinvar_link"] = "https://www.ncbi.nlm.nih.gov/clinvar/variation/" + P["variation_id"] + "/"
P["predicted_disease"] = P["disease_groups"].map(lambda d: next(iter(d), "") if d else "")
P["support"] = P["disease_groups"].map(lambda d: f"{round(next(iter(d.values())) * 5)}/5 nearest" if d else "")
U = P[P["clinvar_today"].isin(["VUS", "conflicting"])].sort_values("harmful", ascending=False)
dedup3 = U.groupby(["gene", "protein_change"], as_index=False).first()
dedup3["dna_variants"] = U.groupby(["gene", "protein_change"]).size().values
dedup3 = dedup3.sort_values("harmful", ascending=False).reset_index(drop=True)
dedup3.insert(0, "rank", dedup3.index + 1)
dedup3["rank_in_gene"] = dedup3.groupby("gene").cumcount() + 1
dedup3 = dedup3.rename(columns={"clinvar_today": "class_now", "alphamissense": "am_pathogenicity"})
dedup3[RANK_COLS].to_csv("/content/drive/MyDrive/biohack/todays_uncertain_variants_ranked.csv", index=False)
print("Ranked list regenerated:", len(dedup3), "unique protein changes")

print()
for card in cards:
    row = dedup3[(dedup3["gene"] == card["gene"]) & (dedup3["protein_change"] == card["variant"])]
    if row.empty:
        print("Not found:", card["gene"], card["variant"]); continue
    r = row.iloc[0]
    p = int(r["pos"])
    nb3, pl, contact = neighbors_3d(card["gene"], p)
    card.update({"rank_overall": int(r["rank"]), "rank_in_gene": int(r["rank_in_gene"]),
                 "predicted_disease": r["predicted_disease"], "disease_support": r["support"],
                 "evidence": r["evidence"], "paralog_evidence": r["paralog_evidence"],
                 "nearest_in_sequence": r["nearest_known"], "nearest_in_3d": nb3,
                 "plddt": None if pd.isna(pl) else round(float(pl), 1), "contact_number": contact,
                 "canonical_position": p})
    if card["gene"] == "CACNA1C":
        v, _ = view_3d("CACNA1C", p)
        open(f"/content/drive/MyDrive/biohack/{card['viewer_file']}", "w").write(v._make_html())
    print(f"{card['gene']} {card['variant']}: rank #{card['rank_overall']} (#{card['rank_in_gene']} in gene) "
          f"| canonical pos {p} | paralog: {card['paralog_evidence'] or '-'}")
json.dump(cards, open("/content/drive/MyDrive/biohack/demo_cards.json", "w"), indent=2)
print("\nDemo cards updated")

Predictions regenerated: 15905
Ranked list regenerated: 12345 unique protein changes

MYH7 p.R403P: rank #23 (#3 in gene) | canonical pos 403 | paralog: -
RHO p.C187S: rank #10 (#1 in gene) | canonical pos 187 | paralog: OPN1LW p.C203R; OPN1MW p.C203R
RPE65 p.H68P: rank #207 (#2 in gene) | canonical pos 68 | paralog: -
ABCA4 p.R2040P: rank #2 (#1 in gene) | canonical pos 2040 | paralog: -
KCNQ1 p.R591P: rank #28 (#7 in gene) | canonical pos 591 | paralog: -
SCN5A p.R808P: rank #671 (#38 in gene) | canonical pos 808 | paralog: SCN1A p.R859C; SCN1A p.R859H; SCN4A p.R669H; SCN8A p.R844Q
CACNA1C p.G402R: rank #320 (#1 in gene) | canonical pos 402 | paralog: CACNA1D p.G403D; CACNA1D p.G403R; CACNA1E p.G348R; CACNA1F p.G369D

Demo cards updated


In [72]:
fig, ax = plt.subplots(figsize=(7.5, 4.3))
without, with_ = [0.831, 0.763], [0.896, 0.923]
x = np.arange(2)
ax.bar(x - 0.18, without, 0.36, color=GREY, label="Without related genes")
ax.bar(x + 0.18, with_, 0.36, color=DARK, label="With related genes")
for xi, a, b_ in zip(x, without, with_):
    ax.text(xi - 0.18, a + 0.01, f"{a:.2f}", ha="center"); ax.text(xi + 0.18, b_ + 0.01, f"{b_:.2f}", ha="center")
ax.set_xticks(x)
ax.set_xticklabels(["Channel variants with no known\nneighbors in own gene (n=122)", "CACNA1C\n(no AlphaMissense score)"])
ax.set_ylabel("AUROC (5-fold cross-validation)"); ax.set_ylim(0.5, 1.0)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(f"{FIG}/3_paralogs.png", dpi=200); plt.close(fig)
print("Updated 3_paralogs.png")

Updated 3_paralogs.png


## 18. 1000 Genomes healthy carriers (mentor suggestion)

In [74]:
subprocess.run("apt-get -qq install -y bcftools > /dev/null 2>&1", shell=True)
print(subprocess.run(["bcftools", "--version"], capture_output=True, text=True).stdout.split("\n")[0])

BASE = ("https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage/"
        "working/20220422_3202_phased_SNV_INDEL_SV")

def kg_url(ch):
    if ch == "chrX":
        return f"{BASE}/1kGP_high_coverage_Illumina.chrX.filtered.SNV_INDEL_SV_phased_panel.v2.vcf.gz"
    return f"{BASE}/1kGP_high_coverage_Illumina.{ch}.filtered.SNV_INDEL_SV_phased_panel.vcf.gz"

kk = out["key"].str.split(":", expand=True)
sites = pd.DataFrame({"gene": out["gene"].values, "chrom": kk[0].values,
                      "gpos": kk[1].astype(int).values, "key": out["key"].values})
os.makedirs("/content/kg", exist_ok=True)

rows = []
for gname, s in sites.groupby("gene"):
    ch = s["chrom"].iloc[0]
    tfile = f"/content/kg/{gname}_targets.tsv"
    s[["chrom", "gpos"]].drop_duplicates().sort_values("gpos") \
        .to_csv(tfile, sep="\t", header=False, index=False)
    region = f"{ch}:{s['gpos'].min()}-{s['gpos'].max()}"
    res = subprocess.run(["bcftools", "query", "-r", region, "-T", tfile, "-i", 'GT="alt"',
                          "-f", "[%CHROM\t%POS\t%REF\t%ALT\t%SAMPLE\t%GT\n]", kg_url(ch)],
                         capture_output=True, text=True)
    if res.returncode != 0:
        print(f"{gname}: FAILED ({res.stderr.strip()[:120]})")
        continue
    n0 = len(rows)
    for line in res.stdout.splitlines():
        c, p, r_, a_, smp, gt = line.split("\t")
        rows.append((gname, f"{c}:{p}:{r_}:{a_}", smp, gt))
    print(f"{gname}: {len(rows) - n0} carrier genotypes")

kgc = pd.DataFrame(rows, columns=["gene", "key", "sample", "gt"])
kgc = kgc[kgc["key"].isin(set(out["key"]))].copy()
kgc["copies"] = kgc["gt"].map(lambda g: sum(x == "1" for x in g.replace("|", "/").split("/")))

ped = pd.read_csv(f"{BASE.rsplit('/', 1)[0]}/1kGP.3202_samples.pedigree_info.txt", sep=r"\s+")
is_child = (ped.iloc[:, 1].astype(str) != "0") | (ped.iloc[:, 2].astype(str) != "0")
children = set(ped.loc[is_child, ped.columns[0]])
kgc["is_child_in_trio"] = kgc["sample"].isin(children)
kgc.to_csv("/content/drive/MyDrive/biohack/kg1000_carriers.csv", index=False)

print()
print("Carrier genotypes at your ClinVar variants:", len(kgc))
print("Variants carried by anyone:", kgc["key"].nunique())
print("Unrelated carriers:", kgc.loc[~kgc["is_child_in_trio"], "sample"].nunique())

bcftools 1.19
ABCA4: 3892 carrier genotypes
BEST1: 192 carrier genotypes
CACNA1C: 351 carrier genotypes
CEP290: 1983 carrier genotypes
CRX: 270 carrier genotypes
GLA: 25 carrier genotypes
GUCY2D: 4264 carrier genotypes
KCNH2: 953 carrier genotypes
KCNJ2: 0 carrier genotypes
KCNQ1: 400 carrier genotypes
LMNA: 1212 carrier genotypes
MYBPC3: 1037 carrier genotypes
MYH7: 1159 carrier genotypes
NR2E3: 789 carrier genotypes
OPA1: 2505 carrier genotypes
PAX6: 0 carrier genotypes
PKP2: 1260 carrier genotypes
PRPH2: 9272 carrier genotypes
RHO: 62 carrier genotypes
RPE65: 304 carrier genotypes
RPGR: 3461 carrier genotypes
SCN5A: 5337 carrier genotypes
USH2A: 21531 carrier genotypes

Carrier genotypes at your ClinVar variants: 54384
Variants carried by anyone: 713
Unrelated carriers: 2594


In [75]:
POP_URL = ("https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage/"
           "20130606_g1k_3202_samples_ped_population.txt")
try:
    popdf = pd.read_csv(POP_URL, sep=r"\s+")
    scol = next(c for c in popdf.columns if c.lower().startswith("sample"))
    pcol = next(c for c in popdf.columns if "super" in c.lower())
    pop_of = dict(zip(popdf[scol], popdf[pcol]))
except Exception as e:
    print("Population file not loaded:", e); popdf, pop_of = None, {}

prot = dict(zip(out["key"], "p." + out["ref_aa"] + out["position"].astype(int).astype(str) + out["alt_aa"]))
rec_by_key = {}
for k_, x in zip(out["key"], recs):
    rec_by_key.setdefault(k_, x)
lab = out[["key", "class_now", "stars_now"]].drop_duplicates("key")
kg = kgc[~kgc["is_child_in_trio"]].merge(lab, on="key", how="left")
kg["superpop"] = kg["sample"].map(pop_of)
XL = {"GLA", "RPGR"}
kg["inherit"] = kg["gene"].map(lambda g: "recessive/mixed" if g in RECESSIVE_OR_MIXED
                               else ("X-linked" if g in XL else "dominant"))
isP = (kg["class_now"] == "P") & (kg["stars_now"] >= 1)

print("1. Healthy carriers of ClinVar-pathogenic variants in DOMINANT genes")
dP = kg[isP & (kg["inherit"] == "dominant")]
per = dP.groupby("key").agg(gene=("gene", "first"), carriers=("sample", "nunique"),
                            stars=("stars_now", "first")).sort_values("carriers", ascending=False)
print(f"   {len(per)} pathogenic variants carried by {dP['sample'].nunique()} people")
for k_, r in per.head(8).iterrows():
    print(f"   {r['gene']} {prot.get(k_)}: {r['carriers']} carriers ({int(r['stars'])} stars)")

print()
print("2. Two hits in RECESSIVE genes (should be affected by textbook genetics)")
rP = kg[isP & (kg["inherit"] == "recessive/mixed")]
two_hit = []
for (smp, gname), d in rP.groupby(["sample", "gene"]):
    hom = (d["gt"] == "1|1").any()
    h1 = d["gt"].str.startswith("1").any()
    h2 = d["gt"].str.endswith("1").any()
    if hom or (h1 and h2 and len(d) >= 2):
        two_hit.append((smp, gname, "homozygous" if hom else "two different, opposite copies",
                        "; ".join(prot.get(k_, k_) for k_ in d["key"])))
th = pd.DataFrame(two_hit, columns=["sample", "gene", "type", "variants"])
print(f"   {len(th)} people")
print(th.head(10).to_string(index=False) if len(th) else "   none")

print()
print("3. People with pathogenic variants in 2+ DOMINANT genes")
ng = dP.groupby("sample")["gene"].nunique()
print(f"   1 gene: {int((ng == 1).sum())} | 2+ genes: {int((ng >= 2).sum())}")

print()
print("4. Your model on uncertain variants carried by 2+ healthy people (dominant genes)")
dV = kg[(kg["class_now"] == "VUS") & (kg["inherit"] == "dominant")]
many = dV.groupby("key")["sample"].nunique()
seen = set(many[many >= 2].index)
allV = [x for k_, x in rec_by_key.items() if x["clinvar_today"] == "VUS"
        and x["gene"] not in RECESSIVE_OR_MIXED and x["gene"] not in XL]
def summ(xs, label):
    hi = np.mean([x["confidence"] == "high" for x in xs])
    am = [x["alphamissense"] for x in xs if x["alphamissense"] is not None]
    print(f"   {label}: n={len(xs)} | our 'high priority' {hi:.0%} | "
          f"median our score {np.median([x['harmful'] for x in xs]):.2f} | "
          f"AlphaMissense likely pathogenic {np.mean([a > 0.564 for a in am]):.0%}")
summ([rec_by_key[k_] for k_ in seen if k_ in rec_by_key], "VUS in 2+ healthy people")
summ(allV, "All dominant-gene VUS     ")

if popdf is not None:
    print()
    print("5. Healthy carriers of dominant-gene pathogenic variants, per 1,000 people")
    unrel = popdf[~popdf[scol].isin(children)]
    denom = unrel[pcol].value_counts()
    carriers = dP.groupby("superpop")["sample"].nunique()
    for sp in denom.index:
        k_, n_ = int(carriers.get(sp, 0)), int(denom[sp])
        lo_, hi_ = wilson(k_, n_)
        print(f"   {sp}: {k_}/{n_} = {1000 * k_ / n_:.0f} per 1,000 (95% CI {10 * lo_:.0f} to {10 * hi_:.0f})")

th.to_csv("/content/drive/MyDrive/biohack/kg1000_two_hit_recessive.csv", index=False)
per.to_csv("/content/drive/MyDrive/biohack/kg1000_healthy_pathogenic_carriers.csv")

1. Healthy carriers of ClinVar-pathogenic variants in DOMINANT genes
   3 pathogenic variants carried by 4 people
   KCNH2 p.L790P: 2 carriers (2 stars)
   RHO p.G51V: 1 carriers (2 stars)
   SCN5A p.E161K: 1 carriers (2 stars)

2. Two hits in RECESSIVE genes (should be affected by textbook genetics)
   2 people
 sample  gene       type variants
HG02658 ABCA4 homozygous p.R2030Q
HG03300 ABCA4 homozygous p.R2107H

3. People with pathogenic variants in 2+ DOMINANT genes
   1 gene: 4 | 2+ genes: 0

4. Your model on uncertain variants carried by 2+ healthy people (dominant genes)
   VUS in 2+ healthy people: n=20 | our 'high priority' 35% | median our score 0.04 | AlphaMissense likely pathogenic 20%
   All dominant-gene VUS     : n=6121 | our 'high priority' 31% | median our score 0.16 | AlphaMissense likely pathogenic 42%

5. Healthy carriers of dominant-gene pathogenic variants, per 1,000 people
   AFR: 0/686 = 0 per 1,000 (95% CI 0 to 6)
   EUR: 1/526 = 2 per 1,000 (95% CI 0 to 11)
   S

## 19. FoldX analysis and mechanism labels

In [77]:
def read_dif(gname):
    p = f"/content/foldx_runs/{gname}/Dif_{gname}_Repair.fxout"
    if not os.path.exists(p):
        return None
    L = open(p).read().splitlines()
    st = next((i for i, l in enumerate(L) if l.startswith("Pdb")), None)
    if st is None:
        return None
    d = pd.read_csv(io.StringIO("\n".join(L[st:])), sep="\t")
    d["idx"] = d["Pdb"].str.extract(r"_(\d+)(?:_\d+)?\.pdb$")[0].astype(int)
    d = d.merge(pd.read_csv(f"/content/foldx_runs/{gname}/mutations.csv"), on="idx", how="left")
    d["gene"] = gname
    return d

lab_map, am_map = {}, {}
v = out[out["residue_check"].isin(["exact", "numbering_offset"])]
for g, r, p, a, c23, am in zip(v["gene"], v["uniprot_ref"], v["uniprot_pos"], v["uniprot_alt"],
                               v["class_2023"], v["am_pathogenicity"]):
    lab_map.setdefault((g, r, int(p), a), c23); am_map.setdefault((g, r, int(p), a), am)
for g, pv_, am in zip(fb["gene"], fb["protein_variant"], fb["am_pathogenicity"]):
    if isinstance(pv_, str):
        k_ = (g, pv_[0], int(pv_[1:-1]), pv_[-1])
        lab_map.setdefault(k_, "gnomad_benign"); am_map.setdefault(k_, am)

TERMS = ["Van der Waals clashes", "disulfide", "Solvation Polar", "Solvation Hydrophobic",
         "Van der Waals", "Backbone Hbond", "Sidechain Hbond", "Electrostatics"]
for gname in ["RHO", "RPE65", "GLA"]:
    if not os.path.exists(f"/content/foldx_runs/{gname}/done.txt"):
        print(f"{gname}: not finished yet"); continue
    d = read_dif(gname)
    if d is None:
        print(f"{gname}: no results file"); continue
    keys = list(zip(d["gene"], d["ref"], d["pos"], d["alt"]))
    d["label"] = [lab_map.get(k_) for k_ in keys]
    d["am"] = [am_map.get(k_) for k_ in keys]
    d["ddg"] = d["total energy"].clip(upper=10)
    L = d[d["label"].isin(["P", "B", "gnomad_benign"])].dropna(subset=["am"]).copy()
    y = (L["label"] == "P").astype(int).values
    print(f"\n===== {gname}: {len(d)} variants with FoldX | labeled: {int(y.sum())} pathogenic, "
          f"{int((1 - y).sum())} benign")
    if len(set(y)) < 2 or min(y.sum(), (1 - y).sum()) < 5:
        print("   too few of one class to test"); continue
    print(f"   Median ΔΔG: pathogenic {L.loc[y == 1, 'ddg'].median():+.2f} | "
          f"benign {L.loc[y == 0, 'ddg'].median():+.2f} kcal/mol")
    print(f"   AUROC, ΔΔG alone: {roc_auc_score(y, L['ddg']):.3f} | "
          f"AlphaMissense alone: {roc_auc_score(y, L['am']):.3f}")
    cols_fx = ["am", "ddg"] + [t for t in TERMS if t in L.columns]
    pa_, pb_ = np.zeros(len(L)), np.zeros(len(L))
    folds = StratifiedKFold(5, shuffle=True, random_state=0)
    for trn, val in folds.split(L, y):
        for cset, store in ((["am"], pa_), (cols_fx, pb_)):
            mdl = fit_lr(L.iloc[trn][cset].fillna(0), y[trn])
            store[val] = mdl.predict_proba(L.iloc[val][cset].fillna(0))[:, 1]
    mu, lo, hi = boot_diff(y, pa_, pb_)
    print(f"   CV AUROC, AlphaMissense: {roc_auc_score(y, pa_):.3f} -> + FoldX physics: "
          f"{roc_auc_score(y, pb_):.3f} (gain {mu:+.3f}, 95% CI {lo:+.3f} to {hi:+.3f})")
    Pv = L[y == 1]
    if len(Pv):
        dom = Pv[[t for t in TERMS if t in Pv.columns]].clip(lower=0).idxmax(axis=1)
        print("   Dominant destabilizing term among pathogenic variants:")
        print("   " + dom.value_counts().to_string().replace("\n", "\n   "))
    d.to_csv(f"/content/drive/MyDrive/biohack/foldx_{gname}_results.csv", index=False)


===== RHO: 135 variants with FoldX | labeled: 100 pathogenic, 5 benign
   Median ΔΔG: pathogenic +4.88 | benign +0.32 kcal/mol
   AUROC, ΔΔG alone: 0.942 | AlphaMissense alone: 0.982
   CV AUROC, AlphaMissense: 0.980 -> + FoldX physics: 0.982 (gain +0.002, 95% CI -0.010 to +0.015)
   Dominant destabilizing term among pathogenic variants:
   Van der Waals clashes    47
   Solvation Polar          19
   Solvation Hydrophobic    17
   Sidechain Hbond           8
   Electrostatics            4
   disulfide                 3
   Backbone Hbond            2

===== RPE65: 106 variants with FoldX | labeled: 68 pathogenic, 3 benign
   too few of one class to test
GLA: not finished yet


In [80]:
TERM_LABEL = {
    "Van der Waals clashes": "misfolding (packing clash)",
    "disulfide": "misfolding (broken disulfide)",
    "Solvation Hydrophobic": "misfolding (lost hydrophobic packing)",
    "Solvation Polar": "buried polar or charged group",
    "Sidechain Hbond": "lost hydrogen bonds", "Backbone Hbond": "lost hydrogen bonds",
    "Electrostatics": "electrostatic change", "Van der Waals": "lost packing contacts",
}
FX = {}
for gname in ["RHO", "RPE65", "GLA"]:
    f = f"/content/drive/MyDrive/biohack/foldx_{gname}_results.csv"
    if not os.path.exists(f):
        continue
    d = pd.read_csv(f)
    for _, r in d.iterrows():
        ddg = float(r["total energy"])
        terms = {t: float(r[t]) for t in TERM_LABEL if t in d.columns}
        dom = max(terms, key=lambda t: terms[t])
        if ddg >= 2:
            lab_ = TERM_LABEL[dom]
        elif ddg >= 1:
            lab_ = "mildly destabilizing"
        else:
            lab_ = "fold stable (if harmful: functional site or gain of function)"
        FX[(gname, r["ref"], int(r["pos"]), r["alt"])] = {
            "label": lab_, "ddg_kcal_mol": round(min(ddg, 10), 2),
            "dominant_term": dom if ddg >= 1 else "", "capped": ddg > 10}
print("FoldX mechanism available for", len(FX), "protein changes")

n_set = 0
for x, (_, r) in zip(recs, out.iterrows()):
    fx = FX.get((r["gene"], r["ref_aa"], int(r["pos"]) if not pd.isna(r["pos"]) else -1, r["alt_aa"]))
    if fx:
        x["mechanism"] = fx
        x["mechanism_status"] = "FoldX on AlphaFold model (no membrane, no ligands)"
        n_set += 1
json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)
print("Predictions with a FoldX mechanism:", n_set)

for card in cards:
    rr = dedup3[(dedup3["gene"] == card["gene"]) & (dedup3["protein_change"] == card["variant"])]
    if rr.empty:
        continue
    rr = rr.iloc[0]
    fx = FX.get((card["gene"], rr["ref_aa"], int(rr["pos"]), rr["alt_aa"]))
    if fx:
        card["foldx"] = fx
        print(f"{card['gene']} {card['variant']}: ΔΔG {fx['ddg_kcal_mol']:+.2f} kcal/mol -> {fx['label']}")
json.dump(cards, open("/content/drive/MyDrive/biohack/demo_cards.json", "w"), indent=2)
print("Demo cards updated")

FoldX mechanism available for 241 protein changes
Predictions with a FoldX mechanism: 246
RHO p.C187S: ΔΔG +5.80 kcal/mol -> misfolding (broken disulfide)
RPE65 p.H68P: ΔΔG +5.05 kcal/mol -> misfolding (packing clash)
Demo cards updated


In [79]:
d_rpe = read_dif("RPE65")
d_rpe.to_csv("/content/drive/MyDrive/biohack/foldx_RPE65_results.csv", index=False)
print("Saved RPE65 FoldX results:", len(d_rpe), "variants")

Saved RPE65 FoldX results: 106 variants


## 20. ACMG PP3/BP4 calibration

*Superseded first attempt: this cell uses cumulative likelihood ratios, which gave invalid cutoffs (see its output). It is kept because the next cell reuses its calibration set. The next cell is the calibration actually used (local likelihood ratios, Tavtigian et al. 2018 thresholds).*

In [81]:
Xtr_all = add_para(tr)
yv = ytr.values
oof = np.zeros(len(tr))
for trn, val in StratifiedKFold(5, shuffle=True, random_state=1).split(Xtr_all, yv):
    mdl = fit_lr(Xtr_all.iloc[trn], yv[trn])
    oof[val] = mdl.predict_proba(Xtr_all.iloc[val])[:, 1]
sP = oof[yv == 1]
sB = np.concatenate([oof[yv == 0], m_para.predict_proba(add_para(fb))[:, 1]])
print(f"Calibration set: {len(sP)} pathogenic, {len(sB)} benign")

def lr_pos(sp, sb, t):
    return (sp >= t).mean() / (max((sb >= t).sum(), 0.5) / len(sb))
def lr_neg(sp, sb, t):
    return (sp <= t).mean() / (max((sb <= t).sum(), 0.5) / len(sb))

grid = np.unique(np.quantile(np.concatenate([sP, sB]), np.linspace(0.005, 0.995, 250)))
rng = np.random.default_rng(7)
boots = [(rng.choice(sP, len(sP)), rng.choice(sB, len(sB))) for _ in range(300)]
lb_pos = np.array([np.percentile([lr_pos(a, b, t) for a, b in boots], 5) for t in grid])
ub_neg = np.array([np.percentile([lr_neg(a, b, t) for a, b in boots], 95) for t in grid])

LEVELS_P = {"Strong": 18.7, "Moderate": 4.33, "Supporting": 2.08}
LEVELS_B = {"Strong": 0.053, "Moderate": 0.23, "Supporting": 0.48}
tP = {k: next((t for t, lb in zip(grid, lb_pos) if lb >= v), None) for k, v in LEVELS_P.items()}
tB = {k: next((t for t, ub in zip(grid[::-1], ub_neg[::-1]) if ub <= v), None) for k, v in LEVELS_B.items()}
print()
print("Score cutoffs (95% bootstrap bound):")
for k in ["Supporting", "Moderate", "Strong"]:
    p_ = "not reached" if tP[k] is None else f"score >= {tP[k]:.3f}"
    b_ = "not reached" if tB[k] is None else f"score <= {tB[k]:.3f}"
    print(f"  {k:10s} | PP3: {p_:20s} | BP4: {b_}")

def evidence_level(s):
    for k in ["Strong", "Moderate", "Supporting"]:
        if tP[k] is not None and s >= tP[k]:
            return f"PP3_{k}"
    for k in ["Strong", "Moderate", "Supporting"]:
        if tB[k] is not None and s <= tB[k]:
            return f"BP4_{k}"
    return "indeterminate"

te_chk = pd.DataFrame([(evidence_level(x["harmful_2023_model"]), x["clinvar_today"])
                       for x in recs if x["role_2023"] == "test"], columns=["evidence_2023_model", "outcome"])
print()
print("Check on 2023 test variants (evidence from the 2023 model vs. what experts decided later):")
print(pd.crosstab(te_chk["evidence_2023_model"], te_chk["outcome"]).to_string())

for x in recs:
    x["computational_evidence"] = evidence_level(x["harmful"])
json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)

for card in cards:
    rr = dedup3[(dedup3["gene"] == card["gene"]) & (dedup3["protein_change"] == card["variant"])]
    if len(rr):
        card["computational_evidence"] = evidence_level(float(rr.iloc[0]["harmful"]))
        print(f"{card['gene']} {card['variant']}: {card['computational_evidence']}")
json.dump(cards, open("/content/drive/MyDrive/biohack/demo_cards.json", "w"), indent=2)

unc_ev = pd.Series([x["computational_evidence"] for x in recs if x["clinvar_today"] in ("VUS", "conflicting")])
print()
print("Today's uncertain variants by evidence level:")
print(unc_ev.value_counts().to_string())
pd.DataFrame({"level": list(tP) + list(tB), "side": ["PP3"] * 3 + ["BP4"] * 3,
              "cutoff": list(tP.values()) + list(tB.values())}) \
  .to_csv("/content/drive/MyDrive/biohack/evidence_calibration.csv", index=False)

Calibration set: 2363 pathogenic, 594 benign

Score cutoffs (95% bootstrap bound):
  Supporting | PP3: score >= 0.031       | BP4: score <= 0.991
  Moderate   | PP3: score >= 0.128       | BP4: score <= 0.912
  Strong     | PP3: score >= 0.793       | BP4: score <= 0.200

Check on 2023 test variants (evidence from the 2023 model vs. what experts decided later):
outcome              B   P
evidence_2023_model       
BP4_Strong           9   1
PP3_Moderate         3  26
PP3_Strong           0  93
PP3_Supporting       8   4
MYH7 p.R403P: PP3_Strong
RHO p.C187S: PP3_Strong
RPE65 p.H68P: PP3_Strong
ABCA4 p.R2040P: PP3_Strong
KCNQ1 p.R591P: PP3_Strong
SCN5A p.R808P: PP3_Strong
CACNA1C p.G402R: PP3_Strong

Today's uncertain variants by evidence level:
BP4_Strong        3555
PP3_Strong        3306
PP3_Moderate      3155
PP3_Supporting    2407


In [82]:
allS = np.concatenate([sP, sB])
allY = np.concatenate([np.ones(len(sP)), np.zeros(len(sB))])
NP, NB = len(sP), len(sB)
W = 150   # number of nearest-scoring calibration variants used at each point

def local_lr(scores, labels, t):
    idx = np.argsort(np.abs(scores - t))[:W]
    nP = labels[idx].sum()
    return (nP / NP) / (max(W - nP, 0.5) / NB)

grid = np.unique(np.quantile(allS, np.linspace(0.01, 0.99, 200)))
rng = np.random.default_rng(11)
boot_lr = []
for _ in range(200):
    bp, bb = rng.choice(sP, NP), rng.choice(sB, NB)
    S = np.concatenate([bp, bb])
    boot_lr.append([local_lr(S, allY, t) for t in grid])
boot_lr = np.array(boot_lr)
lb = np.percentile(boot_lr, 5, axis=0)
ub = np.percentile(boot_lr, 95, axis=0)

def path_cutoff(level):
    cut = None
    for t, l in zip(grid[::-1], lb[::-1]):   # from the top down, while evidence holds
        if l >= level: cut = t
        else: break
    return cut

def benign_cutoff(level):
    cut = None
    for t, u in zip(grid, ub):               # from the bottom up, while evidence holds
        if u <= level: cut = t
        else: break
    return cut

tP = {k: path_cutoff(v) for k, v in LEVELS_P.items()}
tB = {k: benign_cutoff(v) for k, v in LEVELS_B.items()}
print("Score cutoffs (local likelihood ratio, 95% bootstrap bound):")
for k in ["Supporting", "Moderate", "Strong"]:
    p_ = "not reached" if tP[k] is None else f"score >= {tP[k]:.3f}"
    b_ = "not reached" if tB[k] is None else f"score <= {tB[k]:.3f}"
    print(f"  {k:10s} | PP3: {p_:20s} | BP4: {b_}")

te_chk = pd.DataFrame([(evidence_level(x["harmful_2023_model"]), x["clinvar_today"])
                       for x in recs if x["role_2023"] == "test"], columns=["evidence_2023_model", "outcome"])
print()
print("Check on 2023 test variants:")
print(pd.crosstab(te_chk["evidence_2023_model"], te_chk["outcome"]).to_string())

for x in recs:
    x["computational_evidence"] = evidence_level(x["harmful"])
json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)
print()
for card in cards:
    rr = dedup3[(dedup3["gene"] == card["gene"]) & (dedup3["protein_change"] == card["variant"])]
    if len(rr):
        card["computational_evidence"] = evidence_level(float(rr.iloc[0]["harmful"]))
        print(f"{card['gene']} {card['variant']}: {card['computational_evidence']}")
json.dump(cards, open("/content/drive/MyDrive/biohack/demo_cards.json", "w"), indent=2)

unc_ev = pd.Series([x["computational_evidence"] for x in recs if x["clinvar_today"] in ("VUS", "conflicting")])
print()
print("Today's uncertain variants by evidence level:")
print(unc_ev.value_counts().to_string())
pd.DataFrame({"level": list(tP) + list(tB), "side": ["PP3"] * 3 + ["BP4"] * 3,
              "cutoff": list(tP.values()) + list(tB.values())}) \
  .to_csv("/content/drive/MyDrive/biohack/evidence_calibration.csv", index=False)

Score cutoffs (local likelihood ratio, 95% bootstrap bound):
  Supporting | PP3: score >= 0.848       | BP4: score <= 0.230
  Moderate   | PP3: score >= 0.894       | BP4: score <= 0.149
  Strong     | PP3: score >= 1.000       | BP4: score <= 0.038

Check on 2023 test variants:
outcome               B   P
evidence_2023_model        
BP4_Moderate          6   5
BP4_Strong           11   1
BP4_Supporting        1   4
PP3_Moderate          0  87
PP3_Strong            0   3
PP3_Supporting        0   1
indeterminate         2  23

MYH7 p.R403P: PP3_Strong
RHO p.C187S: PP3_Strong
RPE65 p.H68P: PP3_Moderate
ABCA4 p.R2040P: PP3_Strong
KCNQ1 p.R591P: PP3_Strong
SCN5A p.R808P: PP3_Moderate
CACNA1C p.G402R: PP3_Moderate

Today's uncertain variants by evidence level:
BP4_Strong        3926
PP3_Moderate      2631
indeterminate     2546
BP4_Moderate      2307
BP4_Supporting     634
PP3_Supporting     323
PP3_Strong          56


## 21. One-page evidence reports (PDF)

In [83]:
for card in cards:
    print(f"{card['gene']} {card['variant']}: {card.get('computational_evidence')}")
print()
print(pd.Series([x["computational_evidence"] for x in recs
                 if x["clinvar_today"] in ("VUS", "conflicting")]).value_counts().to_string())

MYH7 p.R403P: PP3_Strong
RHO p.C187S: PP3_Strong
RPE65 p.H68P: PP3_Moderate
ABCA4 p.R2040P: PP3_Strong
KCNQ1 p.R591P: PP3_Strong
SCN5A p.R808P: PP3_Moderate
CACNA1C p.G402R: PP3_Moderate

BP4_Strong        3926
PP3_Moderate      2631
indeterminate     2546
BP4_Moderate      2307
BP4_Supporting     634
PP3_Supporting     323
PP3_Strong          56


In [85]:
def dedupe(parts):
    seen, keep = set(), []
    for part in parts:
        name = part.split(" (")[0]
        if name not in seen:
            seen.add(name); keep.append(part)
    return keep

for c in cards:
    c["nearest_in_sequence"] = "; ".join(dedupe(c["nearest_in_sequence"].split("; ")))
    c["nearest_in_3d"] = dedupe(c["nearest_in_3d"])
    rr = dedup3[(dedup3["gene"] == c["gene"]) & (dedup3["protein_change"] == c["variant"])]
    if len(rr):
        dg = rr.iloc[0]["disease_groups"]
        if dg:
            top = max(dg.values())
            c["predicted_disease"] = " / ".join(k for k, v in dg.items() if v == top)
            c["disease_support"] = f"{round(top * 5)}/5 nearest"

STORY_FIX = {
    ("SCN5A", "p.R808P"): "No known disease variants within 5 residues in SCN5A (nearest: R814, 6 residues "
                          "away). Flagged by borrowing from related channels: the matching arginine in the "
                          "voltage sensor causes disease in SCN1A, SCN4A, and SCN8A.",
    ("RPE65", "p.H68P"): "FoldX predicts misfolding (+5.05 kcal/mol), meaning loss of function, the mechanism "
                         "Luxturna treats by adding a working copy. Eligibility requires confirmed "
                         "disease-causing variants in both RPE65 copies.",
}
for c in cards:
    c["story"] = STORY_FIX.get((c["gene"], c["variant"]), c["story"])
json.dump(cards, open("/content/drive/MyDrive/biohack/demo_cards.json", "w"), indent=2)
for c in cards:
    print(f"{c['gene']} {c['variant']}: {c['predicted_disease']} ({c['disease_support']})")

MYH7 p.R403P: Hypertrophic cardiomyopathy (5/5 nearest)
RHO p.C187S: Retinitis pigmentosa (5/5 nearest)
RPE65 p.H68P: Leber congenital amaurosis (5/5 nearest)
ABCA4 p.R2040P: Stargardt disease (4/5 nearest)
KCNQ1 p.R591P: Long QT syndrome (5/5 nearest)
SCN5A p.R808P: Long QT syndrome (3/5 nearest)
CACNA1C p.G402R: Long QT syndrome / Timothy syndrome (5/5 nearest)


In [86]:
!pip -q install reportlab
import matplotlib
from xml.sax.saxutils import escape
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

fdir = matplotlib.get_data_path() + "/fonts/ttf/"
pdfmetrics.registerFont(TTFont("DV", fdir + "DejaVuSans.ttf"))
pdfmetrics.registerFont(TTFont("DVB", fdir + "DejaVuSans-Bold.ttf"))
H = ParagraphStyle("H", fontName="DVB", fontSize=15, leading=19, spaceAfter=4)
S = ParagraphStyle("S", fontName="DVB", fontSize=10.5, leading=14, spaceBefore=7, spaceAfter=2)
B = ParagraphStyle("B", fontName="DV", fontSize=9.5, leading=13)
N = ParagraphStyle("N", fontName="DV", fontSize=8, leading=10.5)

POINTS = {"Strong": 4, "Moderate": 2, "Supporting": 1}
PLAIN = {"PS1-like": "same protein change already pathogenic (different DNA change)",
         "PM5-like": "a different change at a residue with a known pathogenic variant",
         "hotspot": "several known pathogenic variants within 5 residues"}

def plain_evidence(ev):
    return "; ".join(next((f"{k}: {v}" for k, v in PLAIN.items() if part.strip().startswith(k)), part.strip())
                     for part in ev.split(";") if part.strip()) or "none"

story = []
for i, c in enumerate(cards):
    lvl = c.get("computational_evidence", "indeterminate")
    pts = POINTS.get(lvl.split("_")[-1], 0) * (1 if lvl.startswith("PP3") else -1) if "_" in lvl else 0
    fx = c.get("foldx")
    am = "not available" if c["alphamissense"] is None else f"{c['alphamissense']:.3f}"
    lines = [
        ("Summary", f"Currently {c['current_status']} in ClinVar. Ranked #{c['rank_overall']} of 12,345 uncertain "
                    f"protein changes in our panel (#{c['rank_in_gene']} in {c['gene']}). {c['story']}"),
        ("Computational evidence", f"{lvl.replace('_', ' ')} ({pts:+d} points in the ACMG/AMP Bayesian point system). "
                                   f"AlphaMissense: {am}."),
        ("Clinical-rule evidence", plain_evidence(c["evidence"])),
        ("Evidence from related genes", c.get("paralog_evidence") or "none"),
        ("Physical mechanism (FoldX)", f"ΔΔG {fx['ddg_kcal_mol']:+.2f} kcal/mol: {fx['label']}." if fx
                                       else "Not computed for this protein."),
        ("Predicted disease", f"{c['predicted_disease']} ({c['disease_support']} known pathogenic variants)"),
        ("Nearest known variants, sequence", c["nearest_in_sequence"] or "none"),
        ("Nearest known variants, 3D (within 10 Å)", "; ".join(c["nearest_in_3d"]) or "none"),
        ("AlphaFold confidence at this residue", f"pLDDT {c['plddt']}" if c["plddt"] else "not available"),
        ("Recommended confirmatory test", f"{c['suggested_assay']}. A clear result would count as functional "
                                          f"evidence (PS3 if damaging, BS3 if normal)."),
        ("Treatment context (hypothesis for clinicians)", c["treatment_context"] or "none"),
        ("3D view", f"Open {c['viewer_file']} in a web browser."),
    ]
    story.append(Paragraph(escape(f"{c['gene']} {c['variant']}"), H))
    story.append(Paragraph(escape(f"ClinVar: {c['clinvar']}"), N))
    for title, text in lines:
        story.append(Paragraph(escape(title), S))
        story.append(Paragraph(escape(str(text)), B))
    story.append(Spacer(1, 10))
    story.append(Paragraph(escape(
        "Method: transparent logistic model (AlphaMissense, amino-acid chemistry, gene, nearby known variants, "
        "related-gene evidence). Evidence strengths calibrated on 2023 data with local likelihood ratios and the "
        "Bayesian thresholds of Tavtigian et al. (2018). FoldX run on AlphaFold models (no membrane, no ligands)."), N))
    story.append(Paragraph(escape(
        "Research use only. Supports, but does not replace, expert variant classification."), N))
    if i < len(cards) - 1:
        story.append(PageBreak())

path = "/content/drive/MyDrive/biohack/evidence_reports.pdf"
SimpleDocTemplate(path, pagesize=letter, leftMargin=54, rightMargin=54, topMargin=48, bottomMargin=48).build(story)
print("Saved", path)

Saved /content/drive/MyDrive/biohack/evidence_reports.pdf


### Figure 3, legend moved above the bars

In [87]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
without, with_ = [0.831, 0.763], [0.896, 0.923]
x = np.arange(2)
ax.bar(x - 0.18, without, 0.36, color=GREY, label="Without related genes")
ax.bar(x + 0.18, with_, 0.36, color=DARK, label="With related genes")
for xi, a, b_ in zip(x, without, with_):
    ax.text(xi - 0.18, a + 0.01, f"{a:.2f}", ha="center")
    ax.text(xi + 0.18, b_ + 0.01, f"{b_:.2f}", ha="center")
ax.set_xticks(x)
ax.set_xticklabels(["Channel variants with no known\nneighbors in own gene (n=122)",
                    "CACNA1C\n(no AlphaMissense score)"])
ax.set_ylabel("AUROC (5-fold cross-validation)")
ax.set_ylim(0.5, 1.0)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2)
fig.tight_layout()
fig.savefig(f"{FIG}/3_paralogs.png", dpi=200, bbox_inches="tight")
plt.close(fig)
print("Updated 3_paralogs.png")

Updated 3_paralogs.png


## 22. Benchmarks against REVEL and CADD (gnomAD in-silico predictors)

In [88]:
QP = """
query GeneVariants($gene: String!) {
  gene(gene_symbol: $gene, reference_genome: GRCh38) {
    variants(dataset: gnomad_r4) {
      variant_id
      consequence
      in_silico_predictors { id value }
    }
  }
}
"""
r = requests.post(GNOMAD, json={"query": QP, "variables": {"gene": "RHO"}}, timeout=180).json()
ids = set()
for v in r.get("data", {}).get("gene", {}).get("variants", []) or []:
    for p in v.get("in_silico_predictors") or []:
        ids.add(p["id"])
print("Predictors gnomAD provides:", sorted(ids) if ids else r.get("errors", "none"))
pai_id = next((i for i in ids if "primate" in i.lower()), None)

if pai_id is None:
    print("\nPrimateAI isn't available through gnomAD. Keep it as a next step.")
else:
    print(f"\nFound '{pai_id}'. Pulling scores for all genes...")
    pai = {}
    for gname in sorted(out["gene"].unique()):
        rr = requests.post(GNOMAD, json={"query": QP, "variables": {"gene": gname}}, timeout=180)
        if rr.status_code != 200 or "errors" in rr.json():
            print(gname, "failed"); time.sleep(10); continue
        for v in rr.json()["data"]["gene"]["variants"]:
            if v["consequence"] != "missense_variant": continue
            for p in v.get("in_silico_predictors") or []:
                if p["id"] == pai_id and p["value"] not in (None, ""):
                    try:
                        pai["chr" + v["variant_id"].replace("-", ":")] = float(p["value"])
                    except ValueError:
                        pass
        time.sleep(6)
    print("PrimateAI scores collected:", len(pai))

    t_ = te.copy(); t_["pai"] = t_["key"].map(pai)
    f_ = fb.copy(); f_["pai"] = f_["key"].map(pai)
    tp_ = t_[(t_["class_now"] == "P") & t_["pai"].notna()]
    fb_ = f_[f_["pai"].notna() & f_["am_pathogenicity"].notna()]
    print(f"Comparison set: {len(tp_)} later-pathogenic test variants vs {len(fb_)} gnomAD-benign")
    if len(tp_) >= 10 and len(fb_) >= 10:
        y_ = [1] * len(tp_) + [0] * len(fb_)
        for name, s1, s2 in [
            ("PrimateAI", tp_["pai"], fb_["pai"]),
            ("AlphaMissense", tp_["am_pathogenicity"], fb_["am_pathogenicity"]),
            ("Our model", sc4("+ paralogs", tp_), sc4("+ paralogs", fb_)),
        ]:
            print(f"   {name:14s} AUROC {roc_auc_score(y_, list(s1) + list(s2)):.3f}")
    else:
        print("Too few overlapping variants for a fair comparison.")

Predictors gnomAD provides: ['cadd', 'pangolin_largest_ds', 'phylop', 'polyphen_max', 'revel_max', 'sift_max', 'spliceai_ds_max']

PrimateAI isn't available through gnomAD. Keep it as a next step.


In [89]:
pred = {"revel_max": {}, "cadd": {}}
for gname in sorted(out["gene"].unique()):
    rr = requests.post(GNOMAD, json={"query": QP, "variables": {"gene": gname}}, timeout=180)
    if rr.status_code != 200 or "errors" in rr.json():
        print(gname, "failed"); time.sleep(10); continue
    for v in rr.json()["data"]["gene"]["variants"]:
        if v["consequence"] != "missense_variant":
            continue
        k_ = "chr" + v["variant_id"].replace("-", ":")
        for p in v.get("in_silico_predictors") or []:
            if p["id"] in pred and p["value"] not in (None, ""):
                try:
                    pred[p["id"]][k_] = float(p["value"])
                except ValueError:
                    pass
    time.sleep(6)
print("Scores collected: REVEL", len(pred["revel_max"]), "| CADD", len(pred["cadd"]))

t_ = te.copy(); f_ = fb.copy(); v_ = vus.copy()
for d in (t_, f_, v_):
    d["revel"] = d["key"].map(pred["revel_max"])
    d["cadd"] = d["key"].map(pred["cadd"])
    d["ours"] = sc4("+ paralogs", d)

tp_ = t_[(t_["class_now"] == "P") & t_["revel"].notna() & t_["cadd"].notna()]
fb_ = f_[f_["revel"].notna() & f_["cadd"].notna()]
print(f"\n1. Harmful vs benign: {len(tp_)} later-pathogenic test variants vs {len(fb_)} gnomAD-benign")
y_ = [1] * len(tp_) + [0] * len(fb_)
for name, col in [("REVEL", "revel"), ("CADD", "cadd"), ("AlphaMissense", "am_pathogenicity"), ("Our model", "ours")]:
    print(f"   {name:14s} AUROC {roc_auc_score(y_, list(tp_[col]) + list(fb_[col])):.3f}")

vv = v_[v_["revel"].notna() & v_["cadd"].notna()].copy()
k5 = max(1, int(0.05 * len(vv)))
base_v = vv["later_P"].mean()
print(f"\n2. Ranking: {len(vv)} 2023 VUS with REVEL scores, {int(vv['later_P'].sum())} later proven pathogenic")
print(f"   Top 5% = {k5} variants; random would find {k5 * base_v:.1f}")
for name, col in [("REVEL", "revel"), ("CADD", "cadd"), ("AlphaMissense", "am_pathogenicity"), ("Our model", "ours")]:
    hits = int(vv.sort_values(col, ascending=False)["later_P"].head(k5).sum())
    print(f"   {name:14s} {hits} found ({hits / (k5 * base_v):.1f}x random)")

Scores collected: REVEL 38533 | CADD 44946

1. Harmful vs benign: 69 later-pathogenic test variants vs 132 gnomAD-benign
   REVEL          AUROC 0.893
   CADD           AUROC 0.861
   AlphaMissense  AUROC 0.894
   Our model      AUROC 0.953

2. Ranking: 6536 2023 VUS with REVEL scores, 69 later proven pathogenic
   Top 5% = 326 variants; random would find 3.4
   REVEL          17 found (4.9x random)
   CADD           10 found (2.9x random)
   AlphaMissense  13 found (3.8x random)
   Our model      27 found (7.8x random)


In [90]:
rng = np.random.default_rng(21)
y_all = np.array([1] * len(tp_) + [0] * len(fb_))
s_ours = np.concatenate([tp_["ours"].values, fb_["ours"].values])
s_rev = np.concatenate([tp_["revel"].values, fb_["revel"].values])
d_auc = []
for _ in range(1000):
    i = rng.integers(0, len(y_all), len(y_all))
    if y_all[i].min() != y_all[i].max():
        d_auc.append(roc_auc_score(y_all[i], s_ours[i]) - roc_auc_score(y_all[i], s_rev[i]))
print(f"AUROC gain over REVEL: {np.mean(d_auc):+.3f} "
      f"(95% CI {np.percentile(d_auc, 2.5):+.3f} to {np.percentile(d_auc, 97.5):+.3f})")

lp_v, so_v, sr_v = vv["later_P"].values, vv["ours"].values, vv["revel"].values
d_hit = []
for _ in range(500):
    i = rng.integers(0, len(vv), len(vv))
    d_hit.append(lp_v[i][np.argsort(-so_v[i])[:k5]].sum() - lp_v[i][np.argsort(-sr_v[i])[:k5]].sum())
print(f"Top-5% hit gain over REVEL: {np.mean(d_hit):+.1f} "
      f"(95% CI {np.percentile(d_hit, 2.5):+.1f} to {np.percentile(d_hit, 97.5):+.1f})")

AUROC gain over REVEL: +0.062 (95% CI +0.026 to +0.102)
Top-5% hit gain over REVEL: +9.7 (95% CI +2.0 to +19.0)


In [91]:
tools = ["CADD", "AlphaMissense", "REVEL", "Our model"]
aucs = [0.861, 0.894, 0.893, 0.953]
hits = [10, 13, 17, 27]
colors_ = [GREY, GREY, GREY, DARK]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].bar(tools, aucs, color=colors_, width=0.6)
for i, v in enumerate(aucs):
    axes[0].text(i, v + 0.005, f"{v:.3f}", ha="center")
axes[0].set_ylim(0.8, 1.0); axes[0].set_ylabel("AUROC, harmful vs benign")
axes[1].bar(tools, hits, color=colors_, width=0.6)
for i, v in enumerate(hits):
    axes[1].text(i, v + 0.5, str(v), ha="center")
axes[1].axhline(3.4, color=GREY, ls="--", lw=1.2)
axes[1].text(-0.4, 4.2, "random", color="#555", fontsize=10)
axes[1].set_ylim(0, 32); axes[1].set_ylabel("Later proven pathogenic in top 5%")
for ax in axes:
    ax.tick_params(axis="x", labelsize=10)
fig.tight_layout()
fig.savefig(f"{FIG}/6_vs_standard_tools.png", dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved 6_vs_standard_tools.png")

Saved 6_vs_standard_tools.png


## 23. Mouse phenotypes (MGI)

In [93]:
txt = requests.get("https://www.informatics.jax.org/downloads/reports/HMD_HumanPhenotype.rpt", timeout=120).text
GSET = set(out["gene"].unique())
mouse = {}
for line in txt.splitlines():
    cols = line.split("\t")
    if cols and cols[0] in GSET:
        mps = set(re.findall(r"MP:\d{7}", line))
        sym = next((c for c in cols[1:] if c[:1].isupper() and c[1:].islower()), "")
        prev = mouse.get(cols[0], ("", set()))
        mouse[cols[0]] = (sym or prev[0], prev[1] | mps)
print("Genes with mouse phenotype data:", len(mouse), "of", len(GSET))

mp_names = {}
for mp in sorted({m for _, ms in mouse.values() for m in ms}):
    try:
        iri = "http://purl.obolibrary.org/obo/" + mp.replace(":", "_")
        j = requests.get("https://www.ebi.ac.uk/ols4/api/ontologies/mp/terms", params={"iri": iri}, timeout=30).json()
        mp_names[mp] = j["_embedded"]["terms"][0]["label"]
    except Exception:
        mp_names[mp] = mp

RETINA = {"RHO", "ABCA4", "GUCY2D", "CRX", "PRPH2", "RPE65", "CEP290", "BEST1", "NR2E3", "RPGR", "OPA1", "PAX6", "USH2A"}
print()
for g in sorted(GSET):
    sym, mps = mouse.get(g, ("", set()))
    names = sorted(mp_names.get(m, m) for m in mps)
    want = "vision/eye" if g in RETINA else ("cardiovascular" if g != "GLA" else None)
    match = "" if not want else ("matches" if any(want in n for n in names) else "no match")
    print(f"{g:8s} {sym:8s} {match:9s} {', '.join(names)[:110]}")

def mouse_line(g):
    sym, mps = mouse.get(g, ("", set()))
    if not mps:
        return ""
    return f"{sym} mutant mice (MGI): " + ", ".join(sorted(mp_names.get(m, m) for m in mps))

for x in recs:
    x["mouse_evidence"] = mouse_line(x["gene"])
json.dump(recs, open("/content/drive/MyDrive/biohack/predictions.json", "w"))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in x.items()} for x in recs]) \
  .to_csv("/content/drive/MyDrive/biohack/predictions.csv", index=False)
for c in cards:
    c["mouse_evidence"] = mouse_line(c["gene"])
json.dump(cards, open("/content/drive/MyDrive/biohack/demo_cards.json", "w"), indent=2)
pd.DataFrame([{"gene": g, "mouse_symbol": mouse.get(g, ("", set()))[0], "mouse_evidence": mouse_line(g)}
              for g in sorted(GSET)]).to_csv("/content/drive/MyDrive/biohack/mouse_phenotypes_mgi.csv", index=False)
print("\nSaved mouse_phenotypes_mgi.csv and updated predictions + demo cards")

Genes with mouse phenotype data: 23 of 23

ABCA4    Abca4    matches   hematopoietic system phenotype, homeostasis/metabolism phenotype, nervous system phenotype, pigmentation pheno
BEST1    Best1    matches   cellular phenotype, pigmentation phenotype, reproductive system phenotype, vision/eye phenotype
CACNA1C  Cacna1c  matches   adipose tissue phenotype, behavior/neurological phenotype, cardiovascular system phenotype, cellular phenotype
CEP290   Cep290   matches   behavior/neurological phenotype, cardiovascular system phenotype, cellular phenotype, craniofacial phenotype, 
CRX      Crx      matches   behavior/neurological phenotype, cardiovascular system phenotype, endocrine/exocrine gland phenotype, growth/s
GLA      Gla                cardiovascular system phenotype, cellular phenotype, growth/size/body region phenotype, liver/biliary system p
GUCY2D   Gucy2e   matches   nervous system phenotype, vision/eye phenotype
KCNH2    Kcnh2    matches   cardiovascular system phenotype
KCN

## 24. Population labels for the 1000 Genomes carriers

In [94]:
kgc["superpop"] = kgc["sample"].map(pop_of)
kgc.to_csv("/content/drive/MyDrive/biohack/kg1000_carriers.csv", index=False)
print(kgc["superpop"].value_counts(dropna=False).to_string())

superpop
AFR    18078
EUR     9732
SAS     9408
EAS     9319
AMR     7847
